<a href="https://colab.research.google.com/github/YohanaNatasya/ADB-AI-for-Safer-Roads-Innovation-Challenge/blob/main/LOKAGRID-ADB_Road_Safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INSTRUCTION**
1. Upload the ADB_Innovation_Maharashtra.geojson and ADB_Innovation_Thailand.geojson to the local files in google collab
2. Run the code based on the order

# GOOGLE COLLAB PYTHON PREPARATION

In [ ]:
# ============================================================
# REQUIRED PACKAGE INSTALLATION
# ============================================================
!pip install -q geopandas pyogrio shapely fiona tqdm requests osmium rasterio pyproj scikit-learn scipy joblib

# ============================================================
# IMPORT PACKAGES
# ============================================================
import os
import gc
import math
import shutil
import zipfile
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely
import requests
import osmium
import rasterio
import joblib

from pathlib import Path
from itertools import combinations
from pyproj import Transformer
from scipy.optimize import linear_sum_assignment
from shapely.geometry import LineString, Polygon
from shapely.ops import linemerge
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    davies_bouldin_score,
    pairwise_distances,
    silhouette_score,
)
from tqdm.auto import tqdm
from IPython.display import display

import folium
import html

In [ ]:
# ============================================================
# FOLDER AND FILE PATHS
# ============================================================

# Input GeoJSON files
THAILAND_GEOJSON_INPUT = "/content/ADB_Innovation_Thailand.geojson"
MAHARASHTRA_GEOJSON_INPUT = "/content/ADB_Innovation_Maharashtra.geojson"

# ============================================================
# STEP 1 OUTPUT FOLDER
# All Step 1 outputs are now saved here.
# ============================================================
OUTPUT_FOLDER = "/content/1. Ready to Use"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Standardized merged outputs
STANDARDIZED_MERGED_OUTPUT_GPKG = os.path.join(
    OUTPUT_FOLDER,
    "standardized_road_safety_thailand_maharashtra.gpkg",
)

STANDARDIZED_MERGED_OUTPUT_CSV = os.path.join(
    OUTPUT_FOLDER,
    "standardized_road_safety_thailand_maharashtra_attributes.csv",
)

# Step 1 split outputs
FIRST_DATA_PREPARATION_RESULT_VALID = os.path.join(
    OUTPUT_FOLDER,
    "ADB_Innovation_merge_valid.gpkg",
)

FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED = os.path.join(
    OUTPUT_FOLDER,
    "ADB_Innovation_merge_notincluded.gpkg",
)

# ============================================================
# STEP 2 — ADD DERIVED INTERSECTION DATA
# ============================================================
INTERSECTION_OUTPUT_FOLDER = (
    "/content/2. Add Derived Intersection Data"
)
os.makedirs(INTERSECTION_OUTPUT_FOLDER, exist_ok=True)

LOCAL_INTERSECTION_RESULT = os.path.join(
    INTERSECTION_OUTPUT_FOLDER,
    "ADB_Innovation_merge_valid_2.gpkg",
)

# ============================================================
# STEP 3A — DOWNLOAD RAW OPENSTREETMAP DATA
# ============================================================

# Main storage folder for the OSM POI workflow.
EXTERNAL_DATASET_FOLDER = "/content/3. OSM POI Data"
os.makedirs(EXTERNAL_DATASET_FOLDER, exist_ok=True)

# Persistent raw OSM source files downloaded in Step 3A.
RAW_OSM_STORAGE_FOLDER = os.path.join(
    EXTERNAL_DATASET_FOLDER,
    "Raw OSM PBF",
)
os.makedirs(RAW_OSM_STORAGE_FOLDER, exist_ok=True)

# Temporary local working folder.
LOCAL_WORK_ROOT = os.path.join(
    EXTERNAL_DATASET_FOLDER,
    "_local_work",
)
os.makedirs(LOCAL_WORK_ROOT, exist_ok=True)

# Raw OSM PBF download summary.
OSM_RAW_PBF_DOWNLOAD_SUMMARY_CSV = os.path.join(
    EXTERNAL_DATASET_FOLDER,
    "osm_raw_pbf_download_summary.csv",
)

# Thailand raw OSM PBF source.
THAILAND_OSM_PBF_URL = (
    "https://download.geofabrik.de/asia/"
    "thailand-latest.osm.pbf"
)

# Maharashtra is covered by the India Western Zone extract.
MAHARASHTRA_OSM_PBF_URL = (
    "https://download.geofabrik.de/asia/india/"
    "western-zone-latest.osm.pbf"
)

# OSM region configuration.
REGION_CONFIG = {
    "thailand": {
        "region_name": "Thailand",
        "dataset_value": "Thailand",
        "pbf_url": THAILAND_OSM_PBF_URL,
        "storage_pbf_path": os.path.join(
            RAW_OSM_STORAGE_FOLDER,
            "thailand-latest.osm.pbf",
        ),
    },
    "maharashtra": {
        "region_name": "Maharashtra "
        "(source: India Western Zone)",
        "dataset_value": "Maharashtra",
        "pbf_url": MAHARASHTRA_OSM_PBF_URL,
        "storage_pbf_path": os.path.join(
            RAW_OSM_STORAGE_FOLDER,
            "western-zone-latest.osm.pbf",
        ),
    },
}

# ============================================================
# STEP 3B — EXTRACT OSM POI DATA + ADD ROAD-LEVEL METRICS
# ============================================================

# Input road dataset:
# valid roads with the Step 2 intersection metrics already added.
POI_ROAD_INPUT = LOCAL_INTERSECTION_RESULT

# Extracted raw POI files for audit and later reuse.
EXTRACTED_OSM_POI_FOLDER = os.path.join(
    EXTERNAL_DATASET_FOLDER,
    "Extracted POI",
)
os.makedirs(EXTRACTED_OSM_POI_FOLDER, exist_ok=True)

# Final road-level POI output folder.
POI_ROAD_METRICS_OUTPUT_FOLDER = os.path.join(
    EXTERNAL_DATASET_FOLDER,
    "Road Segment POI Metrics",
)
os.makedirs(POI_ROAD_METRICS_OUTPUT_FOLDER, exist_ok=True)

# Step 3B final road output.
SURROUNDING_LAND_USE_RESULT = os.path.join(
    POI_ROAD_METRICS_OUTPUT_FOLDER,
    "ADB_Innovation_merge_valid_3.gpkg",
)

# Step 3B audit summary.
POI_METRICS_SUMMARY_CSV = os.path.join(
    POI_ROAD_METRICS_OUTPUT_FOLDER,
    "poi_road_metrics_summary.csv",
)

# One extracted raw CSV per POI group and dataset.
POI_EXTRACT_OUTPUT_CONFIG = {
    "Thailand": {
        "public_transport": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "thailand_public_transport.csv",
        ),
        "office": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "thailand_office.csv",
        ),
        "school": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "thailand_school.csv",
        ),
        "commercial": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "thailand_commercial.csv",
        ),
        "industrial": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "thailand_industrial.csv",
        ),
    },
    "Maharashtra": {
        "public_transport": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "maharashtra_public_transport.csv",
        ),
        "office": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "maharashtra_office.csv",
        ),
        "school": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "maharashtra_school.csv",
        ),
        "commercial": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "maharashtra_commercial.csv",
        ),
        "industrial": os.path.join(
            EXTRACTED_OSM_POI_FOLDER,
            "maharashtra_industrial.csv",
        ),
    },
}

# ============================================================
# STEP 4 — ADD GHSL BUILT-UP CONTEXT
# ============================================================

# Input: latest Step 3 road-level POI output.
GHSL_ROAD_INPUT = SURROUNDING_LAND_USE_RESULT

# Main Step 4 output folder.
GHSL_OUTPUT_FOLDER = "/content/4. GHSL Built-Up Context"
os.makedirs(GHSL_OUTPUT_FOLDER, exist_ok=True)

# Local GHSL cache.
GHSL_WORK_ROOT = os.path.join(
    GHSL_OUTPUT_FOLDER,
    "GHSL Cache",
)
os.makedirs(GHSL_WORK_ROOT, exist_ok=True)

GHSL_RAW_DIR = os.path.join(
    GHSL_WORK_ROOT,
    "raw_zip",
)
os.makedirs(GHSL_RAW_DIR, exist_ok=True)

GHSL_EXTRACT_DIR = os.path.join(
    GHSL_WORK_ROOT,
    "extracted_tif",
)
os.makedirs(GHSL_EXTRACT_DIR, exist_ok=True)

# Step 4 final GeoPackage output.
GHSL_RESULT = os.path.join(
    GHSL_OUTPUT_FOLDER,
    "ADB_Innovation_merge_valid_4.gpkg",
)

GHSL_RESULT_LAYER = "ADB_Innovation_merge_valid_4"

# Step 4 summary output.
GHSL_SUMMARY_CSV = os.path.join(
    GHSL_OUTPUT_FOLDER,
    "ghsl_built_up_summary.csv",
)

# GHSL Built-Up Surface source settings.
GHSL_EPOCH = 2020

GHSL_BASE_URL = (
    "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/"
    "GHS_BUILT_S_GLOBE_R2023A/"
    "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss/"
    "V1-0/tiles"
)

# Road context settings.
# Full corridor width = 200 m:
# 100 m on each side of the road.
LATERAL_BUFFER_M = 100

# Each road is assessed in consecutive 100 m units.
CONTINUITY_CELL_M = 100

# A unit is categorized as built-up when its mean built-up
# share across the road corridor is at least 15%.
BUILT_UP_CELL_THRESHOLD_PCT = 15.0

# Five samples across the 200 m corridor.
LATERAL_SAMPLE_OFFSETS_M = np.array(
    [-80.0, -40.0, 0.0, 40.0, 80.0],
    dtype="float64",
)

# At least three valid GHSL samples are required for a road unit.
MIN_VALID_LATERAL_SAMPLES = 3

# Keep the raw ZIP cache, but remove extracted TIFFs after use.
DELETE_EXTRACTED_TIF_AFTER_USE = True

# ============================================================
# STEP 5 — POI + GHSL CONTEXT CLUSTERING
# ============================================================

# Input: Step 4 road dataset with POI and GHSL metrics.
CLUSTERING_INPUT = GHSL_RESULT

# Main Step 5 output folder.
CLUSTERING_OUTPUT_FOLDER = (
    "/content/5. POI and GHSL Context Clustering"
)
os.makedirs(CLUSTERING_OUTPUT_FOLDER, exist_ok=True)

# Final road dataset with POI + GHSL cluster code.
CLUSTERING_RESULT = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "ADB_Innovation_merge_valid_5.gpkg",
)

CLUSTERING_RESULT_LAYER = (
    "ADB_Innovation_merge_valid_5"
)

# Reusable fitted K-Means bundle.
CLUSTERING_MODEL_FILE = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "poi_ghsl_context_kmeans_model.joblib",
)

# Automatic K-selection dashboard.
CLUSTERING_K_SELECTION_CSV = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "automatic_k_selection_dashboard.csv",
)

# Cluster-level profile table.
CLUSTERING_PROFILE_CSV = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "poi_ghsl_cluster_profile.csv",
)

# Feature preparation and missing-value audit.
CLUSTERING_FEATURE_AUDIT_CSV = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "clustering_feature_audit.csv",
)

# Feature dictionary reference table.
CLUSTERING_FEATURE_METADATA_CSV = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "clustering_feature_metadata.csv",
)

# Exact feature dictionary used for the model.
CLUSTERING_FEATURE_CONFIG_JSON = os.path.join(
    CLUSTERING_OUTPUT_FOLDER,
    "model_feature_config.json",
)

# ============================================================
# STEP 6 — TARGET SPEED LIMIT BY INTERSECTION DENSITY
# ============================================================

# Input: Step 5 clustered road dataset.
TARGET_SPEED_INPUT = CLUSTERING_RESULT

# Main Step 6 output folder.
FINAL_RESULT_OUTPUT_FOLDER = (
    "/content/6. Final Result"
)
os.makedirs(FINAL_RESULT_OUTPUT_FOLDER, exist_ok=True)

# Final GeoPackage with target speed limit and intersection density.
TARGET_SPEED_RESULT = os.path.join(
    FINAL_RESULT_OUTPUT_FOLDER,
    "ADB_Innovation_target_speed_limit.gpkg",
)

TARGET_SPEED_RESULT_LAYER = (
    "ADB_Innovation_target_speed_limit"
)

# Audit file containing user-defined initial speed and context
# for each Cluster Code / Cluster Name.
TARGET_SPEED_CLUSTER_CONFIG_CSV = os.path.join(
    FINAL_RESULT_OUTPUT_FOLDER,
    "target_speed_cluster_configuration.csv",
)

# ============================================================
# STEP 7 — FINAL RISK ASSESSMENT AND SCORING
# ============================================================

# Input: Step 6 output with target speed limit and
# intersection-density classification.
SCORING_INPUT = TARGET_SPEED_RESULT

# Final dashboard-ready GeoPackage.
SCORING_RESULT = os.path.join(
    FINAL_RESULT_OUTPUT_FOLDER,
    "ADB_Innovation_target_speed_limit_with_scoring.gpkg",
)

SCORING_RESULT_LAYER = (
    "ADB_Innovation_target_speed_limit_with_scoring"
)

# Audit file containing user-entered baseline VRU points
# by Cluster Code and Cluster Name.
SCORING_CLUSTER_VRU_CONFIG_CSV = os.path.join(
    FINAL_RESULT_OUTPUT_FOLDER,
    "cluster_vru_baseline_configuration.csv",
)

# Dashboard-ready scoring summary.
SCORING_SUMMARY_CSV = os.path.join(
    FINAL_RESULT_OUTPUT_FOLDER,
    "road_safety_scoring_summary.csv",
)

# ============================================================
# STEP 8 — INTERACTIVE SAFE SYSTEM ASSESSMENT MAPS
# ============================================================

SCORING_MAP_INPUT = (
    "/content/6. Final Result/"
    "ADB_Innovation_target_speed_limit_with_scoring.gpkg"
)

SCORING_MAP_OUTPUT_FOLDER = (
    "/content/6. Final Result"
)

THAILAND_SAFE_SYSTEM_MAP_HTML = os.path.join(
    SCORING_MAP_OUTPUT_FOLDER,
    "thailand_safe_system_assessment_map.html",
)

MAHARASHTRA_SAFE_SYSTEM_MAP_HTML = os.path.join(
    SCORING_MAP_OUTPUT_FOLDER,
    "maharashtra_safe_system_assessment_map.html",
)

os.makedirs(
    SCORING_MAP_OUTPUT_FOLDER,
    exist_ok=True,
)

print("Original input files:")
print(f"Thailand   : {THAILAND_GEOJSON_INPUT}")
print(f"Maharashtra: {MAHARASHTRA_GEOJSON_INPUT}")

print("\nStep 1 output folder:")
print(OUTPUT_FOLDER)

print("\nStep 2 output file:")
print(LOCAL_INTERSECTION_RESULT)

print("\nStep 3B output file:")
print(SURROUNDING_LAND_USE_RESULT)

print("\nStep 4 input file:")
print(GHSL_ROAD_INPUT)

print("\nStep 4 final output:")
print(GHSL_RESULT)

print("\nStep 5 clustering input:")
print(CLUSTERING_INPUT)

print("\nStep 5 final output:")
print(CLUSTERING_RESULT)

print("\nStep 5 model file:")
print(CLUSTERING_MODEL_FILE)

print("\nStep 5 feature configuration:")
print(CLUSTERING_FEATURE_CONFIG_JSON)

print("\nStep 6 input:")
print(TARGET_SPEED_INPUT)

print("\nStep 6 final output:")
print(TARGET_SPEED_RESULT)

print("\nStep 7 scoring input:")
print(SCORING_INPUT)

print("\nStep 7 final output:")
print(SCORING_RESULT)

# PRE-PROCESSING DATA

## Merge and Standarized Thailand and Maharastra

In [ ]:
# ============================================================
# STEP 1 — LOAD, STANDARDIZE, MERGE, SPLIT, AND SAVE
#
# Input:
#   1. /content/ADB_Innovation_Thailand.geojson
#   2. /content/ADB_Innovation_Maharashtra.geojson
#
# Main primary key:
#   analysis_segment_id
#
# Outputs:
#   1. standardized_road_safety_thailand_maharashtra.gpkg
#   2. standardized_road_safety_thailand_maharashtra_attributes.csv
#   3. ADB_Innovation_merge_valid.gpkg
#   4. ADB_Innovation_merge_notincluded.gpkg
# ============================================================


# ------------------------------------------------------------
# 1. Validate and load local GeoJSON input files
# ------------------------------------------------------------
input_files = {
    "Thailand": THAILAND_GEOJSON_INPUT,
    "Maharashtra": MAHARASHTRA_GEOJSON_INPUT,
}

for dataset_name, file_path in input_files.items():
    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"{dataset_name} input file was not found:\n{file_path}"
        )

print("Loading Thailand GeoJSON...")
thailand_gdf = gpd.read_file(THAILAND_GEOJSON_INPUT)

print("Loading Maharashtra GeoJSON...")
maharashtra_gdf = gpd.read_file(MAHARASHTRA_GEOJSON_INPUT)

if thailand_gdf.empty:
    raise ValueError("Thailand GeoJSON contains zero records.")

if maharashtra_gdf.empty:
    raise ValueError("Maharashtra GeoJSON contains zero records.")

print(f"Thailand input rows    : {len(thailand_gdf):,}")
print(f"Maharashtra input rows : {len(maharashtra_gdf):,}")

print("\nThailand CRS:", thailand_gdf.crs)
print("Maharashtra CRS:", maharashtra_gdf.crs)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def get_col(df, col_name):
    """
    Return the requested column if available.
    Otherwise, return a same-length NaN Series.
    """
    if col_name in df.columns:
        return df[col_name].copy()

    return pd.Series(
        np.nan,
        index=df.index,
        dtype="object",
    )


def get_numeric_col(df, col_name):
    """
    Return a numeric version of the requested column if available.
    Otherwise, return a same-length NaN Series.
    """
    if col_name in df.columns:
        return pd.to_numeric(
            df[col_name],
            errors="coerce",
        )

    return pd.Series(
        np.nan,
        index=df.index,
        dtype="float64",
    )


def standardize_dataset(gdf, dataset_name):
    """
    Standardize Thailand and Maharashtra into one consistent schema.
    """
    df = gdf.copy()

    if dataset_name == "Thailand":
        source_segment_id = get_col(df, "OvertureID")
        source_segment_id_type = "OvertureID"
        road_name = get_col(df, "english_ro")

        source_class = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        source_subtype = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        road_class_raw = get_col(df, "RoadClass")
        road_class_final = road_class_raw.copy()

        road_class_final_source = pd.Series(
            "RoadClass",
            index=df.index,
            dtype="object",
        )

        province_id = get_col(df, "ProvinceID")

        urban_pc = pd.Series(
            np.nan,
            index=df.index,
            dtype="float64",
        )

        sample_size_total = get_numeric_col(
            df,
            "SampleSizeTotal",
        )

        inv_percentile = get_numeric_col(
            df,
            "InvPercentile",
        )

        # Thailand RankedPercentile is already on a 0–100 scale.
        ranked_percentile = get_numeric_col(
            df,
            "RankedPercentile",
        )

        for_analysis = get_col(df, "ForAnalysis")

        pass_value = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        exclude_from_speed_spi = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        no_of_result_segments = get_col(
            df,
            "NO_OF_Result_Segments",
        )

    elif dataset_name == "Maharashtra":
        source_segment_id = get_col(df, "DISSOLVE_ID")
        source_segment_id_type = "DISSOLVE_ID"
        road_name = get_col(df, "names_primary")

        source_class = get_col(df, "class")
        source_subtype = get_col(df, "subtype")

        road_class_raw = get_col(df, "RoadClass")

        # Use RoadClass when available.
        # Otherwise, use Maharashtra class as fallback.
        road_class_final = road_class_raw.copy()
        road_class_final = road_class_final.where(
            road_class_final.notna(),
            source_class,
        )

        road_class_final_source = pd.Series(
            np.where(
                road_class_raw.notna(),
                "RoadClass",
                "class_fallback",
            ),
            index=df.index,
            dtype="object",
        )

        province_id = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        urban_pc = get_numeric_col(df, "UrbanPC")

        sample_size_total = get_numeric_col(
            df,
            "Sample_Size_Total",
        )

        inv_percentile = pd.Series(
            np.nan,
            index=df.index,
            dtype="float64",
        )

        # Maharashtra RankedPercentile is converted from 0–1 to 0–100.
        ranked_percentile = (
            get_numeric_col(
                df,
                "RankedPercentile",
            )
            * 100
        )

        for_analysis = pd.Series(
            np.nan,
            index=df.index,
            dtype="object",
        )

        pass_value = get_col(df, "Pass")

        exclude_from_speed_spi = get_col(
            df,
            "ExcludeFromSpeedSPI",
        )

        no_of_result_segments = pd.Series(
            np.nan,
            index=df.index,
            dtype="float64",
        )

    else:
        raise ValueError(
            "dataset_name must be either 'Thailand' or 'Maharashtra'."
        )

    out = gpd.GeoDataFrame(
        {
            # Dataset identity
            "dataset": dataset_name,
            "object_id": get_col(df, "OBJECTID"),

            # Segment / road identity
            "source_segment_id": source_segment_id,
            "source_segment_id_type": source_segment_id_type,
            "road_name": road_name,

            # Road class fields
            "source_class": source_class,
            "source_subtype": source_subtype,
            "road_class_raw": road_class_raw,
            "road_class_final": road_class_final,
            "road_class_final_source": road_class_final_source,

            # Location / land use
            "land_use": get_col(df, "LandUse"),
            "province_id": province_id,
            "urban_pc": urban_pc,

            # Traffic / exposure
            "sample_size_avg": get_numeric_col(
                df,
                "SampleSize_avg",
            ),
            "sample_size_total": sample_size_total,
            "road_length": get_numeric_col(
                df,
                "RoadLength",
            ),
            "shape_length": get_numeric_col(
                df,
                "Shape_Length",
            ),
            "weighted_sample": get_numeric_col(
                df,
                "WeightedSample",
            ),

            # Percentile / ranking
            "percent": get_numeric_col(df, "Percent_"),
            "percentile": get_numeric_col(df, "Percentile"),
            "inv_percentile": inv_percentile,
            "ranked_percentile": ranked_percentile,
            "percentile_band": get_col(
                df,
                "PercentileBand",
            ),

            # Speed / speeding
            "speed_limit": get_numeric_col(df, "SpeedLimit"),
            "speed_limit_floor": get_numeric_col(
                df,
                "SpeedLimitFloor",
            ),
            "number_over_limit": get_numeric_col(
                df,
                "NumberOverLimit",
            ),
            "median_speed": get_numeric_col(df, "MedianSpeed"),
            "p85_speed": get_numeric_col(
                df,
                "F85thPercentileSpeed",
            ),
            "percent_over_limit": get_numeric_col(
                df,
                "PercentOverLimit",
            ),

            # Analysis flags
            "for_analysis": for_analysis,
            "pass_value": pass_value,
            "exclude_from_speed_spi": exclude_from_speed_spi,
            "analysis_status": get_col(
                df,
                "AnalysisStatus",
            ),

            # Supporting fields
            "street_image_link": get_col(
                df,
                "StreetImageLink",
            ),
            "no_of_result_segments": no_of_result_segments,

            # Geometry
            "geometry": df.geometry,
        },
        geometry="geometry",
        crs=df.crs,
    )

    source_id_text = (
        out["source_segment_id"]
        .astype("string")
        .str.strip()
    )

    missing_source_id_mask = (
        source_id_text.isna()
        | source_id_text.eq("")
    )

    if missing_source_id_mask.any():
        raise ValueError(
            f"{dataset_name} has "
            f"{missing_source_id_mask.sum():,} record(s) "
            "without source_segment_id."
        )

    out["analysis_segment_id"] = (
        out["dataset"].astype("string")
        + "__"
        + out["source_segment_id_type"].astype("string")
        + "__"
        + source_id_text
    )

    final_columns = [
        "dataset",
        "object_id",
        "analysis_segment_id",
        "source_segment_id",
        "source_segment_id_type",
        "road_name",
        "source_class",
        "source_subtype",
        "road_class_raw",
        "road_class_final",
        "road_class_final_source",
        "land_use",
        "province_id",
        "urban_pc",
        "sample_size_avg",
        "sample_size_total",
        "road_length",
        "shape_length",
        "weighted_sample",
        "percent",
        "percentile",
        "inv_percentile",
        "ranked_percentile",
        "percentile_band",
        "speed_limit",
        "speed_limit_floor",
        "number_over_limit",
        "median_speed",
        "p85_speed",
        "percent_over_limit",
        "for_analysis",
        "pass_value",
        "exclude_from_speed_spi",
        "analysis_status",
        "street_image_link",
        "no_of_result_segments",
        "geometry",
    ]

    return out[final_columns]


# ------------------------------------------------------------
# 3. Standardize Thailand and Maharashtra
# ------------------------------------------------------------
print("\nStandardizing Thailand dataset...")
thailand_std_gdf = standardize_dataset(
    thailand_gdf,
    "Thailand",
)

print("Standardizing Maharashtra dataset...")
maharashtra_std_gdf = standardize_dataset(
    maharashtra_gdf,
    "Maharashtra",
)


# ------------------------------------------------------------
# 4. Align CRS before combining
# ------------------------------------------------------------
if thailand_std_gdf.crs is None:
    raise ValueError("Thailand CRS is missing.")

if maharashtra_std_gdf.crs is None:
    raise ValueError("Maharashtra CRS is missing.")

if thailand_std_gdf.crs != maharashtra_std_gdf.crs:
    print(
        "\nCRS differs. "
        "Converting Maharashtra CRS to Thailand CRS..."
    )

    maharashtra_std_gdf = maharashtra_std_gdf.to_crs(
        thailand_std_gdf.crs
    )


# ------------------------------------------------------------
# 5. Merge standardized datasets
# ------------------------------------------------------------
combined_std_gdf = pd.concat(
    [
        thailand_std_gdf,
        maharashtra_std_gdf,
    ],
    ignore_index=True,
)

combined_std_gdf = gpd.GeoDataFrame(
    combined_std_gdf,
    geometry="geometry",
    crs=thailand_std_gdf.crs,
)

duplicate_id_mask = combined_std_gdf[
    "analysis_segment_id"
].duplicated(
    keep=False
)

if duplicate_id_mask.any():
    duplicate_id_examples = (
        combined_std_gdf.loc[
            duplicate_id_mask,
            "analysis_segment_id",
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    raise ValueError(
        "Duplicate analysis_segment_id values were found. "
        "This must be resolved before continuing.\n"
        f"Examples: {duplicate_id_examples}"
    )

print("\nMerge completed successfully.")
print(f"Thailand standardized rows    : {len(thailand_std_gdf):,}")
print(f"Maharashtra standardized rows : {len(maharashtra_std_gdf):,}")
print(f"Combined standardized rows    : {len(combined_std_gdf):,}")
print(f"Combined CRS                  : {combined_std_gdf.crs}")


# ------------------------------------------------------------
# 6. Save the full standardized merged dataset
# ------------------------------------------------------------
for output_file in [
    STANDARDIZED_MERGED_OUTPUT_GPKG,
    STANDARDIZED_MERGED_OUTPUT_CSV,
]:
    if os.path.exists(output_file):
        os.remove(output_file)

combined_std_gdf.to_file(
    STANDARDIZED_MERGED_OUTPUT_GPKG,
    layer="standardized_road_safety",
    driver="GPKG",
)

combined_std_gdf.drop(
    columns="geometry",
    errors="ignore",
).to_csv(
    STANDARDIZED_MERGED_OUTPUT_CSV,
    index=False,
)


# ------------------------------------------------------------
# 7. Select fields needed for the next processing stages
# ------------------------------------------------------------
final_column_order = [
    "analysis_segment_id",
    "dataset",
    "road_name",
    "no_of_result_segments",
    "shape_length",
    "road_class",
    "sample_size_avg",
    "sample_size_total",
    "weighted_sample",
    "percentile",
    "ranked_percentile",
    "speed_limit",
    "number_over_limit",
    "percent_over_limit",
    "median_speed",
    "p85_speed",
    "land_use",
    "street_image_link",
    "geometry",
    "inv_percentile",
    "percentile_band",
    "analysis_status",
]

selected_gdf = combined_std_gdf[
    [
        "analysis_segment_id",
        "dataset",
        "road_name",
        "no_of_result_segments",
        "shape_length",
        "road_class_final",
        "sample_size_avg",
        "sample_size_total",
        "weighted_sample",
        "percentile",
        "ranked_percentile",
        "speed_limit",
        "number_over_limit",
        "percent_over_limit",
        "median_speed",
        "p85_speed",
        "land_use",
        "street_image_link",
        "geometry",
        "inv_percentile",
        "percentile_band",
        "analysis_status",
    ]
].copy()

selected_gdf = selected_gdf.rename(
    columns={
        "road_class_final": "road_class",
    }
)

selected_gdf = gpd.GeoDataFrame(
    selected_gdf,
    geometry="geometry",
    crs=combined_std_gdf.crs,
)

selected_gdf = selected_gdf[
    final_column_order
]


# ------------------------------------------------------------
# 8. Split: valid versus not included
# ------------------------------------------------------------
valid_mask = (
    selected_gdf["analysis_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("valid")
)

valid_gdf = selected_gdf.loc[
    valid_mask
].copy()

notincluded_gdf = selected_gdf.loc[
    ~valid_mask
].copy()


# ------------------------------------------------------------
# 9. Remove old output files before saving
# ------------------------------------------------------------
for output_file in [
    FIRST_DATA_PREPARATION_RESULT_VALID,
    FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED,
]:
    if os.path.exists(output_file):
        os.remove(output_file)


# ------------------------------------------------------------
# 10. Save valid and not-included outputs
# ------------------------------------------------------------
valid_gdf.to_file(
    FIRST_DATA_PREPARATION_RESULT_VALID,
    layer="ADB_Innovation_merge_valid",
    driver="GPKG",
)

notincluded_gdf.to_file(
    FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED,
    layer="ADB_Innovation_merge_notincluded",
    driver="GPKG",
)


# ------------------------------------------------------------
# 11. Validation summary
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("STEP 1 COMPLETED — LOCAL COLAB STORAGE")
print("=" * 78)

print(f"Selected output columns  : {len(selected_gdf.columns):,}")
print(f"Total merged rows        : {len(selected_gdf):,}")
print(f"Valid rows               : {len(valid_gdf):,}")
print(f"Not-included rows        : {len(notincluded_gdf):,}")

print("\nRows by dataset and output group:")
display(
    selected_gdf.assign(
        output_group=np.where(
            valid_mask,
            "Valid",
            "Not included",
        )
    )
    .groupby(
        [
            "dataset",
            "output_group",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="segment_count"
    )
)

print("\nRanked percentile summary:")
display(
    selected_gdf
    .groupby("dataset")["ranked_percentile"]
    .agg(
        [
            "count",
            "min",
            "max",
            "mean",
            "median",
        ]
    )
    .reset_index()
)

print("\nSaved outputs:")
print(f"1. Full merged GeoPackage : {STANDARDIZED_MERGED_OUTPUT_GPKG}")
print(f"2. Full merged CSV        : {STANDARDIZED_MERGED_OUTPUT_CSV}")
print(f"3. Valid GeoPackage       : {FIRST_DATA_PREPARATION_RESULT_VALID}")
print(f"4. Not-included GeoPackage: {FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED}")

## Add Derived Intersection Data

In [ ]:
# =====================================================================
# STEP 2 — ADD DERIVED INTERSECTION DATA
#
# Calculation inputs:
# - FIRST_DATA_PREPARATION_RESULT_VALID
# - FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED
#
# Output:
# - Valid records only, with:
#       intersection_count
#       has_intersection
#       intersection_count_per_km
#
# Local output:
# /content/2. Add Derived Intersection Data/
# ADB_Innovation_merge_valid_1.gpkg
# =====================================================================


# ---------------------------------------------------------------------
# INTERSECTION SETTINGS
# ---------------------------------------------------------------------
PROJECTED_CRS = "EPSG:3857"

# Endpoint coordinates within the same 50 m grid are treated as one node.
SNAP_TOLERANCE_M = 50

# A detected intersection point within 5 m of an endpoint
# is treated as an endpoint intersection.
ENDPOINT_TOLERANCE_M = 5

ROAD_CLASSES_TO_KEEP = [
    "primary",
    "secondary",
    "motorway",
    "trunk",
]

PAIR_CHUNK_SIZE = 50_000


# ---------------------------------------------------------------------
# VALIDATE INPUT FILES
# ---------------------------------------------------------------------
for input_file in [
    FIRST_DATA_PREPARATION_RESULT_VALID,
    FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED,
]:
    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input file not found:\n{input_file}\n\n"
            "Please run Step 1 first using the updated Cell 2 paths."
        )


# ---------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def get_available_columns(gpkg_path, layer_name):
    layer_info = pyogrio.read_info(
        gpkg_path,
        layer=layer_name,
    )

    return list(layer_info["fields"])


def point_key_xy(x, y, tolerance_m=SNAP_TOLERANCE_M):
    return (
        round(float(x) / tolerance_m),
        round(float(y) / tolerance_m),
    )


def point_key(point, tolerance_m=SNAP_TOLERANCE_M):
    return point_key_xy(
        point.x,
        point.y,
        tolerance_m,
    )


def get_endpoint_coords(geom):
    if geom is None or geom.is_empty:
        return []

    endpoints = []

    if geom.geom_type == "LineString":
        coords = list(geom.coords)

        if len(coords) >= 2:
            endpoints.append((coords[0][0], coords[0][1]))
            endpoints.append((coords[-1][0], coords[-1][1]))

    elif geom.geom_type == "MultiLineString":
        for part in geom.geoms:
            endpoints.extend(
                get_endpoint_coords(part)
            )

    return endpoints


def is_near_segment_endpoint(
    point_geom,
    endpoint_coords,
    tolerance_m,
):
    tolerance_sq = tolerance_m ** 2

    for x, y in endpoint_coords:
        dx = point_geom.x - x
        dy = point_geom.y - y

        if (dx * dx + dy * dy) <= tolerance_sq:
            return True

    return False


def extract_points_from_intersection(intersection_geom):
    if intersection_geom is None or intersection_geom.is_empty:
        return []

    if intersection_geom.geom_type == "Point":
        return [intersection_geom]

    if intersection_geom.geom_type == "MultiPoint":
        return list(intersection_geom.geoms)

    if intersection_geom.geom_type == "GeometryCollection":
        points = []

        for part in intersection_geom.geoms:
            points.extend(
                extract_points_from_intersection(part)
            )

        return points

    return []


def safe_vectorized_intersection(
    geom_array_a,
    geom_array_b,
):
    try:
        return shapely.intersection(
            geom_array_a,
            geom_array_b,
        )

    except Exception:
        results = []

        for geom_a, geom_b in zip(
            geom_array_a,
            geom_array_b,
        ):
            try:
                results.append(
                    geom_a.intersection(geom_b)
                )
            except Exception:
                results.append(None)

        return results


def load_minimal_road_data(gpkg_path, dataset_label):
    layer_name = get_first_layer_name(gpkg_path)

    available_columns = get_available_columns(
        gpkg_path,
        layer_name,
    )

    required_columns = [
        "analysis_segment_id",
        "dataset",
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in available_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns in {dataset_label}:\n"
            + "\n".join(missing_columns)
        )

    if "road_class" in available_columns:
        road_class_column = "road_class"

    elif "road_class_final" in available_columns:
        road_class_column = "road_class_final"

    else:
        raise ValueError(
            f"No road class column found in {dataset_label}."
        )

    selected_columns = [
        "dataset",
        "analysis_segment_id",
        road_class_column,
    ]

    road_gdf = gpd.read_file(
        gpkg_path,
        layer=layer_name,
        columns=selected_columns,
        engine="pyogrio",
    )

    road_gdf = road_gdf.rename(
        columns={
            road_class_column: "road_class",
        }
    )

    road_gdf = road_gdf[
        road_gdf.geometry.notna()
        & road_gdf.geometry.geom_type.isin(
            ["LineString", "MultiLineString"]
        )
    ].copy()

    before_filter = len(road_gdf)

    road_gdf = road_gdf[
        road_gdf["road_class"].isin(
            ROAD_CLASSES_TO_KEEP
        )
    ].copy()

    road_gdf = road_gdf[
        road_gdf["analysis_segment_id"].notna()
    ].reset_index(drop=True)

    print("\n" + "=" * 80)
    print(f"{dataset_label.upper()} DATA")
    print("=" * 80)
    print(f"Rows before class filter: {before_filter:,}")
    print(f"Rows after class filter : {len(road_gdf):,}")
    print(f"CRS                     : {road_gdf.crs}")

    return road_gdf


def calculate_intersections_for_group(
    group_gdf,
    group_name,
):
    print("\n" + "=" * 80)
    print(f"PROCESSING: {group_name}")
    print("=" * 80)

    work = group_gdf.to_crs(
        PROJECTED_CRS
    ).reset_index(drop=True).copy()

    n_segments = len(work)

    geoms = np.asarray(
        work.geometry.values,
        dtype=object,
    )

    endpoint_coords_by_segment = []
    endpoint_node_segments = {}

    for segment_index, geom in enumerate(geoms):
        endpoint_coords = get_endpoint_coords(geom)

        endpoint_coords_by_segment.append(
            endpoint_coords
        )

        for x, y in endpoint_coords:
            node_key = point_key_xy(x, y)

            if node_key not in endpoint_node_segments:
                endpoint_node_segments[node_key] = set()

            endpoint_node_segments[node_key].add(
                segment_index
            )

    spatial_index = work.sindex

    try:
        raw_pairs = spatial_index.query(
            work.geometry,
            predicate="intersects",
        )
    except Exception:
        raw_pairs = spatial_index.query_bulk(
            work.geometry,
            predicate="intersects",
        )

    raw_pairs = np.asarray(raw_pairs)

    if raw_pairs.size == 0:
        pair_left = np.array([], dtype=int)
        pair_right = np.array([], dtype=int)

    else:
        pair_left = raw_pairs[0].astype(int)
        pair_right = raw_pairs[1].astype(int)

        keep_mask = pair_left < pair_right

        pair_left = pair_left[keep_mask]
        pair_right = pair_right[keep_mask]

    candidate_pair_count = len(pair_left)

    print(f"Road segments      : {n_segments:,}")
    print(f"Candidate pairs    : {candidate_pair_count:,}")

    segment_intersection_keys = [
        set()
        for _ in range(n_segments)
    ]

    raw_point_hits = 0
    counted_point_hits = 0
    skipped_continuations = 0

    for start_idx in tqdm(
        range(
            0,
            candidate_pair_count,
            PAIR_CHUNK_SIZE,
        ),
        desc=f"Checking intersections: {group_name}",
    ):
        end_idx = min(
            start_idx + PAIR_CHUNK_SIZE,
            candidate_pair_count,
        )

        left_chunk = pair_left[
            start_idx:end_idx
        ]

        right_chunk = pair_right[
            start_idx:end_idx
        ]

        intersection_geometries = (
            safe_vectorized_intersection(
                geoms[left_chunk],
                geoms[right_chunk],
            )
        )

        for segment_i, segment_j, intersection_geom in zip(
            left_chunk,
            right_chunk,
            intersection_geometries,
        ):
            points = extract_points_from_intersection(
                intersection_geom
            )

            if not points:
                continue

            for point_geom in points:
                raw_point_hits += 1

                node_key = point_key(point_geom)

                is_endpoint_i = is_near_segment_endpoint(
                    point_geom,
                    endpoint_coords_by_segment[segment_i],
                    ENDPOINT_TOLERANCE_M,
                )

                is_endpoint_j = is_near_segment_endpoint(
                    point_geom,
                    endpoint_coords_by_segment[segment_j],
                    ENDPOINT_TOLERANCE_M,
                )

                # Two endpoints meeting at one node with only
                # two incident segments are treated as a continuation.
                if is_endpoint_i and is_endpoint_j:
                    incident_count = len(
                        endpoint_node_segments.get(
                            node_key,
                            set(),
                        )
                    )

                    if incident_count <= 2:
                        skipped_continuations += 1
                        continue

                # Keep:
                # - T-junctions
                # - road crossings
                # - endpoint junctions with 3+ segments
                segment_intersection_keys[
                    segment_i
                ].add(node_key)

                segment_intersection_keys[
                    segment_j
                ].add(node_key)

                counted_point_hits += 1

    group_result = work[
        ["analysis_segment_id"]
    ].copy()

    group_result["intersection_count"] = [
        len(intersection_keys)
        for intersection_keys in segment_intersection_keys
    ]

    group_result["has_intersection"] = (
        group_result["intersection_count"] > 0
    ).astype(int)

    group_summary = {
        "dataset": group_name,
        "segments": n_segments,
        "candidate_pairs": candidate_pair_count,
        "raw_point_hits": raw_point_hits,
        "skipped_2_endpoint_continuations": (
            skipped_continuations
        ),
        "counted_point_hits": counted_point_hits,
        "segments_with_intersection": int(
            group_result["has_intersection"].sum()
        ),
        "maximum_intersection_count": int(
            group_result["intersection_count"].max()
        ),
    }

    del work
    del geoms
    del spatial_index
    del endpoint_coords_by_segment
    del endpoint_node_segments
    del segment_intersection_keys

    gc.collect()

    return group_result, group_summary


# ---------------------------------------------------------------------
# LOAD VALID + NOT-INCLUDED ROADS FOR INTERSECTION CALCULATION
# ---------------------------------------------------------------------
valid_calc_gdf = load_minimal_road_data(
    FIRST_DATA_PREPARATION_RESULT_VALID,
    "Valid",
)

notincluded_calc_gdf = load_minimal_road_data(
    FIRST_DATA_PREPARATION_RESULT_NOTINCLUDED,
    "Not Included",
)

if valid_calc_gdf.crs != notincluded_calc_gdf.crs:
    notincluded_calc_gdf = notincluded_calc_gdf.to_crs(
        valid_calc_gdf.crs
    )

calc_gdf = pd.concat(
    [
        valid_calc_gdf,
        notincluded_calc_gdf,
    ],
    ignore_index=True,
)

calc_gdf = gpd.GeoDataFrame(
    calc_gdf,
    geometry="geometry",
    crs=valid_calc_gdf.crs,
)

if calc_gdf["analysis_segment_id"].duplicated().any():
    raise ValueError(
        "Duplicate analysis_segment_id found. "
        "It must be unique for this calculation."
    )

print("\n" + "=" * 80)
print("MERGED TEMPORARY DATA FOR INTERSECTION CALCULATION")
print("=" * 80)
print(f"Total rows: {len(calc_gdf):,}")

del valid_calc_gdf
del notincluded_calc_gdf

gc.collect()


# ---------------------------------------------------------------------
# RUN INTERSECTION ANALYSIS BY DATASET
# ---------------------------------------------------------------------
dataset_values = sorted(
    calc_gdf["dataset"]
    .dropna()
    .unique()
    .tolist()
)

all_intersection_results = []
all_group_summary = []

for dataset_value in dataset_values:
    current_group_gdf = calc_gdf[
        calc_gdf["dataset"] == dataset_value
    ].copy()

    current_result, current_summary = (
        calculate_intersections_for_group(
            current_group_gdf,
            dataset_value,
        )
    )

    all_intersection_results.append(
        current_result
    )

    all_group_summary.append(
        current_summary
    )

    del current_group_gdf
    del current_result

    gc.collect()


intersection_counts_df = pd.concat(
    all_intersection_results,
    ignore_index=True,
)

intersection_counts_df = (
    intersection_counts_df
    .groupby(
        "analysis_segment_id",
        as_index=False,
    )
    .agg(
        intersection_count=(
            "intersection_count",
            "max",
        ),
        has_intersection=(
            "has_intersection",
            "max",
        ),
    )
)

summary_df = pd.DataFrame(all_group_summary)

print("\nIntersection processing summary:")
display(summary_df)

del calc_gdf
del all_intersection_results
del all_group_summary

gc.collect()


# ---------------------------------------------------------------------
# LOAD FULL VALID DATA AND MERGE INTERSECTION RESULTS
# ---------------------------------------------------------------------
valid_layer_name = get_first_layer_name(
    FIRST_DATA_PREPARATION_RESULT_VALID
)

ADB_Innovation_merge_valid_1 = gpd.read_file(
    FIRST_DATA_PREPARATION_RESULT_VALID,
    layer=valid_layer_name,
    engine="pyogrio",
)

if ADB_Innovation_merge_valid_1[
    "analysis_segment_id"
].duplicated().any():
    raise ValueError(
        "Duplicate analysis_segment_id found in Valid dataset."
    )

for column_name in [
    "intersection_count",
    "has_intersection",
    "intersection_count_per_km",
]:
    if column_name in ADB_Innovation_merge_valid_1.columns:
        ADB_Innovation_merge_valid_1 = (
            ADB_Innovation_merge_valid_1.drop(
                columns=column_name
            )
        )

geometry_column = ADB_Innovation_merge_valid_1.geometry.name
input_crs = ADB_Innovation_merge_valid_1.crs

ADB_Innovation_merge_valid_1 = (
    ADB_Innovation_merge_valid_1.merge(
        intersection_counts_df,
        on="analysis_segment_id",
        how="left",
        validate="one_to_one",
    )
)

ADB_Innovation_merge_valid_1 = gpd.GeoDataFrame(
    ADB_Innovation_merge_valid_1,
    geometry=geometry_column,
    crs=input_crs,
)

ADB_Innovation_merge_valid_1[
    "intersection_count"
] = (
    ADB_Innovation_merge_valid_1[
        "intersection_count"
    ]
    .fillna(0)
    .astype(int)
)

ADB_Innovation_merge_valid_1[
    "has_intersection"
] = (
    ADB_Innovation_merge_valid_1[
        "has_intersection"
    ]
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------------------
# ADD INTERSECTION DENSITY PER KM
# Formula:
# intersection_count_per_km =
#     intersection_count / (shape_length in metres / 1,000)
# ---------------------------------------------------------------------
if "shape_length" not in ADB_Innovation_merge_valid_1.columns:
    raise ValueError(
        "Column 'shape_length' was not found. "
        "It is required to calculate intersection_count_per_km."
    )

ADB_Innovation_merge_valid_1["shape_length"] = pd.to_numeric(
    ADB_Innovation_merge_valid_1["shape_length"],
    errors="coerce",
)

valid_length_m = ADB_Innovation_merge_valid_1[
    "shape_length"
].where(
    ADB_Innovation_merge_valid_1["shape_length"] > 0
)

ADB_Innovation_merge_valid_1[
    "intersection_count_per_km"
] = (
    ADB_Innovation_merge_valid_1[
        "intersection_count"
    ]
    * 1000
    / valid_length_m
).round(4)

print("\nIntersection density per km summary:")
display(
    ADB_Innovation_merge_valid_1[
        "intersection_count_per_km"
    ].describe()
)


# ---------------------------------------------------------------------
# SAVE INTERSECTION RESULT
# ---------------------------------------------------------------------
for extension in ["", "-wal", "-shm"]:
    output_file = LOCAL_INTERSECTION_RESULT + extension

    if os.path.exists(output_file):
        os.remove(output_file)

ADB_Innovation_merge_valid_1.to_file(
    LOCAL_INTERSECTION_RESULT,
    layer="ADB_Innovation_merge_valid_1",
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("\nFirst 5 rows:")
display(
    ADB_Innovation_merge_valid_1.head()
)

print("\nSaved locally:")
print(LOCAL_INTERSECTION_RESULT)


# ---------------------------------------------------------------------
# CLEAR UNNECESSARY DATA FROM RAM
# ---------------------------------------------------------------------
del intersection_counts_df
del summary_df
del ADB_Innovation_merge_valid_1

gc.collect()

## Add External Data for Detailed POI near the Street Segment

### Download and Extract data from OSM

In [ ]:
# ==============================================================================
# STEP 3A — DOWNLOAD RAW OPENSTREETMAP PBF DATA
#
# Downloads:
#   1. Thailand OSM PBF
#   2. India Western Zone OSM PBF
#      Used as the source file for Maharashtra.
#
# Output:
#   /content/3. OSM POI Data/Raw OSM PBF/
#
# This step downloads only.
# POI extraction and spatial filtering will be done in Step 3B.
# ==============================================================================


# ------------------------------------------------------------------------------
# VALIDATE REQUIRED VARIABLES FROM CELL 2
# ------------------------------------------------------------------------------
required_variables = [
    "EXTERNAL_DATASET_FOLDER",
    "RAW_OSM_STORAGE_FOLDER",
    "REGION_CONFIG",
    "OSM_RAW_PBF_DOWNLOAD_SUMMARY_CSV",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

print("Using local Colab storage only.")
print(f"OSM workflow root: {EXTERNAL_DATASET_FOLDER}")


# ------------------------------------------------------------------------------
# DOWNLOAD SETTINGS
# ------------------------------------------------------------------------------
# Set to True only when you deliberately want to download fresh copies,
# even when valid local PBF files already exist.
FORCE_REDOWNLOAD_RAW_PBF = False

DOWNLOAD_CHUNK_SIZE = 1024 * 1024


# ------------------------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_file_size_mb(file_path):
    """
    Return file size in MB.
    """
    return round(
        os.path.getsize(file_path) / (1024 * 1024),
        2,
    )


def download_file_with_progress(
    url,
    output_path,
    chunk_size=DOWNLOAD_CHUNK_SIZE,
):
    """
    Download one PBF file with progress and safe temporary output.
    """

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True,
    )

    temporary_output_path = output_path + ".part"

    if os.path.exists(temporary_output_path):
        os.remove(temporary_output_path)

    print(f"\nDownloading from:\n{url}")
    print(f"\nSaving to:\n{output_path}")

    try:
        response = requests.get(
            url,
            stream=True,
            timeout=(30, 300),
        )

        response.raise_for_status()

        total_size = int(
            response.headers.get(
                "content-length",
                0,
            )
        )

        with open(
            temporary_output_path,
            "wb",
        ) as output_file:
            with tqdm(
                total=total_size if total_size > 0 else None,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="Downloading PBF",
                dynamic_ncols=True,
            ) as progress_bar:

                for chunk in response.iter_content(
                    chunk_size=chunk_size
                ):
                    if chunk:
                        output_file.write(chunk)
                        progress_bar.update(len(chunk))

        if os.path.getsize(temporary_output_path) == 0:
            raise IOError(
                "Downloaded file is empty."
            )

        os.replace(
            temporary_output_path,
            output_path,
        )

    except Exception:
        if os.path.exists(temporary_output_path):
            os.remove(temporary_output_path)

        raise

    print(
        "PBF download completed: "
        f"{get_file_size_mb(output_path):,.2f} MB"
    )


def prepare_local_pbf(
    region_key,
    force_redownload=False,
):
    """
    Reuse a valid local PBF if available.
    Otherwise, download it into Raw OSM PBF storage.
    """

    if region_key not in REGION_CONFIG:
        raise ValueError(
            f"Unknown region_key: {region_key}"
        )

    region = REGION_CONFIG[region_key]

    storage_pbf_path = region["storage_pbf_path"]

    print("\n" + "=" * 88)
    print(f"PREPARING RAW PBF — {region['region_name'].upper()}")
    print("=" * 88)

    local_file_exists = (
        os.path.exists(storage_pbf_path)
        and os.path.getsize(storage_pbf_path) > 0
    )

    if local_file_exists and not force_redownload:
        print("Stored local PBF found. Reusing existing file.")
        print(storage_pbf_path)

        download_status = "Reused existing file"

    else:
        if local_file_exists and force_redownload:
            print(
                "Force redownload enabled. "
                "Replacing existing local PBF."
            )

        else:
            print("Stored local PBF not found.")

        download_file_with_progress(
            url=region["pbf_url"],
            output_path=storage_pbf_path,
        )

        download_status = "Downloaded"

    return {
        "region_key": region_key,
        "region_name": region["region_name"],
        "download_status": download_status,
        "pbf_url": region["pbf_url"],
        "local_pbf_path": storage_pbf_path,
        "file_size_mb": get_file_size_mb(
            storage_pbf_path
        ),
    }


# ------------------------------------------------------------------------------
# DOWNLOAD RAW PBF FILES
# ------------------------------------------------------------------------------
REGION_KEYS_TO_DOWNLOAD = [
    "thailand",
    "maharashtra",
]

download_summary_records = []

for region_key in REGION_KEYS_TO_DOWNLOAD:
    summary_record = prepare_local_pbf(
        region_key=region_key,
        force_redownload=FORCE_REDOWNLOAD_RAW_PBF,
    )

    download_summary_records.append(
        summary_record
    )

    gc.collect()


# ------------------------------------------------------------------------------
# SAVE AND DISPLAY DOWNLOAD SUMMARY
# ------------------------------------------------------------------------------
download_summary_df = pd.DataFrame(
    download_summary_records
)

download_summary_df.to_csv(
    OSM_RAW_PBF_DOWNLOAD_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 88)
print("STEP 3A COMPLETED — RAW OSM PBF DOWNLOAD")
print("=" * 88)

display(download_summary_df)

print("\nRaw OSM PBF storage folder:")
print(RAW_OSM_STORAGE_FOLDER)

print("\nDownload summary CSV:")
print(OSM_RAW_PBF_DOWNLOAD_SUMMARY_CSV)

### Add Surounding POI data to the Road Segment

In [ ]:
# ==============================================================================
# STEP 3B — EXTRACT OSM POINT-OF-INTEREST DATA
#           AND ADD ROAD-LEVEL POI METRICS
#
# Input road file:
#   POI_ROAD_INPUT
#
# Raw OSM PBF inputs:
#   - Thailand
#   - India Western Zone (used for Maharashtra)
#
# POI groups and distance limits:
#   - public_transport : 250 m
#   - office           : 250 m
#   - school           : 250 m
#   - commercial       : 250 m
#   - industrial       : 1,000 m
#
# Added road-level columns:
#   <poi_group>_count_<distance>m
#   <poi_group>_min_distance_m_<distance>m
#   <poi_group>_median_distance_m_<distance>m
#
# Distance logic:
#   - Calculated from each POI point to the complete road segment geometry.
#   - This is the shortest geometric distance:
#       * perpendicular distance when the perpendicular point falls on the road;
#       * otherwise, distance to the nearest road endpoint.
#
# Output:
#   SURROUNDING_LAND_USE_RESULT
# ==============================================================================


# ------------------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------------------
POI_DISTANCE_SETTINGS = {
    "public_transport": 250,
    "office": 250,
    "school": 250,
    "commercial": 250,
    "industrial": 1000,
}

EXPECTED_DATASET_VALUES = [
    "Thailand",
    "Maharashtra",
]

POI_GROUPS = list(
    POI_DISTANCE_SETTINGS.keys()
)

POI_OUTPUT_COLUMNS = [
    "OSM_ID",
    "OSM_Type",
    "POI_Group",
    "Feature_Type",
    "Name",
    "Category",
    "Amenity",
    "Shop",
    "Building",
    "Office",
    "Landuse",
    "Highway",
    "Railway",
    "Public_Transport",
    "Street",
    "City",
    "Province_or_State",
    "Postcode",
    "Latitude",
    "Longitude",
]

PBF_PROGRESS_EVERY = 100_000

# Bounding-box prefilter is slightly larger than the maximum search radius.
# Exact road-to-POI filtering is still performed later using the defined radius.
EXTRACTION_BBOX_BUFFER_M = (
    max(POI_DISTANCE_SETTINGS.values()) + 100
)


# ------------------------------------------------------------------------------
# VALIDATE REQUIRED VARIABLES
# ------------------------------------------------------------------------------
required_variables = [
    "POI_ROAD_INPUT",
    "SURROUNDING_LAND_USE_RESULT",
    "POI_METRICS_SUMMARY_CSV",
    "REGION_CONFIG",
    "POI_EXTRACT_OUTPUT_CONFIG",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

print("Using local Colab storage only.")


# ------------------------------------------------------------------------------
# VALIDATE INPUT FILES
# ------------------------------------------------------------------------------
required_files = [
    POI_ROAD_INPUT,
    REGION_CONFIG["thailand"]["storage_pbf_path"],
    REGION_CONFIG["maharashtra"]["storage_pbf_path"],
]

print("\n" + "=" * 90)
print("CHECKING INPUT FILES")
print("=" * 90)

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Required file was not found:\n{file_path}\n\n"
            "Please run Step 3A first for the raw OSM PBF files."
        )

    if os.path.getsize(file_path) == 0:
        raise ValueError(
            f"Required file is empty:\n{file_path}"
        )

    print(f"Found: {file_path}")


# ------------------------------------------------------------------------------
# GENERIC HELPERS
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def normalize_tag_value(value):
    """
    Convert one OSM tag value into cleaned lowercase text.
    """
    if value is None:
        return None

    return str(value).strip().lower()


def is_positive_transport_tag(value):
    """
    Check common positive OSM transport values.
    """
    return value in [
        "yes",
        "designated",
        "official",
        "permissive",
    ]


def create_local_metric_crs(roads_wgs84):
    """
    Create an azimuthal equidistant CRS centered on the road study extent.

    Units are metres, enabling distance calculations against
    the full road geometry.
    """
    min_x, min_y, max_x, max_y = roads_wgs84.total_bounds

    lon_center = (min_x + max_x) / 2
    lat_center = (min_y + max_y) / 2

    return (
        f"+proj=aeqd "
        f"+lat_0={lat_center} "
        f"+lon_0={lon_center} "
        f"+datum=WGS84 "
        f"+units=m "
        f"+no_defs"
    )


def create_wgs84_bbox(
    roads_wgs84,
    buffer_m=EXTRACTION_BBOX_BUFFER_M,
):
    """
    Build a WGS84 bounding box around the road extent.

    This is used only to reduce retained OSM POI records in memory.
    Final inclusion remains based on exact metric distance to each road.
    """
    min_lon, min_lat, max_lon, max_lat = (
        roads_wgs84.total_bounds
    )

    lat_center = (min_lat + max_lat) / 2

    lat_buffer_deg = buffer_m / 110_574

    lon_scale = max(
        111_320 * abs(math.cos(math.radians(lat_center))),
        1,
    )

    lon_buffer_deg = buffer_m / lon_scale

    return (
        min_lon - lon_buffer_deg,
        min_lat - lat_buffer_deg,
        max_lon + lon_buffer_deg,
        max_lat + lat_buffer_deg,
    )


def point_inside_bbox(
    longitude,
    latitude,
    bbox,
):
    """
    Check whether a WGS84 coordinate is inside the extraction bounding box.
    """
    min_lon, min_lat, max_lon, max_lat = bbox

    return (
        min_lon <= longitude <= max_lon
        and min_lat <= latitude <= max_lat
    )


def safe_vectorized_distance(
    geometry_array_a,
    geometry_array_b,
):
    """
    Calculate vectorized Shapely distances.
    Falls back to one-by-one calculation if needed.
    """
    try:
        return np.asarray(
            shapely.distance(
                geometry_array_a,
                geometry_array_b,
            ),
            dtype=float,
        )

    except Exception:
        return np.asarray(
            [
                geometry_a.distance(geometry_b)
                if (
                    geometry_a is not None
                    and geometry_b is not None
                )
                else np.nan
                for geometry_a, geometry_b in zip(
                    geometry_array_a,
                    geometry_array_b,
                )
            ],
            dtype=float,
        )


# ------------------------------------------------------------------------------
# OSM FEATURE CLASSIFICATION
# ------------------------------------------------------------------------------
def classify_public_transport_feature(tags):
    """
    Possible Feature_Type values:
    - bus_stop
    - bus_station
    - train_station
    - metro_station
    - public_transport_facility
    """
    amenity = normalize_tag_value(tags.get("amenity"))
    highway = normalize_tag_value(tags.get("highway"))
    railway = normalize_tag_value(tags.get("railway"))
    public_transport = normalize_tag_value(
        tags.get("public_transport")
    )

    station = normalize_tag_value(tags.get("station"))
    building = normalize_tag_value(tags.get("building"))

    bus = normalize_tag_value(tags.get("bus"))
    train = normalize_tag_value(tags.get("train"))

    subway = normalize_tag_value(tags.get("subway"))
    metro = normalize_tag_value(tags.get("metro"))
    light_rail = normalize_tag_value(tags.get("light_rail"))
    monorail = normalize_tag_value(tags.get("monorail"))

    if (
        station in [
            "subway",
            "metro",
            "light_rail",
            "monorail",
        ]
        or is_positive_transport_tag(subway)
        or is_positive_transport_tag(metro)
        or is_positive_transport_tag(light_rail)
        or is_positive_transport_tag(monorail)
    ):
        return "metro_station"

    if (
        amenity == "bus_station"
        or (
            public_transport == "station"
            and is_positive_transport_tag(bus)
        )
    ):
        return "bus_station"

    if (
        highway == "bus_stop"
        or (
            public_transport in [
                "platform",
                "stop_position",
            ]
            and is_positive_transport_tag(bus)
        )
    ):
        return "bus_stop"

    if (
        railway in [
            "station",
            "halt",
            "tram_stop",
        ]
        or building == "train_station"
        or (
            public_transport == "station"
            and is_positive_transport_tag(train)
        )
    ):
        return "train_station"

    if (
        public_transport in [
            "station",
            "platform",
            "stop_position",
        ]
        or amenity == "ferry_terminal"
    ):
        return "public_transport_facility"

    return None


def classify_commercial_feature(tags):
    """
    Possible Feature_Type values:
    - shop
    - market
    - mall
    - cafe
    - restaurant
    - retail_building
    """
    shop = normalize_tag_value(tags.get("shop"))
    amenity = normalize_tag_value(tags.get("amenity"))
    building = normalize_tag_value(tags.get("building"))

    if (
        shop == "mall"
        or building == "mall"
        or amenity == "shopping_mall"
    ):
        return "mall"

    if amenity == "marketplace":
        return "market"

    if amenity == "cafe":
        return "cafe"

    if amenity == "restaurant":
        return "restaurant"

    if shop is not None and shop != "no":
        return "shop"

    if building == "retail":
        return "retail_building"

    return None


def get_matching_poi_groups(tags):
    """
    Return all matching POI groups for one OSM object's tag dictionary.

    One OSM feature can be included in more than one group
    when it genuinely matches multiple POI definitions.
    """
    matched_groups = {}

    amenity = normalize_tag_value(tags.get("amenity"))
    building = normalize_tag_value(tags.get("building"))
    office = normalize_tag_value(tags.get("office"))
    landuse = normalize_tag_value(tags.get("landuse"))

    public_transport_type = (
        classify_public_transport_feature(tags)
    )

    commercial_type = classify_commercial_feature(
        tags
    )

    if public_transport_type is not None:
        matched_groups[
            "public_transport"
        ] = public_transport_type

    if (
        building in [
            "office",
            "commercial",
        ]
        or office is not None
        or landuse == "commercial"
    ):
        matched_groups["office"] = "office"

    if amenity in [
        "school",
        "university",
    ]:
        matched_groups["school"] = "school"

    if commercial_type is not None:
        matched_groups[
            "commercial"
        ] = commercial_type

    if landuse == "industrial":
        matched_groups["industrial"] = "industrial"

    return matched_groups


def get_output_category(
    tags,
    poi_group,
    feature_type,
):
    """
    Generate a concise category label for extracted POI audit files.
    """
    amenity = normalize_tag_value(tags.get("amenity"))
    highway = normalize_tag_value(tags.get("highway"))
    shop = normalize_tag_value(tags.get("shop"))
    building = normalize_tag_value(tags.get("building"))
    landuse = normalize_tag_value(tags.get("landuse"))
    public_transport = normalize_tag_value(
        tags.get("public_transport")
    )
    railway = normalize_tag_value(tags.get("railway"))
    station = normalize_tag_value(tags.get("station"))

    if poi_group == "school":
        return f"amenity={amenity}"

    if poi_group == "office":
        if normalize_tag_value(tags.get("office")) is not None:
            return "office=*"

        if building is not None:
            return f"building={building}"

        return f"landuse={landuse}"

    if poi_group == "industrial":
        return f"landuse={landuse}"

    if poi_group == "commercial":
        if feature_type == "shop":
            return f"shop={shop}"

        if feature_type == "market":
            return "amenity=marketplace"

        if feature_type == "mall":
            if shop == "mall":
                return "shop=mall"

            if building == "mall":
                return "building=mall"

            return "amenity=shopping_mall"

        if feature_type == "cafe":
            return "amenity=cafe"

        if feature_type == "restaurant":
            return "amenity=restaurant"

        return "building=retail"

    if poi_group == "public_transport":
        if feature_type == "bus_stop":
            if highway == "bus_stop":
                return "highway=bus_stop"

            return f"public_transport={public_transport}"

        if feature_type == "bus_station":
            if amenity == "bus_station":
                return "amenity=bus_station"

            return f"public_transport={public_transport}"

        if feature_type == "train_station":
            if railway is not None:
                return f"railway={railway}"

            return "train_station_related"

        if feature_type == "metro_station":
            if station is not None:
                return f"station={station}"

            return "metro_related"

        return "public_transport_related"

    return None


# ------------------------------------------------------------------------------
# OSM PBF PARSER
# ------------------------------------------------------------------------------
class OSMPOIHandler(osmium.SimpleHandler):
    """
    Parse one PBF once and collect all configured POI groups.
    """

    def __init__(
        self,
        extraction_bbox,
        progress_every=PBF_PROGRESS_EVERY,
    ):
        super().__init__()

        self.extraction_bbox = extraction_bbox
        self.progress_every = progress_every

        self.records_by_group = {
            poi_group: []
            for poi_group in POI_GROUPS
        }

        self.total_seen = 0
        self.nodes_seen = 0
        self.ways_seen = 0
        self.relations_seen = 0

        self.progress_bar = tqdm(
            total=None,
            unit=" objects",
            desc="Parsing OSM PBF",
            dynamic_ncols=True,
        )

    def update_progress(self):
        self.total_seen += 1

        if self.total_seen % self.progress_every == 0:
            self.progress_bar.update(
                self.progress_every
            )

            total_matched = sum(
                len(records)
                for records in self.records_by_group.values()
            )

            self.progress_bar.set_postfix(
                {
                    "nodes": self.nodes_seen,
                    "ways": self.ways_seen,
                    "matched": total_matched,
                }
            )

    def close_progress(self):
        remaining = (
            self.total_seen
            - self.progress_bar.n
        )

        if remaining > 0:
            self.progress_bar.update(remaining)

        total_matched = sum(
            len(records)
            for records in self.records_by_group.values()
        )

        self.progress_bar.set_postfix(
            {
                "nodes": self.nodes_seen,
                "ways": self.ways_seen,
                "relations": self.relations_seen,
                "matched": total_matched,
            }
        )

        self.progress_bar.close()

    def save_record(
        self,
        osm_obj,
        osm_type,
        longitude,
        latitude,
        tags,
        poi_group,
        feature_type,
    ):
        if not point_inside_bbox(
            longitude,
            latitude,
            self.extraction_bbox,
        ):
            return

        self.records_by_group[poi_group].append(
            {
                "OSM_ID": osm_obj.id,
                "OSM_Type": osm_type,
                "POI_Group": poi_group,
                "Feature_Type": feature_type,
                "Name": tags.get("name"),
                "Category": get_output_category(
                    tags,
                    poi_group,
                    feature_type,
                ),
                "Amenity": tags.get("amenity"),
                "Shop": tags.get("shop"),
                "Building": tags.get("building"),
                "Office": tags.get("office"),
                "Landuse": tags.get("landuse"),
                "Highway": tags.get("highway"),
                "Railway": tags.get("railway"),
                "Public_Transport": tags.get(
                    "public_transport"
                ),
                "Street": tags.get("addr:street"),
                "City": tags.get("addr:city"),
                "Province_or_State": (
                    tags.get("addr:province")
                    or tags.get("addr:state")
                ),
                "Postcode": tags.get("addr:postcode"),
                "Latitude": latitude,
                "Longitude": longitude,
            }
        )

    def node(self, node):
        self.nodes_seen += 1
        self.update_progress()

        tags = {
            tag.k: tag.v
            for tag in node.tags
        }

        matched_groups = get_matching_poi_groups(
            tags
        )

        if len(matched_groups) == 0:
            return

        try:
            if not node.location.valid():
                return

            longitude = node.location.lon
            latitude = node.location.lat

        except Exception:
            return

        for poi_group, feature_type in (
            matched_groups.items()
        ):
            self.save_record(
                osm_obj=node,
                osm_type="node",
                longitude=longitude,
                latitude=latitude,
                tags=tags,
                poi_group=poi_group,
                feature_type=feature_type,
            )

    def way(self, way):
        self.ways_seen += 1
        self.update_progress()

        tags = {
            tag.k: tag.v
            for tag in way.tags
        }

        matched_groups = get_matching_poi_groups(
            tags
        )

        if len(matched_groups) == 0:
            return

        coordinates = []

        for way_node in way.nodes:
            try:
                if way_node.location.valid():
                    coordinates.append(
                        (
                            way_node.location.lon,
                            way_node.location.lat,
                        )
                    )
            except Exception:
                pass

        if len(coordinates) == 0:
            return

        try:
            if (
                len(coordinates) >= 4
                and coordinates[0] == coordinates[-1]
            ):
                geometry = Polygon(coordinates)

                if not geometry.is_valid:
                    geometry = geometry.buffer(0)

                point_geometry = geometry.centroid

                longitude = point_geometry.x
                latitude = point_geometry.y

            elif len(coordinates) >= 2:
                geometry = LineString(coordinates)

                point_geometry = geometry.interpolate(
                    0.5,
                    normalized=True,
                )

                longitude = point_geometry.x
                latitude = point_geometry.y

            else:
                longitude, latitude = coordinates[0]

        except Exception:
            return

        for poi_group, feature_type in (
            matched_groups.items()
        ):
            self.save_record(
                osm_obj=way,
                osm_type="way",
                longitude=longitude,
                latitude=latitude,
                tags=tags,
                poi_group=poi_group,
                feature_type=feature_type,
            )

    def relation(self, relation):
        self.relations_seen += 1
        self.update_progress()


# ------------------------------------------------------------------------------
# PREPARE EXTRACTED POI OUTPUT
# ------------------------------------------------------------------------------
def prepare_poi_output_dataframe(records):
    """
    Clean and standardize a single POI group extracted from OSM.
    """
    output_df = pd.DataFrame(
        records,
        columns=POI_OUTPUT_COLUMNS,
    )

    if output_df.empty:
        return output_df

    output_df["Latitude"] = pd.to_numeric(
        output_df["Latitude"],
        errors="coerce",
    )

    output_df["Longitude"] = pd.to_numeric(
        output_df["Longitude"],
        errors="coerce",
    )

    output_df = output_df.dropna(
        subset=[
            "Latitude",
            "Longitude",
        ]
    ).copy()

    output_df = output_df[
        output_df["Latitude"].between(-90, 90)
        & output_df["Longitude"].between(-180, 180)
    ].copy()

    output_df["Name"] = output_df["Name"].fillna(
        "Unnamed OSM Feature"
    )

    output_df = output_df.drop_duplicates(
        subset=[
            "OSM_Type",
            "OSM_ID",
            "POI_Group",
        ]
    )

    output_df = output_df.sort_values(
        by=[
            "Feature_Type",
            "Category",
            "Name",
            "OSM_Type",
            "OSM_ID",
        ],
        ascending=True,
        na_position="last",
    ).reset_index(drop=True)

    return output_df


def dataframe_to_poi_gdf(
    poi_df,
    poi_group,
    dataset_value,
):
    """
    Convert an extracted POI DataFrame to a WGS84 point GeoDataFrame.
    """
    if poi_df.empty:
        return gpd.GeoDataFrame(
            poi_df.copy(),
            geometry=[],
            crs="EPSG:4326",
        )

    poi_gdf = gpd.GeoDataFrame(
        poi_df.copy(),
        geometry=gpd.points_from_xy(
            poi_df["Longitude"],
            poi_df["Latitude"],
        ),
        crs="EPSG:4326",
    )

    poi_gdf["__poi_uid"] = (
        dataset_value
        + "__"
        + poi_group
        + "__"
        + poi_gdf["OSM_Type"].astype(str)
        + "__"
        + poi_gdf["OSM_ID"].astype(str)
    )

    poi_gdf = poi_gdf.drop_duplicates(
        subset="__poi_uid"
    ).reset_index(drop=True)

    return poi_gdf


# ------------------------------------------------------------------------------
# PBF EXTRACTION FUNCTION
# ------------------------------------------------------------------------------
def extract_all_poi_groups_from_pbf(
    region_key,
    roads_wgs84,
):
    """
    Parse one regional PBF once and return all five POI groups.

    The returned POIs are prefiltered to the road study-area bounding box.
    """
    if region_key not in REGION_CONFIG:
        raise ValueError(
            f"Unknown region key: {region_key}"
        )

    region = REGION_CONFIG[region_key]

    pbf_path = region["storage_pbf_path"]

    if not os.path.exists(pbf_path):
        raise FileNotFoundError(
            f"OSM PBF was not found:\n{pbf_path}"
        )

    extraction_bbox = create_wgs84_bbox(
        roads_wgs84,
        buffer_m=EXTRACTION_BBOX_BUFFER_M,
    )

    print("\n" + "=" * 90)
    print(f"EXTRACTING POIs — {region['region_name'].upper()}")
    print("=" * 90)
    print(f"Source PBF: {pbf_path}")
    print(
        "Study-area WGS84 extraction bbox: "
        f"{tuple(round(value, 6) for value in extraction_bbox)}"
    )

    handler = OSMPOIHandler(
        extraction_bbox=extraction_bbox,
    )

    try:
        handler.apply_file(
            pbf_path,
            locations=True,
            idx="flex_mem",
        )

    finally:
        handler.close_progress()

    extraction_info = {
        "region_key": region_key,
        "region_name": region["region_name"],
        "nodes_seen": handler.nodes_seen,
        "ways_seen": handler.ways_seen,
        "relations_seen": handler.relations_seen,
        "total_objects_seen": handler.total_seen,
    }

    print("\nPBF parsing completed:")
    print(f"Nodes processed     : {handler.nodes_seen:,}")
    print(f"Ways processed      : {handler.ways_seen:,}")
    print(f"Relations processed : {handler.relations_seen:,}")
    print(f"Total objects       : {handler.total_seen:,}")

    return handler.records_by_group, extraction_info


# ------------------------------------------------------------------------------
# ROAD-TO-POI METRIC CALCULATION
# ------------------------------------------------------------------------------
def calculate_road_poi_metrics(
    roads_projected,
    poi_projected,
    poi_group,
    distance_m,
):
    """
    Calculate POI count, minimum distance, and median distance
    for each road segment.

    All geometries must use the same local metric CRS in metres.
    """
    count_column = (
        f"{poi_group}_count_{distance_m}m"
    )

    min_distance_column = (
        f"{poi_group}_min_distance_m_{distance_m}m"
    )

    median_distance_column = (
        f"{poi_group}_median_distance_m_{distance_m}m"
    )

    if poi_projected.empty:
        return pd.DataFrame(
            columns=[
                "analysis_segment_id",
                count_column,
                min_distance_column,
                median_distance_column,
            ]
        )

    road_buffer_gdf = roads_projected[
        [
            "analysis_segment_id",
            "geometry",
        ]
    ].copy()

    # Buffering the complete road geometry identifies POIs
    # within the required proximity to that segment.
    road_buffer_gdf["geometry"] = (
        road_buffer_gdf.geometry.buffer(
            distance_m
        )
    )

    road_buffer_gdf = gpd.GeoDataFrame(
        road_buffer_gdf,
        geometry="geometry",
        crs=roads_projected.crs,
    )

    joined_df = gpd.sjoin(
        road_buffer_gdf,
        poi_projected[
            [
                "__poi_uid",
                "geometry",
            ]
        ],
        how="inner",
        predicate="intersects",
    )

    if joined_df.empty:
        return pd.DataFrame(
            columns=[
                "analysis_segment_id",
                count_column,
                min_distance_column,
                median_distance_column,
            ]
        )

    road_poi_pairs = joined_df[
        [
            "analysis_segment_id",
            "__poi_uid",
        ]
    ].drop_duplicates()

    road_geometry_lookup = dict(
        zip(
            roads_projected[
                "analysis_segment_id"
            ],
            roads_projected.geometry,
        )
    )

    poi_geometry_lookup = dict(
        zip(
            poi_projected["__poi_uid"],
            poi_projected.geometry,
        )
    )

    road_geometries = np.asarray(
        road_poi_pairs[
            "analysis_segment_id"
        ].map(road_geometry_lookup),
        dtype=object,
    )

    poi_geometries = np.asarray(
        road_poi_pairs[
            "__poi_uid"
        ].map(poi_geometry_lookup),
        dtype=object,
    )

    road_poi_pairs["distance_m"] = (
        safe_vectorized_distance(
            road_geometries,
            poi_geometries,
        )
    )

    # Retain only numerical distances inside the intended threshold.
    road_poi_pairs = road_poi_pairs[
        road_poi_pairs["distance_m"].notna()
        & (
            road_poi_pairs["distance_m"]
            <= distance_m + 1e-6
        )
    ].copy()

    if road_poi_pairs.empty:
        return pd.DataFrame(
            columns=[
                "analysis_segment_id",
                count_column,
                min_distance_column,
                median_distance_column,
            ]
        )

    metric_df = (
        road_poi_pairs
        .groupby(
            "analysis_segment_id",
            as_index=False,
        )
        .agg(
            **{
                count_column: (
                    "__poi_uid",
                    "nunique",
                ),
                min_distance_column: (
                    "distance_m",
                    "min",
                ),
                median_distance_column: (
                    "distance_m",
                    "median",
                ),
            }
        )
    )

    metric_df[count_column] = (
        metric_df[count_column]
        .astype(int)
    )

    metric_df[min_distance_column] = (
        metric_df[min_distance_column]
        .round(2)
    )

    metric_df[median_distance_column] = (
        metric_df[median_distance_column]
        .round(2)
    )

    print(
        f"  {poi_group:<18} | "
        f"POIs: {len(poi_projected):,} | "
        f"road-POI pairs: {len(road_poi_pairs):,} | "
        f"segments with POI: {len(metric_df):,}"
    )

    del road_buffer_gdf
    del joined_df
    del road_poi_pairs

    gc.collect()

    return metric_df


# ------------------------------------------------------------------------------
# LOAD ROAD DATA
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("LOADING ROAD DATA")
print("=" * 90)

road_layer_name = get_first_layer_name(
    POI_ROAD_INPUT
)

road_gdf = gpd.read_file(
    POI_ROAD_INPUT,
    layer=road_layer_name,
    engine="pyogrio",
).copy()

if road_gdf.empty:
    raise ValueError(
        "Road GeoPackage contains zero records."
    )

if road_gdf.crs is None:
    raise ValueError(
        "Road GeoPackage has no CRS."
    )

required_road_columns = [
    "analysis_segment_id",
    "dataset",
]

missing_road_columns = [
    column
    for column in required_road_columns
    if column not in road_gdf.columns
]

if missing_road_columns:
    raise ValueError(
        "Missing required road columns:\n"
        + "\n".join(missing_road_columns)
    )

if road_gdf["analysis_segment_id"].isna().any():
    raise ValueError(
        "analysis_segment_id contains missing values."
    )

if road_gdf["analysis_segment_id"].duplicated().any():
    duplicate_ids = road_gdf.loc[
        road_gdf["analysis_segment_id"].duplicated(
            keep=False
        ),
        "analysis_segment_id",
    ].head(20).tolist()

    raise ValueError(
        "analysis_segment_id must be unique.\n"
        f"Examples: {duplicate_ids}"
    )

road_dataset_values = sorted(
    road_gdf["dataset"]
    .dropna()
    .unique()
    .tolist()
)

unexpected_dataset_values = [
    dataset_value
    for dataset_value in road_dataset_values
    if dataset_value not in EXPECTED_DATASET_VALUES
]

if unexpected_dataset_values:
    raise ValueError(
        "Unexpected dataset values found:\n"
        f"{unexpected_dataset_values}"
    )

road_geometry_column = road_gdf.geometry.name
road_crs = road_gdf.crs

print(f"Road GeoPackage : {POI_ROAD_INPUT}")
print(f"Layer           : {road_layer_name}")
print(f"Total roads     : {len(road_gdf):,}")
print(f"Road CRS        : {road_crs}")
print(f"Datasets        : {road_dataset_values}")


# ------------------------------------------------------------------------------
# CREATE BASE METRIC TABLE
# ------------------------------------------------------------------------------
output_metric_columns = []

for poi_group, distance_m in (
    POI_DISTANCE_SETTINGS.items()
):
    output_metric_columns.extend(
        [
            f"{poi_group}_count_{distance_m}m",
            f"{poi_group}_min_distance_m_{distance_m}m",
            f"{poi_group}_median_distance_m_{distance_m}m",
        ]
    )

poi_metric_table = road_gdf[
    ["analysis_segment_id"]
].copy()

metric_parts_by_group = {
    poi_group: []
    for poi_group in POI_GROUPS
}

extraction_summary_records = []

roads_wgs84 = road_gdf.to_crs(
    epsg=4326
).copy()


# ------------------------------------------------------------------------------
# EXTRACT POIs AND CALCULATE ROAD-LEVEL METRICS BY DATASET
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("EXTRACTING POIs AND CALCULATING ROAD-LEVEL METRICS")
print("=" * 90)

for dataset_value in EXPECTED_DATASET_VALUES:

    country_roads_wgs84 = roads_wgs84[
        roads_wgs84["dataset"] == dataset_value
    ].copy()

    country_roads_wgs84 = country_roads_wgs84[
        country_roads_wgs84.geometry.notna()
        & country_roads_wgs84.geometry.geom_type.isin(
            [
                "LineString",
                "MultiLineString",
            ]
        )
    ].copy()

    if country_roads_wgs84.empty:
        print(
            f"\nNo valid road line geometry for "
            f"{dataset_value}. Skipping."
        )
        continue

    region_key = (
        "thailand"
        if dataset_value == "Thailand"
        else "maharashtra"
    )

    local_metric_crs = create_local_metric_crs(
        country_roads_wgs84
    )

    country_roads_projected = (
        country_roads_wgs84.to_crs(
            local_metric_crs
        )
    )

    print("\n" + "-" * 90)
    print(f"DATASET: {dataset_value}")
    print("-" * 90)
    print(
        f"Road segments used for POI analysis: "
        f"{len(country_roads_projected):,}"
    )
    print(f"Local metric CRS: {local_metric_crs}")

    records_by_group, pbf_info = (
        extract_all_poi_groups_from_pbf(
            region_key=region_key,
            roads_wgs84=country_roads_wgs84,
        )
    )

    for poi_group in POI_GROUPS:

        distance_m = POI_DISTANCE_SETTINGS[
            poi_group
        ]

        poi_df = prepare_poi_output_dataframe(
            records_by_group[poi_group]
        )

        poi_output_path = (
            POI_EXTRACT_OUTPUT_CONFIG[
                dataset_value
            ][poi_group]
        )

        poi_df.to_csv(
            poi_output_path,
            index=False,
            encoding="utf-8-sig",
        )

        poi_gdf_wgs84 = dataframe_to_poi_gdf(
            poi_df=poi_df,
            poi_group=poi_group,
            dataset_value=dataset_value,
        )

        if poi_gdf_wgs84.empty:
            metric_df = pd.DataFrame(
                columns=[
                    "analysis_segment_id",
                    f"{poi_group}_count_{distance_m}m",
                    (
                        f"{poi_group}_min_distance_m_"
                        f"{distance_m}m"
                    ),
                    (
                        f"{poi_group}_median_distance_m_"
                        f"{distance_m}m"
                    ),
                ]
            )

            print(
                f"  {poi_group:<18} | "
                "no extracted POIs in the road study area."
            )

        else:
            poi_gdf_projected = poi_gdf_wgs84.to_crs(
                local_metric_crs
            )

            metric_df = calculate_road_poi_metrics(
                roads_projected=country_roads_projected,
                poi_projected=poi_gdf_projected,
                poi_group=poi_group,
                distance_m=distance_m,
            )

            del poi_gdf_projected

        metric_parts_by_group[poi_group].append(
            metric_df
        )

        extraction_summary_records.append(
            {
                "dataset": dataset_value,
                "region_source": pbf_info["region_name"],
                "poi_group": poi_group,
                "search_radius_m": distance_m,
                "osm_nodes_processed": pbf_info[
                    "nodes_seen"
                ],
                "osm_ways_processed": pbf_info[
                    "ways_seen"
                ],
                "osm_relations_processed": pbf_info[
                    "relations_seen"
                ],
                "extracted_poi_records": len(poi_df),
                "segments_with_one_or_more_poi": (
                    len(metric_df)
                ),
                "raw_poi_csv": poi_output_path,
            }
        )

        del poi_df
        del poi_gdf_wgs84
        del metric_df

        gc.collect()

    del records_by_group
    del country_roads_wgs84
    del country_roads_projected

    gc.collect()


# ------------------------------------------------------------------------------
# COMBINE ALL POI METRICS USING analysis_segment_id
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("COMBINING POI METRICS BY ANALYSIS_SEGMENT_ID")
print("=" * 90)

for poi_group, distance_m in (
    POI_DISTANCE_SETTINGS.items()
):
    count_column = (
        f"{poi_group}_count_{distance_m}m"
    )

    min_distance_column = (
        f"{poi_group}_min_distance_m_{distance_m}m"
    )

    median_distance_column = (
        f"{poi_group}_median_distance_m_{distance_m}m"
    )

    group_parts = metric_parts_by_group[
        poi_group
    ]

    if len(group_parts) == 0:
        poi_metric_table[count_column] = 0
        poi_metric_table[min_distance_column] = np.nan
        poi_metric_table[
            median_distance_column
        ] = np.nan

        print(
            f"{poi_group}: no calculation result."
        )
        continue

    combined_metric_df = pd.concat(
        group_parts,
        ignore_index=True,
    )

    if combined_metric_df.empty:
        poi_metric_table[count_column] = 0
        poi_metric_table[min_distance_column] = np.nan
        poi_metric_table[
            median_distance_column
        ] = np.nan

        print(
            f"{poi_group}: no road-POI matches found."
        )
        continue

    if combined_metric_df[
        "analysis_segment_id"
    ].duplicated().any():
        duplicate_ids = combined_metric_df.loc[
            combined_metric_df[
                "analysis_segment_id"
            ].duplicated(keep=False),
            "analysis_segment_id",
        ].head(20).tolist()

        raise ValueError(
            f"Duplicate analysis_segment_id found "
            f"for {poi_group}.\n"
            f"Examples: {duplicate_ids}"
        )

    poi_metric_table = poi_metric_table.merge(
        combined_metric_df,
        on="analysis_segment_id",
        how="left",
        validate="one_to_one",
    )

    poi_metric_table[count_column] = (
        poi_metric_table[count_column]
        .fillna(0)
        .astype(int)
    )

    poi_metric_table[min_distance_column] = (
        pd.to_numeric(
            poi_metric_table[min_distance_column],
            errors="coerce",
        )
        .round(2)
    )

    poi_metric_table[
        median_distance_column
    ] = (
        pd.to_numeric(
            poi_metric_table[
                median_distance_column
            ],
            errors="coerce",
        )
        .round(2)
    )

    print(
        f"{poi_group:<18} | "
        f"segments with POI: "
        f"{int((poi_metric_table[count_column] > 0).sum()):,}"
    )

    del combined_metric_df

    gc.collect()


# ------------------------------------------------------------------------------
# MERGE POI METRICS BACK TO THE FULL ROAD DATASET
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("MERGING POI METRICS BACK TO ROAD DATA")
print("=" * 90)

road_gdf = road_gdf.drop(
    columns=output_metric_columns,
    errors="ignore",
)

ready_to_use_gdf = road_gdf.merge(
    poi_metric_table,
    on="analysis_segment_id",
    how="left",
    validate="one_to_one",
)

ready_to_use_gdf = gpd.GeoDataFrame(
    ready_to_use_gdf,
    geometry=road_geometry_column,
    crs=road_crs,
)

for poi_group, distance_m in (
    POI_DISTANCE_SETTINGS.items()
):
    count_column = (
        f"{poi_group}_count_{distance_m}m"
    )

    min_distance_column = (
        f"{poi_group}_min_distance_m_{distance_m}m"
    )

    median_distance_column = (
        f"{poi_group}_median_distance_m_{distance_m}m"
    )

    ready_to_use_gdf[count_column] = (
        ready_to_use_gdf[count_column]
        .fillna(0)
        .astype(int)
    )

    ready_to_use_gdf[min_distance_column] = (
        pd.to_numeric(
            ready_to_use_gdf[min_distance_column],
            errors="coerce",
        )
        .round(2)
    )

    ready_to_use_gdf[
        median_distance_column
    ] = (
        pd.to_numeric(
            ready_to_use_gdf[
                median_distance_column
            ],
            errors="coerce",
        )
        .round(2)
    )

print(
    f"Final road segments ready for saving: "
    f"{len(ready_to_use_gdf):,}"
)


# ------------------------------------------------------------------------------
# CREATE POI METRIC SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("POI METRIC SUMMARY")
print("=" * 90)

summary_rows = []

for dataset_value in EXPECTED_DATASET_VALUES:

    dataset_subset = ready_to_use_gdf[
        ready_to_use_gdf["dataset"] == dataset_value
    ].copy()

    for poi_group, distance_m in (
        POI_DISTANCE_SETTINGS.items()
    ):
        count_column = (
            f"{poi_group}_count_{distance_m}m"
        )

        min_distance_column = (
            f"{poi_group}_min_distance_m_"
            f"{distance_m}m"
        )

        median_distance_column = (
            f"{poi_group}_median_distance_m_"
            f"{distance_m}m"
        )

        segments_with_poi = dataset_subset[
            count_column
        ] > 0

        summary_rows.append(
            {
                "dataset": dataset_value,
                "poi_group": poi_group,
                "search_radius_m": distance_m,
                "segments_with_one_or_more_poi": int(
                    segments_with_poi.sum()
                ),
                "maximum_poi_count_on_one_segment": int(
                    dataset_subset[count_column].max()
                ),
                "total_poi_count_across_segments": int(
                    dataset_subset[count_column].sum()
                ),
                "minimum_observed_distance_m": (
                    dataset_subset[
                        min_distance_column
                    ].min()
                ),
                "median_of_segment_median_distance_m": (
                    dataset_subset[
                        median_distance_column
                    ].median()
                ),
            }
        )

poi_metric_summary_df = pd.DataFrame(
    summary_rows
)

display(poi_metric_summary_df)

poi_metric_summary_df.to_csv(
    POI_METRICS_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------------------------
# SAVE FINAL ROAD DATASET
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("SAVING FINAL ROAD DATASET WITH POI METRICS")
print("=" * 90)

for extension in ["", "-wal", "-shm"]:
    existing_file = (
        SURROUNDING_LAND_USE_RESULT
        + extension
    )

    if os.path.exists(existing_file):
        os.remove(existing_file)

ready_to_use_gdf.to_file(
    SURROUNDING_LAND_USE_RESULT,
    layer="ADB_Innovation_merge_valid_3",
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("Saved final GeoPackage:")
print(SURROUNDING_LAND_USE_RESULT)

print("\nSaved POI metric summary:")
print(POI_METRICS_SUMMARY_CSV)

print("\nFirst 5 rows with added POI columns:")
display(
    ready_to_use_gdf[
        [
            "analysis_segment_id",
            "dataset",
        ]
        + output_metric_columns
    ].head()
)


# ------------------------------------------------------------------------------
# CLEAR UNNECESSARY DATA FROM RAM
# ------------------------------------------------------------------------------
del roads_wgs84
del road_gdf
del poi_metric_table
del metric_parts_by_group
del ready_to_use_gdf
del poi_metric_summary_df

gc.collect()

print("\nSTEP 3B COMPLETED.")

## Add GHSL Built Up Data

In [ ]:
# ==============================================================================
# STEP 4 — ADD GHSL BUILT-UP CONTEXT TO ROAD SEGMENTS
#
# Input:
#   GHSL_ROAD_INPUT
#
# Output:
#   GHSL_RESULT
#
# Added main result fields:
#   - built_up_share_pct
#   - built_up_continuity_pct
#
# Supporting audit fields are also added:
#   - ghsl_epoch
#   - ghsl_resolution_arcsec
#   - built_up_buffer_each_side_m
#   - built_up_continuity_cell_m
#   - built_up_threshold_pct
#   - built_up_sample_count
#   - ghsl_status
#   - geometry_length_m_ghsl
#   - total_continuity_cells
#   - valid_continuity_cells
#   - built_up_cells
#
# No classification fields are created:
#   - built_up_continuity_class
#   - settlement_builtup_context
# ==============================================================================


# ------------------------------------------------------------------------------
# 1. VALIDATE REQUIRED VARIABLES AND INPUT FILE
# ------------------------------------------------------------------------------
required_variables = [
    "GHSL_ROAD_INPUT",
    "GHSL_OUTPUT_FOLDER",
    "GHSL_WORK_ROOT",
    "GHSL_RAW_DIR",
    "GHSL_EXTRACT_DIR",
    "GHSL_RESULT",
    "GHSL_RESULT_LAYER",
    "GHSL_SUMMARY_CSV",
    "GHSL_EPOCH",
    "GHSL_BASE_URL",
    "LATERAL_BUFFER_M",
    "CONTINUITY_CELL_M",
    "BUILT_UP_CELL_THRESHOLD_PCT",
    "LATERAL_SAMPLE_OFFSETS_M",
    "MIN_VALID_LATERAL_SAMPLES",
    "DELETE_EXTRACTED_TIF_AFTER_USE",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

if not os.path.exists(GHSL_ROAD_INPUT):
    raise FileNotFoundError(
        f"Step 3 output file was not found:\n"
        f"{GHSL_ROAD_INPUT}\n\n"
        "Please run Step 3B before running Step 4."
    )


# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_column_name(columns, preferred_name):
    """
    Find a column name without case sensitivity.
    """
    lookup = {
        str(column).strip().lower(): column
        for column in columns
    }

    requested = preferred_name.strip().lower()

    if requested not in lookup:
        raise KeyError(
            f"Column '{preferred_name}' was not found.\n"
            f"Available columns:\n{list(columns)}"
        )

    return lookup[requested]


def get_first_layer_name(gpkg_path):
    """
    Return the first layer name inside a GeoPackage.
    """
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def valid_file(file_path, min_bytes=1_000):
    """
    Check whether a file exists and is sufficiently large.
    """
    return (
        os.path.exists(file_path)
        and os.path.getsize(file_path) >= min_bytes
    )


def valid_zip(file_path):
    """
    Check whether a file is a valid ZIP archive.
    """
    return (
        valid_file(file_path)
        and zipfile.is_zipfile(file_path)
    )


def download_file(url, destination):
    """
    Download one GHSL ZIP file safely with progress tracking.
    """
    temporary_path = destination + ".part"

    if os.path.exists(temporary_path):
        os.remove(temporary_path)

    try:
        response = requests.get(
            url,
            stream=True,
            timeout=900,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 "
                    "(GHSL road built-up context analysis)"
                )
            },
        )

        response.raise_for_status()

        total_size = int(
            response.headers.get(
                "content-length",
                0,
            )
        )

        with open(
            temporary_path,
            "wb",
        ) as file, tqdm(
            total=total_size if total_size > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=os.path.basename(destination),
            dynamic_ncols=True,
        ) as progress_bar:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)
                    progress_bar.update(len(chunk))

        if not zipfile.is_zipfile(temporary_path):
            raise RuntimeError(
                "Downloaded GHSL file is not a valid ZIP:\n"
                f"{url}"
            )

        os.replace(
            temporary_path,
            destination,
        )

    except Exception:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)

        raise


def ghsl_tile_row(latitude):
    """
    GHSL 10-degree tile row.
    R1 starts at 80°N–90°N and increases southward.
    """
    return int(
        np.floor((90 - latitude) / 10)
    ) + 1


def ghsl_tile_col(longitude):
    """
    GHSL 10-degree tile column.
    C1 starts at 180°W–170°W and increases eastward.
    """
    return int(
        np.floor((longitude + 180) / 10)
    ) + 1


def ghsl_tile_filename(row, col):
    """
    Create the official GHSL ZIP filename.
    """
    return (
        f"GHS_BUILT_S_E{GHSL_EPOCH}_"
        f"GLOBE_R2023A_4326_3ss_"
        f"V1_0_R{row}_C{col}.zip"
    )


def tile_key(row, col):
    """
    Convert row and column values into one compact tile key.
    """
    return int(row) * 100 + int(col)


def tile_from_key(key):
    """
    Convert a compact tile key back to GHSL row and column.
    """
    return int(key) // 100, int(key) % 100


def tiles_from_bounds(
    bounds,
    expansion_degrees=0.01,
):
    """
    Return all GHSL 10-degree tiles touched by a WGS84 extent.

    The small extent expansion protects road corridors close
    to a tile boundary.
    """
    west, south, east, north = bounds

    west -= expansion_degrees
    south -= expansion_degrees
    east += expansion_degrees
    north += expansion_degrees

    row_north = ghsl_tile_row(north)
    row_south = ghsl_tile_row(south)

    col_west = ghsl_tile_col(west)
    col_east = ghsl_tile_col(east)

    return [
        (row, col)
        for row in range(
            min(row_north, row_south),
            max(row_north, row_south) + 1,
        )
        for col in range(
            min(col_west, col_east),
            max(col_west, col_east) + 1,
        )
    ]


def ensure_tile_available(row, col):
    """
    Reuse a cached GHSL ZIP when available.
    Otherwise, download it into the local Step 4 cache.
    """
    filename = ghsl_tile_filename(row, col)

    zip_path = os.path.join(
        GHSL_RAW_DIR,
        filename,
    )

    if valid_zip(zip_path):
        return zip_path

    print(f"Downloading GHSL tile: R{row}_C{col}")

    download_file(
        url=f"{GHSL_BASE_URL}/{filename}",
        destination=zip_path,
    )

    return zip_path


def extract_tile_tif(zip_path, row, col):
    """
    Extract the GeoTIFF inside a GHSL ZIP tile.
    """
    tif_path = os.path.join(
        GHSL_EXTRACT_DIR,
        f"GHSL_R{row}_C{col}.tif",
    )

    if valid_file(tif_path):
        return tif_path

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:
        tif_members = [
            member
            for member in zip_file.namelist()
            if member.lower().endswith(
                (".tif", ".tiff")
            )
        ]

        if len(tif_members) == 0:
            raise RuntimeError(
                f"No GeoTIFF found inside:\n{zip_path}"
            )

        with zip_file.open(
            tif_members[0]
        ) as source, open(
            tif_path,
            "wb",
        ) as target:
            shutil.copyfileobj(
                source,
                target,
            )

    return tif_path


def to_single_linestring(geometry):
    """
    Convert LineString or MultiLineString into one usable LineString.
    """
    if geometry is None or geometry.is_empty:
        raise ValueError("Geometry is empty.")

    if geometry.geom_type == "LineString":
        return geometry

    if geometry.geom_type == "MultiLineString":
        merged = linemerge(geometry)

        if merged.geom_type == "LineString":
            return merged

        line_parts = [
            part
            for part in merged.geoms
            if (
                part.geom_type == "LineString"
                and not part.is_empty
            )
        ]

        if len(line_parts) == 0:
            raise ValueError(
                "MultiLineString cannot be converted "
                "to a usable LineString."
            )

        return max(
            line_parts,
            key=lambda part: part.length,
        )

    raise ValueError(
        f"Unsupported geometry type: {geometry.geom_type}"
    )


def remove_duplicate_vertices(coordinates):
    """
    Remove repeated neighboring vertices before processing a line.
    """
    coordinates = np.asarray(
        coordinates,
        dtype="float64",
    )[:, :2]

    if len(coordinates) < 2:
        return coordinates

    keep = np.concatenate(
        [
            np.array([True]),
            np.any(
                np.diff(coordinates, axis=0) != 0,
                axis=1,
            ),
        ]
    )

    return coordinates[keep]


def line_midpoints_and_normals(
    line,
    continuity_cell_m,
):
    """
    Divide one road into conceptual 100 m units.

    Returns:
    - unit midpoint coordinates
    - local perpendicular direction for each unit
    - actual unit lengths
    - complete line length
    """
    coordinates = remove_duplicate_vertices(
        line.coords
    )

    if len(coordinates) < 2:
        raise ValueError(
            "Line has fewer than two distinct vertices."
        )

    segment_vectors = np.diff(
        coordinates,
        axis=0,
    )

    segment_lengths = np.hypot(
        segment_vectors[:, 0],
        segment_vectors[:, 1],
    )

    valid_segments = segment_lengths > 1e-9

    segment_vectors = segment_vectors[
        valid_segments
    ]

    segment_lengths = segment_lengths[
        valid_segments
    ]

    if len(segment_lengths) == 0:
        raise ValueError("Line length is zero.")

    cumulative_segment_end = np.cumsum(
        segment_lengths
    )

    total_length_m = float(
        cumulative_segment_end[-1]
    )

    number_of_units = max(
        1,
        int(
            np.ceil(
                total_length_m / continuity_cell_m
            )
        ),
    )

    unit_starts = (
        np.arange(
            number_of_units,
            dtype="float64",
        )
        * continuity_cell_m
    )

    unit_ends = np.minimum(
        unit_starts + continuity_cell_m,
        total_length_m,
    )

    unit_lengths = unit_ends - unit_starts

    unit_midpoints_m = (
        unit_starts + unit_ends
    ) / 2.0

    segment_index = np.searchsorted(
        cumulative_segment_end,
        unit_midpoints_m,
        side="right",
    )

    segment_index = np.clip(
        segment_index,
        0,
        len(segment_lengths) - 1,
    )

    segment_start_distance = np.where(
        segment_index == 0,
        0.0,
        cumulative_segment_end[
            segment_index - 1
        ],
    )

    distance_within_segment = (
        unit_midpoints_m
        - segment_start_distance
    )

    position_fraction = (
        distance_within_segment
        / segment_lengths[segment_index]
    )

    midpoints_xy = (
        coordinates[segment_index]
        + segment_vectors[segment_index]
        * position_fraction[:, None]
    )

    direction_vectors = (
        segment_vectors[segment_index]
        / segment_lengths[
            segment_index,
            None,
        ]
    )

    normal_vectors = np.column_stack(
        [
            -direction_vectors[:, 1],
            direction_vectors[:, 0],
        ]
    )

    return (
        midpoints_xy,
        normal_vectors,
        unit_lengths,
        total_length_m,
    )


def build_road_units_and_lateral_samples(
    roads_to_process,
    total_road_rows,
):
    """
    Create 100 m road units and five lateral sample points per unit.

    The sample offsets represent five equal-width strips inside
    the 200 m-wide corridor surrounding each road segment.
    """
    road_generation_status = np.full(
        total_road_rows,
        "not_processed",
        dtype=object,
    )

    unit_road_pos_chunks = []
    unit_length_chunks = []

    point_unit_id_chunks = []
    point_lon_chunks = []
    point_lat_chunks = []

    next_unit_id = 0

    for utm_epsg in sorted(
        roads_to_process["_utm_epsg"].unique()
    ):
        zone_rows = roads_to_process.loc[
            roads_to_process["_utm_epsg"] == utm_epsg
        ].copy()

        zone_utm = zone_rows.to_crs(
            epsg=int(utm_epsg)
        )

        transformer_to_wgs84 = Transformer.from_crs(
            f"EPSG:{int(utm_epsg)}",
            "EPSG:4326",
            always_xy=True,
        )

        road_positions = zone_utm[
            "_road_pos"
        ].to_numpy()

        road_geometries = zone_utm.geometry.to_numpy()

        for road_pos, geometry in tqdm(
            zip(
                road_positions,
                road_geometries,
            ),
            total=len(zone_utm),
            desc=(
                "Creating 100 m units | "
                f"EPSG:{int(utm_epsg)}"
            ),
        ):
            try:
                line = to_single_linestring(
                    geometry
                )

                (
                    midpoints_xy,
                    normal_vectors,
                    unit_lengths,
                    _,
                ) = line_midpoints_and_normals(
                    line=line,
                    continuity_cell_m=CONTINUITY_CELL_M,
                )

                unit_count = len(unit_lengths)

                lateral_xy = (
                    midpoints_xy[:, None, :]
                    + normal_vectors[:, None, :]
                    * LATERAL_SAMPLE_OFFSETS_M[
                        None,
                        :,
                        None,
                    ]
                )

                lateral_x = lateral_xy[:, :, 0].ravel()
                lateral_y = lateral_xy[:, :, 1].ravel()

                lateral_lon, lateral_lat = (
                    transformer_to_wgs84.transform(
                        lateral_x,
                        lateral_y,
                    )
                )

                unit_ids = np.arange(
                    next_unit_id,
                    next_unit_id + unit_count,
                    dtype="int64",
                )

                unit_road_pos_chunks.append(
                    np.full(
                        unit_count,
                        road_pos,
                        dtype="int32",
                    )
                )

                unit_length_chunks.append(
                    unit_lengths.astype("float32")
                )

                point_unit_id_chunks.append(
                    np.repeat(
                        unit_ids,
                        len(
                            LATERAL_SAMPLE_OFFSETS_M
                        ),
                    )
                )

                point_lon_chunks.append(
                    np.asarray(
                        lateral_lon,
                        dtype="float64",
                    )
                )

                point_lat_chunks.append(
                    np.asarray(
                        lateral_lat,
                        dtype="float64",
                    )
                )

                next_unit_id += unit_count

                road_generation_status[
                    road_pos
                ] = "ready"

            except Exception as error:
                road_generation_status[
                    road_pos
                ] = (
                    "geometry_error: "
                    f"{str(error)[:160]}"
                )

    if next_unit_id == 0:
        raise RuntimeError(
            "No valid 100 m road units could be generated."
        )

    return {
        "road_generation_status": road_generation_status,
        "unit_road_pos": np.concatenate(
            unit_road_pos_chunks
        ),
        "unit_length_m": np.concatenate(
            unit_length_chunks
        ),
        "point_unit_id": np.concatenate(
            point_unit_id_chunks
        ),
        "point_lon": np.concatenate(
            point_lon_chunks
        ),
        "point_lat": np.concatenate(
            point_lat_chunks
        ),
    }


def pixel_area_m2_by_row(source_raster):
    """
    Estimate pixel area in m² for each raster row.

    GHSL is stored in EPSG:4326, where longitude width varies
    in metres according to latitude.
    """
    transform = source_raster.transform

    row_indices = np.arange(
        source_raster.height,
        dtype="float64",
    )

    latitude_north = (
        transform.f
        + row_indices * transform.e
    )

    latitude_south = (
        latitude_north + transform.e
    )

    longitude_width_radians = np.deg2rad(
        abs(transform.a)
    )

    earth_radius_m = 6_371_008.8

    pixel_area_m2 = (
        earth_radius_m ** 2
        * longitude_width_radians
        * np.abs(
            np.sin(
                np.deg2rad(latitude_north)
            )
            - np.sin(
                np.deg2rad(latitude_south)
            )
        )
    )

    return pixel_area_m2.astype("float32")


def sample_ghsl_built_fraction(
    point_lon,
    point_lat,
    point_tile_keys,
    tile_zip_paths,
):
    """
    Sample GHSL built surface at all lateral road-context points.

    Built-up fraction =
    GHSL built surface (m²) / raster pixel area (m²)
    """
    point_built_fraction = np.full(
        len(point_lon),
        np.nan,
        dtype="float32",
    )

    tile_sort_order = np.argsort(
        point_tile_keys,
        kind="stable",
    )

    sorted_tile_keys = point_tile_keys[
        tile_sort_order
    ]

    split_positions = np.flatnonzero(
        np.diff(sorted_tile_keys)
    ) + 1

    tile_index_groups = np.split(
        tile_sort_order,
        split_positions,
    )

    for point_indices in tqdm(
        tile_index_groups,
        desc="Sampling GHSL built-surface raster",
    ):
        current_tile_key = int(
            point_tile_keys[
                point_indices[0]
            ]
        )

        row, col = tile_from_key(
            current_tile_key
        )

        zip_path = tile_zip_paths[
            (row, col)
        ]

        tif_path = extract_tile_tif(
            zip_path=zip_path,
            row=row,
            col=col,
        )

        with rasterio.open(tif_path) as source_raster:
            if source_raster.crs is None:
                raise ValueError(
                    f"GHSL tile has no CRS:\n{tif_path}"
                )

            if source_raster.crs.to_epsg() != 4326:
                raise ValueError(
                    "Expected a GHSL tile in EPSG:4326, "
                    f"but R{row}_C{col} has CRS "
                    f"{source_raster.crs}."
                )

            raster_data = source_raster.read(
                1,
                out_dtype="float32",
            )

            raster_rows, raster_cols = (
                rasterio.transform.rowcol(
                    source_raster.transform,
                    point_lon[point_indices],
                    point_lat[point_indices],
                )
            )

            raster_rows = np.asarray(
                raster_rows,
                dtype="int64",
            )

            raster_cols = np.asarray(
                raster_cols,
                dtype="int64",
            )

            inside_raster = (
                (raster_rows >= 0)
                & (
                    raster_rows
                    < source_raster.height
                )
                & (raster_cols >= 0)
                & (
                    raster_cols
                    < source_raster.width
                )
            )

            if inside_raster.any():
                inside_point_indices = point_indices[
                    inside_raster
                ]

                inside_rows = raster_rows[
                    inside_raster
                ]

                inside_cols = raster_cols[
                    inside_raster
                ]

                built_surface_m2 = raster_data[
                    inside_rows,
                    inside_cols,
                ]

                valid = np.isfinite(
                    built_surface_m2
                )

                if source_raster.nodata is not None:
                    valid &= (
                        built_surface_m2
                        != source_raster.nodata
                    )

                row_pixel_area_m2 = (
                    pixel_area_m2_by_row(
                        source_raster
                    )[inside_rows]
                )

                valid &= row_pixel_area_m2 > 0

                if valid.any():
                    built_fraction = np.clip(
                        (
                            built_surface_m2[valid]
                            / row_pixel_area_m2[valid]
                        ),
                        0.0,
                        1.0,
                    )

                    point_built_fraction[
                        inside_point_indices[valid]
                    ] = built_fraction.astype(
                        "float32"
                    )

            del raster_data

        if DELETE_EXTRACTED_TIF_AFTER_USE:
            os.remove(tif_path)

        gc.collect()

    return point_built_fraction


# ------------------------------------------------------------------------------
# 3. PREPARE FOLDERS AND LOAD STEP 3 ROAD DATA
# ------------------------------------------------------------------------------
for folder_path in [
    GHSL_OUTPUT_FOLDER,
    GHSL_WORK_ROOT,
    GHSL_RAW_DIR,
    GHSL_EXTRACT_DIR,
]:
    os.makedirs(
        folder_path,
        exist_ok=True,
    )

road_layer_name = get_first_layer_name(
    GHSL_ROAD_INPUT
)

roads_original = gpd.read_file(
    GHSL_ROAD_INPUT,
    layer=road_layer_name,
    engine="pyogrio",
).copy()

if roads_original.empty:
    raise ValueError(
        "The Step 3 road GeoPackage contains zero rows."
    )

if roads_original.crs is None:
    raise ValueError(
        "The Step 3 road GeoPackage has no CRS."
    )

dataset_col = get_column_name(
    roads_original.columns,
    "dataset",
)

if "analysis_segment_id" not in roads_original.columns:
    raise ValueError(
        "Required column 'analysis_segment_id' is missing."
    )

if roads_original["analysis_segment_id"].isna().any():
    raise ValueError(
        "analysis_segment_id contains missing values."
    )

if roads_original["analysis_segment_id"].duplicated().any():
    raise ValueError(
        "analysis_segment_id must be unique."
    )

roads_wgs84 = roads_original.to_crs(
    "EPSG:4326"
).copy()

roads_wgs84["_road_pos"] = np.arange(
    len(roads_wgs84),
    dtype="int32",
)

roads_wgs84["_dataset_clean"] = (
    roads_wgs84[dataset_col]
    .astype("string")
    .str.strip()
    .str.lower()
)

target_datasets = [
    "thailand",
    "maharashtra",
]

is_target_dataset = roads_wgs84[
    "_dataset_clean"
].isin(target_datasets)

is_valid_line_geometry = (
    roads_wgs84.geometry.notna()
    & ~roads_wgs84.geometry.is_empty
    & roads_wgs84.geometry.geom_type.isin(
        [
            "LineString",
            "MultiLineString",
        ]
    )
)

roads_to_process = roads_wgs84.loc[
    is_target_dataset
    & is_valid_line_geometry
].copy()

if roads_to_process.empty:
    raise ValueError(
        "No valid LineString/MultiLineString road rows "
        "were found for Thailand or Maharashtra."
    )

representative_points = (
    roads_to_process.geometry.representative_point()
)

utm_zone = (
    np.floor(
        (
            representative_points.x.to_numpy()
            + 180
        ) / 6
    ).astype(int)
    + 1
)

# Thailand and Maharashtra are both in the northern hemisphere.
roads_to_process["_utm_epsg"] = (
    32600 + utm_zone
).astype(int)

print("=" * 90)
print("STEP 4 INPUT DATA LOADED")
print("=" * 90)
print(f"Input GeoPackage      : {GHSL_ROAD_INPUT}")
print(f"Input layer           : {road_layer_name}")
print(f"Total road rows       : {len(roads_original):,}")
print(f"Road rows to process  : {len(roads_to_process):,}")
print(f"Road CRS              : {roads_original.crs}")

print("\nDataset counts to process:")
display(
    roads_to_process[
        dataset_col
    ]
    .value_counts()
    .rename_axis(dataset_col)
    .reset_index(name="segment_count")
)

print("\nUTM zones used:")
print(
    sorted(
        roads_to_process[
            "_utm_epsg"
        ].unique()
    )
)


# ------------------------------------------------------------------------------
# 4. IDENTIFY AND DOWNLOAD REQUIRED GHSL TILES
# ------------------------------------------------------------------------------
country_tile_sets = {}

for dataset_name in target_datasets:
    country_roads = roads_to_process.loc[
        roads_to_process[
            "_dataset_clean"
        ] == dataset_name
    ]

    if country_roads.empty:
        continue

    country_tile_sets[dataset_name] = set(
        tiles_from_bounds(
            country_roads.total_bounds
        )
    )

all_required_tiles = sorted(
    set().union(
        *country_tile_sets.values()
    )
)

print("\n" + "=" * 90)
print("GHSL TILE COVERAGE")
print("=" * 90)

for dataset_name, tiles in country_tile_sets.items():
    print(
        f"{dataset_name.title()} "
        f"({len(tiles)} tile(s)): "
        f"{[f'R{row}_C{col}' for row, col in sorted(tiles)]}"
    )

print(
    "\nTotal unique GHSL tiles to cache: "
    f"{len(all_required_tiles)}"
)

tile_zip_paths = {}

for row, col in tqdm(
    all_required_tiles,
    desc="Downloading / checking GHSL ZIP tiles",
):
    tile_zip_paths[(row, col)] = (
        ensure_tile_available(
            row,
            col,
        )
    )


# ------------------------------------------------------------------------------
# 5. CREATE 100 M ROAD UNITS AND LATERAL GHSL SAMPLE POINTS
# ------------------------------------------------------------------------------
road_unit_data = build_road_units_and_lateral_samples(
    roads_to_process=roads_to_process,
    total_road_rows=len(roads_original),
)

road_generation_status = road_unit_data[
    "road_generation_status"
]

unit_road_pos = road_unit_data[
    "unit_road_pos"
]

unit_length_m = road_unit_data[
    "unit_length_m"
]

point_unit_id = road_unit_data[
    "point_unit_id"
]

point_lon = road_unit_data[
    "point_lon"
]

point_lat = road_unit_data[
    "point_lat"
]

point_tile_rows = np.floor(
    (90 - point_lat) / 10
).astype("int16") + 1

point_tile_cols = np.floor(
    (point_lon + 180) / 10
).astype("int16") + 1

point_tile_keys = (
    point_tile_rows.astype("int32") * 100
    + point_tile_cols.astype("int32")
)

point_tiles_used = {
    tile_from_key(key)
    for key in np.unique(point_tile_keys)
}

unexpected_tiles = point_tiles_used.difference(
    set(all_required_tiles)
)

if unexpected_tiles:
    print(
        "\nAdditional GHSL tiles are needed for "
        "lateral road-context sample points:"
    )

    print(
        [
            f"R{row}_C{col}"
            for row, col in sorted(unexpected_tiles)
        ]
    )

    for row, col in sorted(unexpected_tiles):
        tile_zip_paths[(row, col)] = (
            ensure_tile_available(
                row,
                col,
            )
        )

print("\n" + "=" * 90)
print("100 M CONTINUITY UNITS CREATED")
print("=" * 90)
print(f"Road units                : {len(unit_road_pos):,}")
print(f"Lateral GHSL sample points: {len(point_lon):,}")
print(
    "Lateral samples per unit  : "
    f"{len(LATERAL_SAMPLE_OFFSETS_M)}"
)
print(
    "Total GHSL tiles used     : "
    f"{len(point_tiles_used)}"
)


# ------------------------------------------------------------------------------
# 6. SAMPLE GHSL BUILT-SURFACE FRACTION
# ------------------------------------------------------------------------------
point_built_fraction = sample_ghsl_built_fraction(
    point_lon=point_lon,
    point_lat=point_lat,
    point_tile_keys=point_tile_keys,
    tile_zip_paths=tile_zip_paths,
)

valid_point_mask = np.isfinite(
    point_built_fraction
)

unit_valid_lateral_sample_count = np.bincount(
    point_unit_id[valid_point_mask],
    minlength=len(unit_road_pos),
)

unit_built_fraction_sum = np.bincount(
    point_unit_id[valid_point_mask],
    weights=point_built_fraction[
        valid_point_mask
    ],
    minlength=len(unit_road_pos),
)

unit_built_share_pct = np.full(
    len(unit_road_pos),
    np.nan,
    dtype="float32",
)

unit_has_enough_coverage = (
    unit_valid_lateral_sample_count
    >= MIN_VALID_LATERAL_SAMPLES
)

unit_built_share_pct[
    unit_has_enough_coverage
] = (
    100
    * unit_built_fraction_sum[
        unit_has_enough_coverage
    ]
    / unit_valid_lateral_sample_count[
        unit_has_enough_coverage
    ]
)

unit_is_built_up = (
    unit_has_enough_coverage
    & (
        unit_built_share_pct
        >= BUILT_UP_CELL_THRESHOLD_PCT
    )
)


# ------------------------------------------------------------------------------
# 7. AGGREGATE 100 M UNITS BACK TO ORIGINAL ROAD SEGMENTS
# ------------------------------------------------------------------------------
road_count = len(roads_original)

road_total_unit_count = np.bincount(
    unit_road_pos,
    minlength=road_count,
)

road_valid_unit_count = np.bincount(
    unit_road_pos[
        unit_has_enough_coverage
    ],
    minlength=road_count,
)

road_built_unit_count = np.bincount(
    unit_road_pos[
        unit_is_built_up
    ],
    minlength=road_count,
)

road_total_geometry_length_m = np.bincount(
    unit_road_pos,
    weights=unit_length_m,
    minlength=road_count,
)

road_valid_geometry_length_m = np.bincount(
    unit_road_pos[
        unit_has_enough_coverage
    ],
    weights=unit_length_m[
        unit_has_enough_coverage
    ],
    minlength=road_count,
)

road_weighted_built_share_sum = np.bincount(
    unit_road_pos[
        unit_has_enough_coverage
    ],
    weights=(
        unit_built_share_pct[
            unit_has_enough_coverage
        ]
        * unit_length_m[
            unit_has_enough_coverage
        ]
    ),
    minlength=road_count,
)

road_built_up_share_pct = np.full(
    road_count,
    np.nan,
    dtype="float32",
)

road_has_valid_length = (
    road_valid_geometry_length_m > 0
)

road_built_up_share_pct[
    road_has_valid_length
] = (
    road_weighted_built_share_sum[
        road_has_valid_length
    ]
    / road_valid_geometry_length_m[
        road_has_valid_length
    ]
)

road_built_up_continuity_pct = np.full(
    road_count,
    np.nan,
    dtype="float32",
)

road_has_valid_units = (
    road_valid_unit_count > 0
)

road_built_up_continuity_pct[
    road_has_valid_units
] = (
    100
    * road_built_unit_count[
        road_has_valid_units
    ]
    / road_valid_unit_count[
        road_has_valid_units
    ]
)

road_ghsl_status = np.full(
    road_count,
    "not_target_dataset",
    dtype=object,
)

target_positions = roads_wgs84.loc[
    is_target_dataset,
    "_road_pos",
].to_numpy()

road_ghsl_status[
    target_positions
] = "invalid_or_unsupported_geometry"

ready_positions = np.where(
    road_generation_status == "ready"
)[0]

road_ghsl_status[
    ready_positions
] = "no_ghsl_coverage"

full_coverage_positions = np.where(
    (road_total_unit_count > 0)
    & (
        road_valid_unit_count
        == road_total_unit_count
    )
)[0]

road_ghsl_status[
    full_coverage_positions
] = "ok"

partial_coverage_positions = np.where(
    (road_valid_unit_count > 0)
    & (
        road_valid_unit_count
        < road_total_unit_count
    )
)[0]

road_ghsl_status[
    partial_coverage_positions
] = "partial_ghsl_coverage"


# ------------------------------------------------------------------------------
# 8. ATTACH GHSL RESULTS TO THE STEP 3 ROAD DATA
# ------------------------------------------------------------------------------
roads_output = roads_original.copy()

roads_output["ghsl_epoch"] = GHSL_EPOCH

roads_output["ghsl_resolution_arcsec"] = 3

roads_output["built_up_buffer_each_side_m"] = (
    LATERAL_BUFFER_M
)

roads_output["built_up_continuity_cell_m"] = (
    CONTINUITY_CELL_M
)

roads_output["built_up_threshold_pct"] = (
    BUILT_UP_CELL_THRESHOLD_PCT
)

roads_output["built_up_sample_count"] = len(
    LATERAL_SAMPLE_OFFSETS_M
)

roads_output["ghsl_status"] = road_ghsl_status

roads_output["geometry_length_m_ghsl"] = np.round(
    road_total_geometry_length_m,
    2,
)

roads_output["total_continuity_cells"] = (
    road_total_unit_count.astype("int32")
)

roads_output["valid_continuity_cells"] = (
    road_valid_unit_count.astype("int32")
)

roads_output["built_up_cells"] = (
    road_built_unit_count.astype("int32")
)

roads_output["built_up_share_pct"] = np.round(
    road_built_up_share_pct,
    2,
)

roads_output["built_up_continuity_pct"] = np.round(
    road_built_up_continuity_pct,
    2,
)


# ------------------------------------------------------------------------------
# 9. BUILD AND SAVE GHSL SUMMARY
# ------------------------------------------------------------------------------
summary_by_dataset = (
    roads_output[
        [
            dataset_col,
            "ghsl_status",
            "built_up_share_pct",
            "built_up_continuity_pct",
        ]
    ]
    .groupby(
        dataset_col,
        dropna=False,
    )
    .agg(
        segments=(
            "ghsl_status",
            "size",
        ),
        successful_segments=(
            "ghsl_status",
            lambda values: int(
                (values == "ok").sum()
            ),
        ),
        partial_coverage_segments=(
            "ghsl_status",
            lambda values: int(
                (
                    values
                    == "partial_ghsl_coverage"
                ).sum()
            ),
        ),
        median_built_up_share_pct=(
            "built_up_share_pct",
            "median",
        ),
        median_built_up_continuity_pct=(
            "built_up_continuity_pct",
            "median",
        ),
    )
    .reset_index()
)

summary_by_dataset.to_csv(
    GHSL_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------------------------
# 10. SAVE STEP 4 RESULT
# ------------------------------------------------------------------------------
for extension in ["", "-wal", "-shm"]:
    output_file = GHSL_RESULT + extension

    if os.path.exists(output_file):
        os.remove(output_file)

roads_output.to_file(
    GHSL_RESULT,
    layer=GHSL_RESULT_LAYER,
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("\n" + "=" * 90)
print("STEP 4 COMPLETED — GHSL BUILT-UP CONTEXT")
print("=" * 90)

print("\nSaved GeoPackage:")
print(GHSL_RESULT)

print("\nSaved summary CSV:")
print(GHSL_SUMMARY_CSV)

print("\nSummary by dataset:")
display(summary_by_dataset)

preview_columns = [
    column
    for column in [
        "analysis_segment_id",
        dataset_col,
        "road_name",
        "road_class",
        "built_up_share_pct",
        "built_up_continuity_pct",
        "ghsl_status",
    ]
    if column in roads_output.columns
]

print("\nOutput preview:")
display(
    roads_output[
        preview_columns
    ].head(10)
)


# ------------------------------------------------------------------------------
# 11. CLEAR TEMPORARY DATA FROM RAM
# ------------------------------------------------------------------------------
del roads_original
del roads_wgs84
del roads_to_process
del road_unit_data
del point_built_fraction
del roads_output

gc.collect()

# CLUSTERING

In [ ]:
# ==============================================================================
# STEP 5 — POI + GHSL CONTEXT CLUSTERING
#
# MODEL INPUTS
# ------------------------------------------------------------------------------
# 1. POI density per km:
#    - calculated using count / (shape_length / 1000)
#    - log1p transformed before clustering
#
# 2. Median POI distance:
#    - original distance in metres
#    - capped at the POI search threshold
#    - no log transformation
#
# 3. GHSL built-up context:
#    - built_up_share_pct
#    - built_up_continuity_pct
#    - original percentage values
#
# All final model features are country-balanced weighted-standardised.
#
# Motorway treatment:
# - Exact cleaned road_class = motorway is excluded from K-Means.
# - Motorway segments are assigned Cluster Z.
# ==============================================================================


# ------------------------------------------------------------------------------
# 0. GENERAL SETTINGS
# ------------------------------------------------------------------------------
warnings.filterwarnings("ignore")

PRIMARY_KEY = "analysis_segment_id"

COUNTRY_COLUMN = "dataset"
ROAD_CLASS_COLUMN = "road_class"
ROAD_LENGTH_COLUMN = "shape_length"

THAILAND_LABEL = "Thailand"
MAHARASHTRA_LABEL = "Maharashtra"

EXCLUDED_ROAD_CLASSES = {
    "motorway",
}

NEW_CLUSTER_COLUMN = (
    "poi_ghsl_context_cluster_code"
)

PREFERRED_K_VALUES = [
    3,
    4,
    5,
    6,
    7,
    8,
]

BASE_RANDOM_SEED = 42
N_INIT = 25
MAX_ITER = 300

N_STABILITY_RUNS = 5
MATCHED_SAMPLE_PER_COUNTRY = 2_500

SILHOUETTE_SAMPLE_FULL = 5_000
SILHOUETTE_SAMPLE_COUNTRY = 2_000
SILHOUETTE_SAMPLE_STABILITY = 1_500

MIN_CLUSTER_SHARE_GUARDRAIL = 0.03
MIN_PAIRWISE_ARI_GUARDRAIL = 0.80

K_SELECTION_CRITERIA = {
    "mean_country_silhouette": {
        "ascending": False,
        "weight": 2.0,
    },
    "davies_bouldin_score": {
        "ascending": True,
        "weight": 1.0,
    },
    "mean_pairwise_ARI": {
        "ascending": False,
        "weight": 2.0,
    },
    "min_pairwise_ARI": {
        "ascending": False,
        "weight": 1.0,
    },
    "mean_centroid_matching_distance": {
        "ascending": True,
        "weight": 1.0,
    },
    "min_cluster_share": {
        "ascending": False,
        "weight": 1.0,
    },
    "max_cluster_share": {
        "ascending": True,
        "weight": 0.5,
    },
}


# ------------------------------------------------------------------------------
# 1. MODEL FEATURE DICTIONARY
#
# This is the single control point for K-Means inputs.
#
# To revise the model later:
# - Change source_column
# - Change calculation_method
# - Change transform_method
# - Add or remove feature entries
#
# Supported calculation_method:
# - count_per_km
# - distance_capped
# - direct_value
#
# Supported transform_method:
# - normal
# - log1p
# - sqrt
# - log10p1
# ------------------------------------------------------------------------------
MODEL_FEATURE_CONFIG = {
    "public_transport_density": {
        "source_column": (
            "public_transport_count_250m"
        ),
        "prepared_column": (
            "public_transport_count_250m_per_km"
        ),
        "calculation_method": "count_per_km",
        "transform_method": "log1p",
        "minimum_value": 0,
        "missing_value": 0,
        "profile_group": "POI density",
        "profile_round_digits": 4,
        "description": (
            "Public-transport POI count per kilometre"
        ),
    },
    "office_density": {
        "source_column": "office_count_250m",
        "prepared_column": (
            "office_count_250m_per_km"
        ),
        "calculation_method": "count_per_km",
        "transform_method": "log1p",
        "minimum_value": 0,
        "missing_value": 0,
        "profile_group": "POI density",
        "profile_round_digits": 4,
        "description": (
            "Office POI count per kilometre"
        ),
    },
    "school_density": {
        "source_column": "school_count_250m",
        "prepared_column": (
            "school_count_250m_per_km"
        ),
        "calculation_method": "count_per_km",
        "transform_method": "log1p",
        "minimum_value": 0,
        "missing_value": 0,
        "profile_group": "POI density",
        "profile_round_digits": 4,
        "description": (
            "School POI count per kilometre"
        ),
    },
    "commercial_density": {
        "source_column": "commercial_count_250m",
        "prepared_column": (
            "commercial_count_250m_per_km"
        ),
        "calculation_method": "count_per_km",
        "transform_method": "log1p",
        "minimum_value": 0,
        "missing_value": 0,
        "profile_group": "POI density",
        "profile_round_digits": 4,
        "description": (
            "Commercial POI count per kilometre"
        ),
    },
    "industrial_density": {
        "source_column": "industrial_count_1000m",
        "prepared_column": (
            "industrial_count_1000m_per_km"
        ),
        "calculation_method": "count_per_km",
        "transform_method": "log1p",
        "minimum_value": 0,
        "missing_value": 0,
        "profile_group": "POI density",
        "profile_round_digits": 4,
        "description": (
            "Industrial POI count per kilometre"
        ),
    },
    "public_transport_distance": {
        "source_column": (
            "public_transport_median_distance_m_250m"
        ),
        "prepared_column": (
            "public_transport_median_distance_capped_m"
        ),
        "calculation_method": "distance_capped",
        "transform_method": "normal",
        "presence_count_column": (
            "public_transport_count_250m"
        ),
        "radius_cap_m": 250,
        "minimum_value": 0,
        "profile_group": "Median POI distance",
        "profile_round_digits": 2,
        "description": (
            "Median public-transport POI distance "
            "to the full road geometry"
        ),
    },
    "office_distance": {
        "source_column": (
            "office_median_distance_m_250m"
        ),
        "prepared_column": (
            "office_median_distance_capped_m"
        ),
        "calculation_method": "distance_capped",
        "transform_method": "normal",
        "presence_count_column": (
            "office_count_250m"
        ),
        "radius_cap_m": 250,
        "minimum_value": 0,
        "profile_group": "Median POI distance",
        "profile_round_digits": 2,
        "description": (
            "Median office POI distance "
            "to the full road geometry"
        ),
    },
    "school_distance": {
        "source_column": (
            "school_median_distance_m_250m"
        ),
        "prepared_column": (
            "school_median_distance_capped_m"
        ),
        "calculation_method": "distance_capped",
        "transform_method": "normal",
        "presence_count_column": (
            "school_count_250m"
        ),
        "radius_cap_m": 250,
        "minimum_value": 0,
        "profile_group": "Median POI distance",
        "profile_round_digits": 2,
        "description": (
            "Median school POI distance "
            "to the full road geometry"
        ),
    },
    "commercial_distance": {
        "source_column": (
            "commercial_median_distance_m_250m"
        ),
        "prepared_column": (
            "commercial_median_distance_capped_m"
        ),
        "calculation_method": "distance_capped",
        "transform_method": "normal",
        "presence_count_column": (
            "commercial_count_250m"
        ),
        "radius_cap_m": 250,
        "minimum_value": 0,
        "profile_group": "Median POI distance",
        "profile_round_digits": 2,
        "description": (
            "Median commercial POI distance "
            "to the full road geometry"
        ),
    },
    "industrial_distance": {
        "source_column": (
            "industrial_median_distance_m_1000m"
        ),
        "prepared_column": (
            "industrial_median_distance_capped_m"
        ),
        "calculation_method": "distance_capped",
        "transform_method": "normal",
        "presence_count_column": (
            "industrial_count_1000m"
        ),
        "radius_cap_m": 1000,
        "minimum_value": 0,
        "profile_group": "Median POI distance",
        "profile_round_digits": 2,
        "description": (
            "Median industrial POI distance "
            "to the full road geometry"
        ),
    },
    "built_up_share": {
        "source_column": "built_up_share_pct",
        "prepared_column": "built_up_share_pct",
        "calculation_method": "direct_value",
        "transform_method": "normal",
        "minimum_value": 0,
        "maximum_value": 100,
        "missing_strategy": (
            "country_median_global_fallback"
        ),
        "profile_group": "GHSL",
        "profile_round_digits": 2,
        "description": (
            "GHSL built-up share around the road corridor"
        ),
    },
    "built_up_continuity": {
        "source_column": (
            "built_up_continuity_pct"
        ),
        "prepared_column": (
            "built_up_continuity_pct"
        ),
        "calculation_method": "direct_value",
        "transform_method": "normal",
        "minimum_value": 0,
        "maximum_value": 100,
        "missing_strategy": (
            "country_median_global_fallback"
        ),
        "profile_group": "GHSL",
        "profile_round_digits": 2,
        "description": (
            "GHSL built-up continuity along the road"
        ),
    },
}


# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS — GENERAL
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    """
    Return the first layer name inside a GeoPackage.
    """
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def clean_category(series, upper=False):
    """
    Standardize categorical values safely.
    """
    output = (
        series.fillna("UNKNOWN")
        .astype("string")
        .str.strip()
    )

    if upper:
        output = output.str.upper()
    else:
        output = output.str.lower()

    return output.replace(
        {
            "": "UNKNOWN" if upper else "unknown",
            "nan": "UNKNOWN" if upper else "unknown",
            "none": "UNKNOWN" if upper else "unknown",
        }
    )


def apply_feature_transform(
    values,
    transform_method,
):
    """
    Apply a transformation stated in MODEL_FEATURE_CONFIG.
    """
    values = pd.to_numeric(
        values,
        errors="coerce",
    ).astype(float)

    if values.isna().any():
        raise ValueError(
            "Feature transformation received missing values."
        )

    if transform_method == "normal":
        return values

    if transform_method == "log1p":
        if (values < 0).any():
            raise ValueError(
                "log1p cannot be applied to negative values."
            )

        return np.log1p(values)

    if transform_method == "sqrt":
        if (values < 0).any():
            raise ValueError(
                "sqrt cannot be applied to negative values."
            )

        return np.sqrt(values)

    if transform_method == "log10p1":
        if (values < 0).any():
            raise ValueError(
                "log10p1 cannot be applied to negative values."
            )

        return np.log10(1 + values)

    raise ValueError(
        f"Unsupported transform_method: "
        f"'{transform_method}'"
    )


def build_feature_from_spec(
    frame,
    feature_name,
    feature_spec,
    country_column,
    road_length_column,
):
    """
    Build one prepared feature and one transformed model feature
    from a MODEL_FEATURE_CONFIG entry.
    """
    required_keys = [
        "source_column",
        "prepared_column",
        "calculation_method",
        "transform_method",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in feature_spec
    ]

    if missing_keys:
        raise ValueError(
            f"Feature '{feature_name}' is missing "
            f"configuration key(s): {missing_keys}"
        )

    source_column = feature_spec[
        "source_column"
    ]

    prepared_column = feature_spec[
        "prepared_column"
    ]

    calculation_method = feature_spec[
        "calculation_method"
    ]

    transform_method = feature_spec[
        "transform_method"
    ]

    if source_column not in frame.columns:
        raise KeyError(
            f"Feature '{feature_name}' requires source column "
            f"'{source_column}', but it is not available."
        )

    source_numeric = pd.to_numeric(
        frame[source_column],
        errors="coerce",
    )

    source_missing_mask = source_numeric.isna()

    minimum_value = feature_spec.get(
        "minimum_value",
        None,
    )

    maximum_value = feature_spec.get(
        "maximum_value",
        None,
    )

    if calculation_method == "count_per_km":
        if road_length_column not in frame.columns:
            raise KeyError(
                f"Feature '{feature_name}' needs "
                f"'{road_length_column}' for per-km conversion."
            )

        missing_value = feature_spec.get(
            "missing_value",
            0,
        )

        prepared_values = source_numeric.fillna(
            missing_value
        )

        if minimum_value is not None:
            prepared_values = prepared_values.clip(
                lower=minimum_value
            )

        if maximum_value is not None:
            prepared_values = prepared_values.clip(
                upper=maximum_value
            )

        road_length_km = (
            pd.to_numeric(
                frame[road_length_column],
                errors="coerce",
            )
            / 1000.0
        )

        invalid_length_mask = (
            road_length_km.isna()
            | (road_length_km <= 0)
        )

        if invalid_length_mask.any():
            raise ValueError(
                f"Feature '{feature_name}' found "
                f"{invalid_length_mask.sum():,} invalid "
                f"road lengths."
            )

        prepared_values = (
            prepared_values / road_length_km
        )

        default_value_mask = source_missing_mask

        imputation_rule = (
            f"Missing values set to {missing_value} "
            "before per-km conversion"
        )

    elif calculation_method == "distance_capped":
        presence_count_column = feature_spec.get(
            "presence_count_column"
        )

        radius_cap_m = feature_spec.get(
            "radius_cap_m"
        )

        if presence_count_column is None:
            raise ValueError(
                f"Feature '{feature_name}' needs "
                "'presence_count_column'."
            )

        if radius_cap_m is None:
            raise ValueError(
                f"Feature '{feature_name}' needs "
                "'radius_cap_m'."
            )

        if presence_count_column not in frame.columns:
            raise KeyError(
                f"Feature '{feature_name}' requires "
                f"'{presence_count_column}', but it is missing."
            )

        count_numeric = pd.to_numeric(
            frame[presence_count_column],
            errors="coerce",
        ).fillna(0)

        count_numeric = count_numeric.clip(
            lower=0
        )

        invalid_distance_mask = (
            source_numeric.isna()
            | (source_numeric < 0)
            | (count_numeric <= 0)
        )

        prepared_values = source_numeric.clip(
            lower=minimum_value
            if minimum_value is not None
            else 0,
            upper=radius_cap_m,
        )

        prepared_values = prepared_values.where(
            ~invalid_distance_mask,
            radius_cap_m,
        ).fillna(radius_cap_m)

        default_value_mask = invalid_distance_mask

        imputation_rule = (
            f"Use {radius_cap_m} m when POI count is zero, "
            "distance is missing, or distance is invalid"
        )

    elif calculation_method == "direct_value":
        prepared_values = source_numeric.copy()

        if minimum_value is not None:
            prepared_values = prepared_values.clip(
                lower=minimum_value
            )

        if maximum_value is not None:
            prepared_values = prepared_values.clip(
                upper=maximum_value
            )

        missing_strategy = feature_spec.get(
            "missing_strategy",
            "error",
        )

        if missing_strategy == (
            "country_median_global_fallback"
        ):
            country_median = (
                prepared_values.groupby(
                    frame[country_column]
                )
                .transform("median")
            )

            global_median = prepared_values.median()

            if pd.isna(global_median):
                raise ValueError(
                    f"Feature '{feature_name}' has no valid "
                    "values for country/global imputation."
                )

            prepared_values = (
                prepared_values
                .fillna(country_median)
                .fillna(global_median)
            )

            default_value_mask = source_missing_mask

            imputation_rule = (
                "Country median; global median fallback"
            )

        elif missing_strategy == "zero":
            prepared_values = prepared_values.fillna(0)

            default_value_mask = source_missing_mask

            imputation_rule = (
                "Missing values set to zero"
            )

        elif missing_strategy == "error":
            if prepared_values.isna().any():
                raise ValueError(
                    f"Feature '{feature_name}' contains "
                    "missing values and its missing_strategy "
                    "is 'error'."
                )

            default_value_mask = pd.Series(
                False,
                index=frame.index,
            )

            imputation_rule = (
                "No missing-value replacement applied"
            )

        else:
            raise ValueError(
                f"Unsupported missing_strategy "
                f"'{missing_strategy}' for feature "
                f"'{feature_name}'."
            )

    else:
        raise ValueError(
            f"Unsupported calculation_method "
            f"'{calculation_method}' for feature "
            f"'{feature_name}'."
        )

    if prepared_values.isna().any():
        raise ValueError(
            f"Feature '{feature_name}' still contains missing "
            "values after preparation."
        )

    model_values = apply_feature_transform(
        values=prepared_values,
        transform_method=transform_method,
    )

    model_column = (
        f"{transform_method}__{prepared_column}"
    )

    audit_record = {
        "feature_name": feature_name,
        "source_column": source_column,
        "prepared_column": prepared_column,
        "model_column": model_column,
        "calculation_method": calculation_method,
        "transform_method": transform_method,
        "profile_group": feature_spec.get(
            "profile_group",
            "Other",
        ),
        "description": feature_spec.get(
            "description",
            "",
        ),
        "source_missing_rows": int(
            source_missing_mask.sum()
        ),
        "rows_using_default_or_imputation": int(
            default_value_mask.sum()
        ),
        "imputation_rule": imputation_rule,
        "minimum_prepared_value": float(
            prepared_values.min()
        ),
        "median_prepared_value": float(
            prepared_values.median()
        ),
        "maximum_prepared_value": float(
            prepared_values.max()
        ),
    }

    return (
        prepared_values,
        model_values,
        audit_record,
    )


def build_model_features_from_config(
    source_df,
    feature_config,
    country_column,
    road_length_column,
):
    """
    Create prepared and transformed model features from the dictionary.
    """
    working_df = source_df.copy()

    model_feature_df = pd.DataFrame(
        index=working_df.index
    )

    metadata_rows = []
    audit_rows = []

    for feature_name, feature_spec in (
        feature_config.items()
    ):
        (
            prepared_values,
            model_values,
            audit_record,
        ) = build_feature_from_spec(
            frame=working_df,
            feature_name=feature_name,
            feature_spec=feature_spec,
            country_column=country_column,
            road_length_column=road_length_column,
        )

        prepared_column = feature_spec[
            "prepared_column"
        ]

        model_column = audit_record[
            "model_column"
        ]

        working_df[prepared_column] = (
            prepared_values.astype(float)
        )

        model_feature_df[model_column] = (
            model_values.astype(float)
        )

        metadata_rows.append(
            {
                "feature_name": feature_name,
                "source_column": feature_spec[
                    "source_column"
                ],
                "prepared_column": prepared_column,
                "model_column": model_column,
                "calculation_method": feature_spec[
                    "calculation_method"
                ],
                "transform_method": feature_spec[
                    "transform_method"
                ],
                "profile_group": feature_spec.get(
                    "profile_group",
                    "Other",
                ),
                "profile_round_digits": feature_spec.get(
                    "profile_round_digits",
                    2,
                ),
                "description": feature_spec.get(
                    "description",
                    "",
                ),
            }
        )

        audit_rows.append(audit_record)

    if model_feature_df.empty:
        raise ValueError(
            "MODEL_FEATURE_CONFIG created no model features."
        )

    if not np.isfinite(
        model_feature_df.to_numpy(
            dtype=float
        )
    ).all():
        raise ValueError(
            "The feature dictionary produced "
            "non-finite model values."
        )

    return (
        working_df,
        model_feature_df,
        pd.DataFrame(metadata_rows),
        pd.DataFrame(audit_rows),
    )


# ------------------------------------------------------------------------------
# 3. HELPER FUNCTIONS — K-MEANS AND K SELECTION
# ------------------------------------------------------------------------------
def weighted_standardise(
    matrix,
    sample_weight,
):
    """
    Apply country-balanced weighted standardisation.
    """
    sample_weight = np.asarray(
        sample_weight,
        dtype=float,
    )

    weighted_mean = np.average(
        matrix,
        axis=0,
        weights=sample_weight,
    )

    weighted_variance = np.average(
        (matrix - weighted_mean) ** 2,
        axis=0,
        weights=sample_weight,
    )

    weighted_scale = np.sqrt(
        weighted_variance
    )

    weighted_scale[
        weighted_scale < 1e-12
    ] = 1.0

    scaled_matrix = (
        matrix - weighted_mean
    ) / weighted_scale

    return (
        scaled_matrix,
        weighted_mean,
        weighted_scale,
    )


def fit_kmeans(
    X,
    n_clusters,
    random_state,
    sample_weight=None,
):
    """
    Fit K-Means using the configured parameters.
    """
    model = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=N_INIT,
        max_iter=MAX_ITER,
    )

    if sample_weight is None:
        model.fit(X)
    else:
        model.fit(
            X,
            sample_weight=sample_weight,
        )

    return model


def safe_silhouette_score(
    X,
    labels,
    sample_size,
    random_state,
):
    """
    Return silhouette score safely.
    """
    if len(X) < 3 or len(np.unique(labels)) < 2:
        return np.nan

    try:
        return silhouette_score(
            X,
            labels,
            sample_size=min(
                sample_size,
                len(X),
            ),
            random_state=random_state,
        )

    except Exception:
        return np.nan


def calculate_model_metrics(
    X,
    labels,
    source_df,
    random_state,
    silhouette_sample_size,
):
    """
    Calculate quality and cluster-balance metrics.
    """
    cluster_sizes = pd.Series(
        labels
    ).value_counts()

    silhouette_all = safe_silhouette_score(
        X=X,
        labels=labels,
        sample_size=silhouette_sample_size,
        random_state=random_state,
    )

    country_scores = {}

    for country in [
        THAILAND_LABEL,
        MAHARASHTRA_LABEL,
    ]:
        country_mask = (
            source_df[COUNTRY_COLUMN].to_numpy()
            == country
        )

        country_scores[country] = safe_silhouette_score(
            X=X[country_mask],
            labels=labels[country_mask],
            sample_size=min(
                SILHOUETTE_SAMPLE_COUNTRY,
                int(country_mask.sum()),
            ),
            random_state=random_state,
        )

    valid_country_scores = [
        score
        for score in country_scores.values()
        if pd.notna(score)
    ]

    mean_country_silhouette = (
        float(np.mean(valid_country_scores))
        if valid_country_scores
        else np.nan
    )

    try:
        davies_bouldin = davies_bouldin_score(
            X,
            labels,
        )

    except Exception:
        davies_bouldin = np.nan

    country_mix = pd.crosstab(
        pd.Series(
            labels,
            name="cluster_id",
        ),
        source_df[COUNTRY_COLUMN],
        normalize="index",
    )

    most_country_concentrated_cluster_share = (
        country_mix.max(axis=1).max()
        if len(country_mix) > 0
        else np.nan
    )

    return {
        "silhouette_all": silhouette_all,
        "silhouette_thailand": country_scores.get(
            THAILAND_LABEL,
            np.nan,
        ),
        "silhouette_maharashtra": country_scores.get(
            MAHARASHTRA_LABEL,
            np.nan,
        ),
        "mean_country_silhouette": (
            mean_country_silhouette
        ),
        "davies_bouldin_score": davies_bouldin,
        "min_cluster_size": int(
            cluster_sizes.min()
        ),
        "min_cluster_share": float(
            cluster_sizes.min() / len(labels)
        ),
        "max_cluster_size": int(
            cluster_sizes.max()
        ),
        "max_cluster_share": float(
            cluster_sizes.max() / len(labels)
        ),
        "most_country_concentrated_cluster_share": (
            most_country_concentrated_cluster_share
        ),
    }


def create_matched_sampling_plan(
    source_df,
    requested_sample_per_country,
):
    """
    Build equal Thailand/Maharashtra samples matched by
    road_class and land_use for stability testing.
    """
    counts = (
        source_df.groupby(
            [
                COUNTRY_COLUMN,
                "sampling_stratum",
            ]
        )
        .size()
        .unstack(fill_value=0)
    )

    all_strata = sorted(
        source_df["sampling_stratum"]
        .unique()
    )

    thailand_counts = (
        counts.loc[THAILAND_LABEL]
        if THAILAND_LABEL in counts.index
        else pd.Series(
            0,
            index=all_strata,
        )
    )

    maharashtra_counts = (
        counts.loc[MAHARASHTRA_LABEL]
        if MAHARASHTRA_LABEL in counts.index
        else pd.Series(
            0,
            index=all_strata,
        )
    )

    plan = pd.DataFrame(
        {
            "thailand_available": (
                thailand_counts.reindex(
                    all_strata,
                    fill_value=0,
                )
            ),
            "maharashtra_available": (
                maharashtra_counts.reindex(
                    all_strata,
                    fill_value=0,
                )
            ),
        }
    )

    plan.index.name = "sampling_stratum"

    plan["common_capacity"] = np.minimum(
        plan["thailand_available"],
        plan["maharashtra_available"],
    )

    plan = plan.loc[
        plan["common_capacity"] > 0
    ].copy()

    total_common_capacity = int(
        plan["common_capacity"].sum()
    )

    if total_common_capacity == 0:
        return None, 0

    actual_target = min(
        requested_sample_per_country,
        total_common_capacity,
    )

    raw_allocation = (
        actual_target
        * plan["common_capacity"]
        / total_common_capacity
    )

    plan["sample_per_country"] = np.floor(
        raw_allocation
    ).astype(int)

    remaining = int(
        actual_target
        - plan["sample_per_country"].sum()
    )

    plan["remainder"] = (
        raw_allocation
        - plan["sample_per_country"]
    )

    for stratum in plan.sort_values(
        "remainder",
        ascending=False,
    ).index:
        if remaining <= 0:
            break

        if (
            plan.loc[
                stratum,
                "sample_per_country",
            ]
            < plan.loc[
                stratum,
                "common_capacity",
            ]
        ):
            plan.loc[
                stratum,
                "sample_per_country",
            ] += 1

            remaining -= 1

    return (
        plan.loc[
            plan["sample_per_country"] > 0
        ].copy(),
        actual_target,
    )


def draw_matched_sample(
    source_df,
    matched_plan,
    random_seed,
):
    """
    Draw one country-balanced sample from the matched plan.
    """
    sample_parts = []

    for row_number, (
        stratum,
        row,
    ) in enumerate(
        matched_plan.iterrows()
    ):
        n_per_country = int(
            row["sample_per_country"]
        )

        for country_number, country in enumerate(
            [
                THAILAND_LABEL,
                MAHARASHTRA_LABEL,
            ]
        ):
            candidates = source_df.loc[
                (
                    source_df[COUNTRY_COLUMN]
                    == country
                )
                & (
                    source_df["sampling_stratum"]
                    == stratum
                )
            ]

            sample_parts.append(
                candidates.sample(
                    n=n_per_country,
                    replace=False,
                    random_state=(
                        random_seed
                        + row_number * 100
                        + country_number
                    ),
                )
            )

    return pd.concat(
        sample_parts,
        axis=0,
    )


def draw_equal_country_sample(
    source_df,
    requested_sample_per_country,
    random_seed,
):
    """
    Fallback sampling if there are no shared strata.
    """
    thailand_df = source_df.loc[
        source_df[COUNTRY_COLUMN]
        == THAILAND_LABEL
    ]

    maharashtra_df = source_df.loc[
        source_df[COUNTRY_COLUMN]
        == MAHARASHTRA_LABEL
    ]

    actual_target = min(
        requested_sample_per_country,
        len(thailand_df),
        len(maharashtra_df),
    )

    if actual_target < 2:
        raise ValueError(
            "Not enough non-motorway roads per country "
            "for stability testing."
        )

    return pd.concat(
        [
            thailand_df.sample(
                n=actual_target,
                replace=False,
                random_state=random_seed,
            ),
            maharashtra_df.sample(
                n=actual_target,
                replace=False,
                random_state=random_seed + 1,
            ),
        ],
        axis=0,
    )


def centroid_matching_distance(
    centres_a,
    centres_b,
):
    """
    Compare centroids after optimal label matching.
    """
    distance_matrix = pairwise_distances(
        centres_a,
        centres_b,
    )

    row_index, column_index = (
        linear_sum_assignment(
            distance_matrix
        )
    )

    return float(
        distance_matrix[
            row_index,
            column_index,
        ].mean()
    )


def create_cluster_code_map(labels):
    """
    Convert numeric labels into Cluster A, Cluster B, etc.
    """
    cluster_ids = sorted(
        np.unique(labels)
    )

    return {
        int(cluster_id): chr(65 + position)
        for position, cluster_id in enumerate(
            cluster_ids
        )
    }


def select_k_automatically(dashboard):
    """
    Apply quality guardrails first, then weighted metric ranking.
    """
    dashboard = dashboard.copy()

    dashboard[
        "passes_cluster_size_guardrail"
    ] = (
        dashboard["min_cluster_share"]
        >= MIN_CLUSTER_SHARE_GUARDRAIL
    )

    dashboard[
        "passes_stability_guardrail"
    ] = (
        dashboard["min_pairwise_ARI"]
        >= MIN_PAIRWISE_ARI_GUARDRAIL
    )

    dashboard["selection_candidate"] = (
        dashboard[
            "passes_cluster_size_guardrail"
        ]
        & dashboard[
            "passes_stability_guardrail"
        ]
    )

    candidate_df = dashboard.loc[
        dashboard["selection_candidate"]
    ].copy()

    fallback_used = False

    if candidate_df.empty:
        fallback_used = True
        candidate_df = dashboard.copy()
        dashboard["selection_candidate"] = True

    weighted_rank_total = np.zeros(
        len(candidate_df),
        dtype=float,
    )

    total_weight = 0.0

    for metric, settings in (
        K_SELECTION_CRITERIA.items()
    ):
        rank_column = f"rank_{metric}"

        candidate_df[rank_column] = (
            candidate_df[metric]
            .rank(
                method="min",
                ascending=settings["ascending"],
                na_option="bottom",
            )
        )

        weighted_rank_total += (
            candidate_df[rank_column].to_numpy()
            * settings["weight"]
        )

        total_weight += settings["weight"]

    candidate_df[
        "automatic_selection_score"
    ] = weighted_rank_total / total_weight

    candidate_df = candidate_df.sort_values(
        [
            "automatic_selection_score",
            "k",
        ],
        ascending=[
            True,
            True,
        ],
    ).reset_index(drop=True)

    candidate_df[
        "automatic_selection_rank"
    ] = np.arange(len(candidate_df)) + 1

    selected_k = int(
        candidate_df.iloc[0]["k"]
    )

    selection_columns = [
        "k",
        "automatic_selection_score",
        "automatic_selection_rank",
    ] + [
        f"rank_{metric}"
        for metric in K_SELECTION_CRITERIA
    ]

    dashboard = dashboard.merge(
        candidate_df[selection_columns],
        on="k",
        how="left",
        validate="one_to_one",
    )

    dashboard[
        "selected_automatically"
    ] = (
        dashboard["k"] == selected_k
    )

    return (
        selected_k,
        dashboard,
        fallback_used,
    )


def sort_cluster_codes(cluster_codes):
    """
    Sort Cluster A, B, C, ... and Cluster Z last.
    """
    return sorted(
        cluster_codes,
        key=lambda value: (
            str(value).strip().upper() == "CLUSTER Z",
            str(value),
        ),
    )


# ------------------------------------------------------------------------------
# 4. VALIDATE PATH VARIABLES AND LOAD INPUT
# ------------------------------------------------------------------------------
required_variables = [
    "CLUSTERING_INPUT",
    "CLUSTERING_OUTPUT_FOLDER",
    "CLUSTERING_RESULT",
    "CLUSTERING_RESULT_LAYER",
    "CLUSTERING_MODEL_FILE",
    "CLUSTERING_K_SELECTION_CSV",
    "CLUSTERING_PROFILE_CSV",
    "CLUSTERING_FEATURE_AUDIT_CSV",
    "CLUSTERING_FEATURE_METADATA_CSV",
    "CLUSTERING_FEATURE_CONFIG_JSON",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

if not os.path.exists(CLUSTERING_INPUT):
    raise FileNotFoundError(
        f"Clustering input was not found:\n"
        f"{CLUSTERING_INPUT}"
    )

print("=" * 90)
print("STEP 5 — POI + GHSL CONTEXT CLUSTERING")
print("=" * 90)

input_layer_name = get_first_layer_name(
    CLUSTERING_INPUT
)

gdf_original = gpd.read_file(
    CLUSTERING_INPUT,
    layer=input_layer_name,
    engine="pyogrio",
).copy()

if gdf_original.empty:
    raise ValueError(
        "The clustering input contains zero road records."
    )

if PRIMARY_KEY not in gdf_original.columns:
    raise KeyError(
        f"Missing required primary key: {PRIMARY_KEY}"
    )

if gdf_original[PRIMARY_KEY].isna().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' contains missing values."
    )

if gdf_original[PRIMARY_KEY].duplicated().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' must be unique."
    )

print(f"Input GeoPackage : {CLUSTERING_INPUT}")
print(f"Input layer      : {input_layer_name}")
print(f"Road records     : {len(gdf_original):,}")
print(f"CRS              : {gdf_original.crs}")


# ------------------------------------------------------------------------------
# 5. KEEP VALID ROAD SEGMENTS
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("PREPARING VALID ROAD SEGMENTS")
print("=" * 90)

if "analysis_status" in gdf_original.columns:
    valid_mask = (
        gdf_original["analysis_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("valid")
    )
else:
    valid_mask = pd.Series(
        True,
        index=gdf_original.index,
    )

gdf_valid = gdf_original.loc[
    valid_mask
].copy()

if gdf_valid.empty:
    raise ValueError(
        "No valid road segments are available for clustering."
    )

geometry_column = gdf_valid.geometry.name

df_valid = pd.DataFrame(
    gdf_valid.drop(
        columns=geometry_column
    )
).copy()

required_base_columns = [
    PRIMARY_KEY,
    COUNTRY_COLUMN,
    ROAD_CLASS_COLUMN,
    ROAD_LENGTH_COLUMN,
]

missing_base_columns = [
    column
    for column in required_base_columns
    if column not in df_valid.columns
]

if missing_base_columns:
    raise KeyError(
        "Missing required base columns:\n"
        + "\n".join(
            f"- {column}"
            for column in missing_base_columns
        )
    )

df_valid[ROAD_LENGTH_COLUMN] = pd.to_numeric(
    df_valid[ROAD_LENGTH_COLUMN],
    errors="coerce",
)

invalid_length_mask = (
    df_valid[ROAD_LENGTH_COLUMN].isna()
    | (
        df_valid[ROAD_LENGTH_COLUMN]
        <= 0
    )
)

if invalid_length_mask.any():
    invalid_examples = df_valid.loc[
        invalid_length_mask,
        [
            PRIMARY_KEY,
            COUNTRY_COLUMN,
            ROAD_CLASS_COLUMN,
            ROAD_LENGTH_COLUMN,
        ],
    ].head(20)

    raise ValueError(
        f"{invalid_length_mask.sum():,} road segment(s) "
        f"have missing or non-positive "
        f"'{ROAD_LENGTH_COLUMN}'.\n\n"
        f"Examples:\n{invalid_examples}"
    )

df_valid["road_class_clean"] = clean_category(
    df_valid[ROAD_CLASS_COLUMN],
    upper=False,
)

if "land_use" in df_valid.columns:
    df_valid["land_use_sampling"] = clean_category(
        df_valid["land_use"],
        upper=True,
    )
else:
    df_valid["land_use_sampling"] = "UNKNOWN"

available_countries = set(
    df_valid[COUNTRY_COLUMN]
    .dropna()
    .astype(str)
    .unique()
)

required_countries = {
    THAILAND_LABEL,
    MAHARASHTRA_LABEL,
}

if not required_countries.issubset(
    available_countries
):
    raise ValueError(
        f"Expected country labels: "
        f"{required_countries}\n"
        f"Available labels: {available_countries}"
    )

print(
    f"Valid road segments retained: "
    f"{len(df_valid):,}"
)


# ------------------------------------------------------------------------------
# 6. BUILD MODEL FEATURES FROM DICTIONARY
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("BUILDING FEATURES FROM MODEL_FEATURE_CONFIG")
print("=" * 90)

(
    df_features,
    model_feature_df,
    feature_metadata_df,
    feature_audit_df,
) = build_model_features_from_config(
    source_df=df_valid,
    feature_config=MODEL_FEATURE_CONFIG,
    country_column=COUNTRY_COLUMN,
    road_length_column=ROAD_LENGTH_COLUMN,
)

feature_metadata_df.to_csv(
    CLUSTERING_FEATURE_METADATA_CSV,
    index=False,
    encoding="utf-8-sig",
)

feature_audit_df.to_csv(
    CLUSTERING_FEATURE_AUDIT_CSV,
    index=False,
    encoding="utf-8-sig",
)

with open(
    CLUSTERING_FEATURE_CONFIG_JSON,
    "w",
    encoding="utf-8",
) as json_file:
    json.dump(
        MODEL_FEATURE_CONFIG,
        json_file,
        indent=4,
        ensure_ascii=False,
    )

print("\nFeature specification:")
display(feature_metadata_df)

print("\nFeature preparation audit:")
display(feature_audit_df)


# ------------------------------------------------------------------------------
# 7. EXCLUDE MOTORWAYS AND BUILD MODEL MATRIX
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("BUILDING COUNTRY-BALANCED K-MEANS MATRIX")
print("=" * 90)

motorway_mask = (
    df_features["road_class_clean"]
    .isin(EXCLUDED_ROAD_CLASSES)
)

motorway_df = df_features.loc[
    motorway_mask
].copy()

model_df = df_features.loc[
    ~motorway_mask
].copy()

if model_df.empty:
    raise ValueError(
        "All valid roads are motorways. "
        "No records remain for K-Means."
    )

model_feature_df = model_feature_df.loc[
    model_df.index
].copy()

active_model_features = [
    column
    for column in model_feature_df.columns
    if model_feature_df[column].var() > 1e-12
]

if len(active_model_features) < 2:
    raise ValueError(
        "Fewer than two active model features remain "
        "after removing zero-variance inputs."
    )

model_feature_df = model_feature_df[
    active_model_features
].copy()

numeric_matrix = model_feature_df.to_numpy(
    dtype=float
)

if not np.isfinite(numeric_matrix).all():
    raise ValueError(
        "The K-Means matrix contains non-finite values."
    )

n_total = len(model_df)

n_thailand = int(
    (
        model_df[COUNTRY_COLUMN]
        == THAILAND_LABEL
    ).sum()
)

n_maharashtra = int(
    (
        model_df[COUNTRY_COLUMN]
        == MAHARASHTRA_LABEL
    ).sum()
)

if n_thailand == 0 or n_maharashtra == 0:
    raise ValueError(
        "Both Thailand and Maharashtra need non-motorway "
        "segments for country-balanced clustering."
    )

country_weight_map = {
    THAILAND_LABEL: n_total / (2 * n_thailand),
    MAHARASHTRA_LABEL: (
        n_total / (2 * n_maharashtra)
    ),
}

country_balanced_weights = (
    model_df[COUNTRY_COLUMN]
    .map(country_weight_map)
    .to_numpy(dtype=float)
)

model_df["sampling_stratum"] = (
    model_df["road_class_clean"]
    + " || "
    + model_df["land_use_sampling"]
)

model_df["_model_row_position"] = np.arange(
    len(model_df),
    dtype=int,
)

(
    X_weighted,
    numeric_weighted_mean,
    numeric_weighted_scale,
) = weighted_standardise(
    matrix=numeric_matrix,
    sample_weight=country_balanced_weights,
)

print(
    f"Motorways assigned to Cluster Z: "
    f"{len(motorway_df):,}"
)

print(
    f"Non-motorway roads in K-Means: "
    f"{len(model_df):,}"
)

print(
    f"Final matrix shape: {X_weighted.shape}"
)

print("\nActive K-Means features:")
for column in active_model_features:
    print(f"- {column}")


# ------------------------------------------------------------------------------
# 8. PREPARE STABILITY SAMPLE AND ELIGIBLE K VALUES
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("PREPARING STABILITY SAMPLE")
print("=" * 90)

matched_plan, actual_sample_per_country = (
    create_matched_sampling_plan(
        source_df=model_df,
        requested_sample_per_country=(
            MATCHED_SAMPLE_PER_COUNTRY
        ),
    )
)

if matched_plan is None:
    stability_sampling_method = (
        "Equal country sample fallback "
        "(no common road_class × land_use strata)"
    )

    actual_sample_per_country = min(
        MATCHED_SAMPLE_PER_COUNTRY,
        n_thailand,
        n_maharashtra,
    )
else:
    stability_sampling_method = (
        "Matched road_class × land_use sample"
    )

if actual_sample_per_country < 2:
    raise ValueError(
        "Fewer than two records per country are available "
        "for stability testing."
    )

K_VALUES = [
    k
    for k in PREFERRED_K_VALUES
    if (
        k <= len(model_df)
        and k <= actual_sample_per_country * 2
    )
]

if len(K_VALUES) == 0:
    raise ValueError(
        "No eligible K value remains after checking "
        "model and stability sample sizes."
    )

print(
    f"Stability sample method: "
    f"{stability_sampling_method}"
)

print(
    f"Stability sample per country: "
    f"{actual_sample_per_country:,}"
)

print(f"K values to test: {K_VALUES}")


# ------------------------------------------------------------------------------
# 9. TEST CANDIDATE K VALUES
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("TESTING CANDIDATE K VALUES")
print("=" * 90)

full_metrics_rows = []

for k in K_VALUES:
    print(
        f"Running country-balanced K-Means | K = {k}"
    )

    test_model = fit_kmeans(
        X=X_weighted,
        n_clusters=k,
        random_state=BASE_RANDOM_SEED,
        sample_weight=country_balanced_weights,
    )

    metrics = calculate_model_metrics(
        X=X_weighted,
        labels=test_model.labels_,
        source_df=model_df,
        random_state=BASE_RANDOM_SEED + k,
        silhouette_sample_size=SILHOUETTE_SAMPLE_FULL,
    )

    metrics["k"] = k

    full_metrics_rows.append(metrics)

full_metrics_df = pd.DataFrame(
    full_metrics_rows
)


# ------------------------------------------------------------------------------
# 10. STABILITY TEST
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("RUNNING K-MEANS STABILITY TEST")
print("=" * 90)

stability_metrics_rows = []

stability_assignments_by_k = {
    k: []
    for k in K_VALUES
}

stability_centres_by_k = {
    k: []
    for k in K_VALUES
}

for run_number in range(
    1,
    N_STABILITY_RUNS + 1,
):
    run_seed = (
        BASE_RANDOM_SEED
        + run_number * 1000
    )

    print(
        f"Stability run "
        f"{run_number}/{N_STABILITY_RUNS}"
    )

    if matched_plan is not None:
        sample_df = draw_matched_sample(
            source_df=model_df,
            matched_plan=matched_plan,
            random_seed=run_seed,
        )
    else:
        sample_df = draw_equal_country_sample(
            source_df=model_df,
            requested_sample_per_country=(
                actual_sample_per_country
            ),
            random_seed=run_seed,
        )

    sample_positions = sample_df[
        "_model_row_position"
    ].to_numpy(dtype=int)

    X_sample = X_weighted[
        sample_positions
    ]

    for k in K_VALUES:
        stability_model = fit_kmeans(
            X=X_sample,
            n_clusters=k,
            random_state=run_seed + k,
            sample_weight=None,
        )

        stability_labels = stability_model.labels_

        stability_metrics = calculate_model_metrics(
            X=X_sample,
            labels=stability_labels,
            source_df=sample_df,
            random_state=run_seed + k,
            silhouette_sample_size=(
                SILHOUETTE_SAMPLE_STABILITY
            ),
        )

        stability_metrics["run_number"] = run_number
        stability_metrics["k"] = k

        stability_metrics_rows.append(
            stability_metrics
        )

        stability_assignment_df = sample_df[
            [PRIMARY_KEY]
        ].copy()

        stability_assignment_df["cluster_id"] = (
            stability_labels
        )

        stability_assignments_by_k[k].append(
            stability_assignment_df
        )

        stability_centres_by_k[k].append(
            stability_model.cluster_centers_.copy()
        )

    del sample_df
    del X_sample

    gc.collect()

stability_metrics_df = pd.DataFrame(
    stability_metrics_rows
)

stability_summary_rows = []

for k in K_VALUES:
    ari_scores = []
    centroid_distances = []

    for run_a, run_b in combinations(
        range(N_STABILITY_RUNS),
        2,
    ):
        assignments_a = stability_assignments_by_k[k][
            run_a
        ].rename(
            columns={
                "cluster_id": "cluster_a",
            }
        )

        assignments_b = stability_assignments_by_k[k][
            run_b
        ].rename(
            columns={
                "cluster_id": "cluster_b",
            }
        )

        overlap = assignments_a.merge(
            assignments_b,
            on=PRIMARY_KEY,
            how="inner",
            validate="one_to_one",
        )

        if len(overlap) >= 2:
            ari_scores.append(
                adjusted_rand_score(
                    overlap["cluster_a"],
                    overlap["cluster_b"],
                )
            )

        centroid_distances.append(
            centroid_matching_distance(
                stability_centres_by_k[k][run_a],
                stability_centres_by_k[k][run_b],
            )
        )

    stability_subset = stability_metrics_df.loc[
        stability_metrics_df["k"] == k
    ].copy()

    stability_summary_rows.append(
        {
            "k": k,
            "mean_pairwise_ARI": (
                float(np.mean(ari_scores))
                if ari_scores
                else np.nan
            ),
            "min_pairwise_ARI": (
                float(np.min(ari_scores))
                if ari_scores
                else np.nan
            ),
            "mean_centroid_matching_distance": (
                float(np.mean(centroid_distances))
                if centroid_distances
                else np.nan
            ),
            "mean_stability_country_silhouette": (
                stability_subset[
                    "mean_country_silhouette"
                ].mean()
            ),
        }
    )

stability_summary_df = pd.DataFrame(
    stability_summary_rows
)


# ------------------------------------------------------------------------------
# 11. AUTOMATIC K SELECTION
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("AUTOMATIC K SELECTION")
print("=" * 90)

dashboard = full_metrics_df.merge(
    stability_summary_df,
    on="k",
    how="left",
    validate="one_to_one",
).sort_values(
    "k"
).reset_index(drop=True)

(
    SELECTED_K,
    dashboard,
    fallback_used,
) = select_k_automatically(
    dashboard
)

dashboard[
    "stability_sampling_method"
] = stability_sampling_method

dashboard[
    "stability_sample_per_country"
] = actual_sample_per_country

dashboard.to_csv(
    CLUSTERING_K_SELECTION_CSV,
    index=False,
    encoding="utf-8-sig",
)

display_columns = [
    "k",
    "mean_country_silhouette",
    "davies_bouldin_score",
    "mean_pairwise_ARI",
    "min_pairwise_ARI",
    "mean_centroid_matching_distance",
    "min_cluster_share",
    "max_cluster_share",
    "selection_candidate",
    "automatic_selection_score",
    "automatic_selection_rank",
    "selected_automatically",
]

display(
    dashboard[
        [
            column
            for column in display_columns
            if column in dashboard.columns
        ]
    ].style.format(
        {
            "mean_country_silhouette": "{:.4f}",
            "davies_bouldin_score": "{:.4f}",
            "mean_pairwise_ARI": "{:.4f}",
            "min_pairwise_ARI": "{:.4f}",
            "mean_centroid_matching_distance": "{:.4f}",
            "min_cluster_share": "{:.2%}",
            "max_cluster_share": "{:.2%}",
            "automatic_selection_score": "{:.3f}",
        }
    )
)

print(
    f"\nAutomatically selected K = {SELECTED_K}"
)

if fallback_used:
    print(
        "\nWarning: no K passed both guardrails. "
        "The highest-ranked available K was selected."
    )


# ------------------------------------------------------------------------------
# 12. FIT FINAL MODEL
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("FITTING FINAL SELECTED MODEL")
print("=" * 90)

final_model = fit_kmeans(
    X=X_weighted,
    n_clusters=SELECTED_K,
    random_state=BASE_RANDOM_SEED,
    sample_weight=country_balanced_weights,
)

final_labels = final_model.labels_

cluster_code_map = create_cluster_code_map(
    final_labels
)

selected_dashboard_row = dashboard.loc[
    dashboard["selected_automatically"]
].iloc[0]

model_bundle = {
    "model_type": (
        "Country-balanced weighted KMeans "
        "using dictionary-driven POI and GHSL features"
    ),
    "selected_k": int(SELECTED_K),
    "primary_key": PRIMARY_KEY,
    "road_class_column": ROAD_CLASS_COLUMN,
    "road_length_column": ROAD_LENGTH_COLUMN,
    "road_length_unit": "metres",
    "excluded_road_classes": sorted(
        EXCLUDED_ROAD_CLASSES
    ),
    "cluster_Z_definition": (
        "Exact cleaned road_class = motorway; "
        "excluded from K-Means due to pedestrian-risk context."
    ),
    "model_feature_config": MODEL_FEATURE_CONFIG,
    "feature_metadata": feature_metadata_df,
    "active_model_features": active_model_features,
    "numeric_weighted_mean": numeric_weighted_mean,
    "numeric_weighted_scale": numeric_weighted_scale,
    "country_weight_map": country_weight_map,
    "cluster_code_map": cluster_code_map,
    "automatic_k_selection": {
        "selected_k": int(SELECTED_K),
        "fallback_used": bool(fallback_used),
        "minimum_cluster_share_guardrail": (
            MIN_CLUSTER_SHARE_GUARDRAIL
        ),
        "minimum_pairwise_ARI_guardrail": (
            MIN_PAIRWISE_ARI_GUARDRAIL
        ),
        "stability_sampling_method": (
            stability_sampling_method
        ),
        "stability_sample_per_country": int(
            actual_sample_per_country
        ),
        "selected_k_metrics": {
            "mean_country_silhouette": float(
                selected_dashboard_row[
                    "mean_country_silhouette"
                ]
            ),
            "davies_bouldin_score": float(
                selected_dashboard_row[
                    "davies_bouldin_score"
                ]
            ),
            "mean_pairwise_ARI": float(
                selected_dashboard_row[
                    "mean_pairwise_ARI"
                ]
            ),
            "min_pairwise_ARI": float(
                selected_dashboard_row[
                    "min_pairwise_ARI"
                ]
            ),
            "min_cluster_share": float(
                selected_dashboard_row[
                    "min_cluster_share"
                ]
            ),
            "max_cluster_share": float(
                selected_dashboard_row[
                    "max_cluster_share"
                ]
            ),
        },
    },
    "automatic_k_selection_table": dashboard,
    "kmeans_model": final_model,
}

joblib.dump(
    model_bundle,
    CLUSTERING_MODEL_FILE,
)

print("Reusable model saved:")
print(CLUSTERING_MODEL_FILE)


# ------------------------------------------------------------------------------
# 13. CREATE FINAL CLUSTER ASSIGNMENTS
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("ASSIGNING CLUSTER CODES")
print("=" * 90)

learned_assignment_df = model_df[
    [PRIMARY_KEY]
].copy()

learned_assignment_df[NEW_CLUSTER_COLUMN] = [
    f"Cluster {cluster_code_map[int(label)]}"
    for label in final_labels
]

motorway_assignment_df = motorway_df[
    [PRIMARY_KEY]
].copy()

motorway_assignment_df[
    NEW_CLUSTER_COLUMN
] = "Cluster Z"

assignment_df = pd.concat(
    [
        learned_assignment_df,
        motorway_assignment_df,
    ],
    ignore_index=True,
)

if assignment_df[PRIMARY_KEY].isna().any():
    raise RuntimeError(
        "Final assignment table has missing primary keys."
    )

if assignment_df[PRIMARY_KEY].duplicated().any():
    duplicate_ids = assignment_df.loc[
        assignment_df[PRIMARY_KEY].duplicated(
            keep=False
        ),
        PRIMARY_KEY,
    ].head(20).tolist()

    raise RuntimeError(
        "Duplicate analysis_segment_id values found "
        "in final assignments.\n"
        f"Examples: {duplicate_ids}"
    )

valid_primary_keys = set(
    df_features[PRIMARY_KEY]
)

assigned_primary_keys = set(
    assignment_df[PRIMARY_KEY]
)

missing_assignment_ids = (
    valid_primary_keys
    - assigned_primary_keys
)

if missing_assignment_ids:
    raise RuntimeError(
        f"{len(missing_assignment_ids):,} valid road segment(s) "
        "have no cluster assignment.\n"
        f"Examples: {list(missing_assignment_ids)[:20]}"
    )

print(
    f"K-Means assignments : "
    f"{len(learned_assignment_df):,}"
)

print(
    f"Cluster Z assignments: "
    f"{len(motorway_assignment_df):,}"
)


# ------------------------------------------------------------------------------
# 14. ATTACH CLUSTER CODE USING analysis_segment_id
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("ADDING CLUSTER FIELD TO ROAD DATA")
print("=" * 90)

cluster_lookup = assignment_df.set_index(
    PRIMARY_KEY
)[NEW_CLUSTER_COLUMN]

gdf_output = gdf_original.copy()

if NEW_CLUSTER_COLUMN in gdf_output.columns:
    print(
        f"Existing '{NEW_CLUSTER_COLUMN}' found. "
        "It will be overwritten."
    )

gdf_output[NEW_CLUSTER_COLUMN] = gdf_output[
    PRIMARY_KEY
].map(cluster_lookup)

gdf_output[NEW_CLUSTER_COLUMN] = (
    gdf_output[NEW_CLUSTER_COLUMN]
    .astype(object)
)

gdf_output.loc[
    gdf_output[NEW_CLUSTER_COLUMN].isna(),
    NEW_CLUSTER_COLUMN,
] = None

unassigned_valid_count = int(
    gdf_output.loc[
        valid_mask,
        NEW_CLUSTER_COLUMN,
    ]
    .isna()
    .sum()
)

if unassigned_valid_count > 0:
    raise RuntimeError(
        f"{unassigned_valid_count:,} valid segment(s) could "
        "not be assigned using analysis_segment_id."
    )

print("\nCluster distribution:")
display(
    gdf_output.loc[
        valid_mask,
        NEW_CLUSTER_COLUMN,
    ]
    .value_counts()
    .rename_axis(NEW_CLUSTER_COLUMN)
    .reset_index(name="segment_count")
)


# ------------------------------------------------------------------------------
# 15. BUILD CLUSTER PROFILE
#
# Profile values use prepared features:
# - POI counts: original counts per km
# - POI distances: raw capped metres
# - GHSL: original percentages
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("BUILDING CLUSTER PROFILE")
print("=" * 90)

profile_source_df = df_features.copy()

profile_source_df[NEW_CLUSTER_COLUMN] = (
    profile_source_df[PRIMARY_KEY].map(
        cluster_lookup
    )
)

if profile_source_df[
    NEW_CLUSTER_COLUMN
].isna().any():
    raise RuntimeError(
        "A valid road segment is missing a cluster code "
        "during profile creation."
    )

profile_rows = []

for cluster_code in sort_cluster_codes(
    profile_source_df[
        NEW_CLUSTER_COLUMN
    ]
    .dropna()
    .unique()
    .tolist()
):
    cluster_df = profile_source_df.loc[
        profile_source_df[NEW_CLUSTER_COLUMN]
        == cluster_code
    ].copy()

    assignment_type = (
        "Rule-based motorway exclusion"
        if cluster_code == "Cluster Z"
        else "K-Means POI + GHSL context archetype"
    )

    profile_row = {
        NEW_CLUSTER_COLUMN: cluster_code,
        "assignment_type": assignment_type,
        "segment_count": len(cluster_df),
        "share_of_valid_roads_pct": round(
            100
            * len(cluster_df)
            / len(profile_source_df),
            2,
        ),
        "thailand_share_pct": round(
            100
            * cluster_df[COUNTRY_COLUMN]
            .eq(THAILAND_LABEL)
            .mean(),
            2,
        ),
        "maharashtra_share_pct": round(
            100
            * cluster_df[COUNTRY_COLUMN]
            .eq(MAHARASHTRA_LABEL)
            .mean(),
            2,
        ),
        "median_shape_length_m": round(
            cluster_df[ROAD_LENGTH_COLUMN].median(),
            2,
        ),
    }

    for _, metadata_row in (
        feature_metadata_df.iterrows()
    ):
        prepared_column = metadata_row[
            "prepared_column"
        ]

        profile_round_digits = int(
            metadata_row[
                "profile_round_digits"
            ]
        )

        profile_row[
            f"median__{prepared_column}"
        ] = round(
            cluster_df[prepared_column].median(),
            profile_round_digits,
        )

    profile_rows.append(profile_row)

cluster_profile_df = pd.DataFrame(
    profile_rows
)

cluster_profile_df.to_csv(
    CLUSTERING_PROFILE_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(cluster_profile_df)


# ------------------------------------------------------------------------------
# 16. SAVE FINAL GEOPACKAGE
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("SAVING FINAL CLUSTERED GEOPACKAGE")
print("=" * 90)

for extension in [
    "",
    "-wal",
    "-shm",
]:
    output_file = CLUSTERING_RESULT + extension

    if os.path.exists(output_file):
        os.remove(output_file)

gdf_output.to_file(
    CLUSTERING_RESULT,
    layer=CLUSTERING_RESULT_LAYER,
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("Saved GeoPackage:")
print(CLUSTERING_RESULT)

print("\nSaved K-selection dashboard:")
print(CLUSTERING_K_SELECTION_CSV)

print("\nSaved cluster profile:")
print(CLUSTERING_PROFILE_CSV)

print("\nSaved feature metadata:")
print(CLUSTERING_FEATURE_METADATA_CSV)

print("\nSaved feature audit:")
print(CLUSTERING_FEATURE_AUDIT_CSV)

print("\nSaved feature configuration:")
print(CLUSTERING_FEATURE_CONFIG_JSON)


# ------------------------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("STEP 5 COMPLETED — POI + GHSL CONTEXT CLUSTERING")
print("=" * 90)

print(f"Selected K: {SELECTED_K}")

print("\nSelected K result:")
display(
    dashboard.loc[
        dashboard["selected_automatically"]
    ]
)

print("\nCluster profile:")
display(cluster_profile_df)

print("\nOutput preview:")
preview_columns = [
    column
    for column in [
        PRIMARY_KEY,
        COUNTRY_COLUMN,
        ROAD_CLASS_COLUMN,
        "built_up_share_pct",
        "built_up_continuity_pct",
        NEW_CLUSTER_COLUMN,
    ]
    if column in gdf_output.columns
]

display(
    gdf_output[
        preview_columns
    ].head(10)
)


# ------------------------------------------------------------------------------
# 18. CLEAR TEMPORARY MEMORY
# ------------------------------------------------------------------------------
del gdf_valid
del df_valid
del df_features
del model_feature_df
del motorway_df
del model_df
del numeric_matrix
del X_weighted
del learned_assignment_df
del motorway_assignment_df
del assignment_df
del profile_source_df
del gdf_output

gc.collect()

In [ ]:
# ==============================================================================
# STEP 5A — INTERACTIVE CLUSTER NAMING
#
# Reads:
#   /content/5. POI and GHSL Context Clustering/
#   ADB_Innovation_merge_valid_5.gpkg
#
# Adds / updates:
# - poi_ghsl_context_cluster_name
#
# Removes, if present:
# - poi_ghsl_context_cluster_insight
#
# The same GeoPackage is replaced only after the new cluster-name
# field is validated successfully.
# ==============================================================================


# ------------------------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------------------------
PRIMARY_KEY = "analysis_segment_id"
COUNTRY_COLUMN = "dataset"
ROAD_LENGTH_COLUMN = "shape_length"

CLUSTER_CODE_COLUMN = (
    "poi_ghsl_context_cluster_code"
)

CLUSTER_NAME_COLUMN = (
    "poi_ghsl_context_cluster_name"
)

LEGACY_INSIGHT_COLUMN = (
    "poi_ghsl_context_cluster_insight"
)

THAILAND_LABEL = "Thailand"
MAHARASHTRA_LABEL = "Maharashtra"

# Use Cell 2 output when available.
CLUSTERED_GPKG = globals().get(
    "CLUSTERING_RESULT",
    (
        "/content/5. POI and GHSL Context Clustering/"
        "ADB_Innovation_merge_valid_5.gpkg"
    ),
)

# Keep True to also name Cluster Z.
INCLUDE_CLUSTER_Z = True

# When rerunning the cell:
# pressing Enter keeps the existing cluster name.
KEEP_EXISTING_VALUE_ON_BLANK = True

# Remove the former insight field if it exists.
REMOVE_LEGACY_INSIGHT_COLUMN = True


# ------------------------------------------------------------------------------
# 1. PROFILE CONFIGURATION
#
# Mirrors the Step 5 clustering interpretation:
# - POI density: count per km
# - POI distance: median distance capped at search radius
# - GHSL: built-up share and continuity
# ------------------------------------------------------------------------------
POI_PROFILE_CONFIG = [
    {
        "poi_group": "public transport",
        "count_column": (
            "public_transport_count_250m"
        ),
        "distance_column": (
            "public_transport_median_distance_m_250m"
        ),
        "radius_cap_m": 250,
    },
    {
        "poi_group": "office",
        "count_column": "office_count_250m",
        "distance_column": (
            "office_median_distance_m_250m"
        ),
        "radius_cap_m": 250,
    },
    {
        "poi_group": "school",
        "count_column": "school_count_250m",
        "distance_column": (
            "school_median_distance_m_250m"
        ),
        "radius_cap_m": 250,
    },
    {
        "poi_group": "commercial",
        "count_column": "commercial_count_250m",
        "distance_column": (
            "commercial_median_distance_m_250m"
        ),
        "radius_cap_m": 250,
    },
    {
        "poi_group": "industrial",
        "count_column": "industrial_count_1000m",
        "distance_column": (
            "industrial_median_distance_m_1000m"
        ),
        "radius_cap_m": 1000,
    },
]

GHSL_COLUMNS = [
    "built_up_share_pct",
    "built_up_continuity_pct",
]


# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def sort_cluster_codes(cluster_codes):
    """
    Sort Cluster A, Cluster B, etc.
    Keep Cluster Z last.
    """
    return sorted(
        cluster_codes,
        key=lambda value: (
            str(value).strip().upper() == "CLUSTER Z",
            str(value),
        ),
    )


def clean_saved_text(value):
    """
    Convert null-like values into empty strings.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return text


def get_existing_cluster_name(
    frame,
    cluster_code,
):
    """
    Retrieve the most common saved cluster name.

    This supports rerunning the cell without losing previously
    saved names.
    """
    if CLUSTER_NAME_COLUMN not in frame.columns:
        return ""

    values = (
        frame.loc[
            frame[CLUSTER_CODE_COLUMN] == cluster_code,
            CLUSTER_NAME_COLUMN,
        ]
        .map(clean_saved_text)
    )

    values = values.loc[
        values.ne("")
    ]

    if values.empty:
        return ""

    return values.value_counts().index[0]


def format_number(value, digits=2):
    if pd.isna(value):
        return "N/A"

    return f"{float(value):,.{digits}f}"


def format_percentage(value, digits=2):
    if pd.isna(value):
        return "N/A"

    return f"{float(value):.{digits}f}%"


def prepare_profile_features(valid_df):
    """
    Recreate interpretable Step 5 values:
    - POI count per km
    - capped median POI distance
    - GHSL values with country/global fallback
    """
    profile_df = valid_df.copy()

    profile_df[ROAD_LENGTH_COLUMN] = pd.to_numeric(
        profile_df[ROAD_LENGTH_COLUMN],
        errors="coerce",
    )

    invalid_length_mask = (
        profile_df[ROAD_LENGTH_COLUMN].isna()
        | (profile_df[ROAD_LENGTH_COLUMN] <= 0)
    )

    if invalid_length_mask.any():
        raise ValueError(
            f"{invalid_length_mask.sum():,} valid segment(s) "
            f"have missing or non-positive "
            f"'{ROAD_LENGTH_COLUMN}'."
        )

    profile_df["_road_length_km"] = (
        profile_df[ROAD_LENGTH_COLUMN] / 1000.0
    )

    for poi_config in POI_PROFILE_CONFIG:
        poi_group = poi_config["poi_group"]
        count_column = poi_config["count_column"]
        distance_column = poi_config["distance_column"]
        radius_cap_m = poi_config["radius_cap_m"]

        count_numeric = pd.to_numeric(
            profile_df[count_column],
            errors="coerce",
        ).fillna(0).clip(lower=0)

        density_column = (
            f"_{poi_group}_density_per_km"
        )

        profile_df[density_column] = (
            count_numeric
            / profile_df["_road_length_km"]
        )

        raw_distance = pd.to_numeric(
            profile_df[distance_column],
            errors="coerce",
        )

        invalid_distance_mask = (
            raw_distance.isna()
            | (raw_distance < 0)
            | (count_numeric <= 0)
        )

        capped_distance_column = (
            f"_{poi_group}_median_distance_capped_m"
        )

        profile_df[capped_distance_column] = (
            raw_distance
            .clip(
                lower=0,
                upper=radius_cap_m,
            )
            .where(
                ~invalid_distance_mask,
                radius_cap_m,
            )
            .fillna(radius_cap_m)
        )

    for ghsl_column in GHSL_COLUMNS:
        ghsl_values = pd.to_numeric(
            profile_df[ghsl_column],
            errors="coerce",
        ).clip(
            lower=0,
            upper=100,
        )

        country_median = (
            ghsl_values.groupby(
                profile_df[COUNTRY_COLUMN]
            )
            .transform("median")
        )

        global_median = ghsl_values.median()

        if pd.isna(global_median):
            raise ValueError(
                f"All values are missing for "
                f"'{ghsl_column}'."
            )

        profile_df[ghsl_column] = (
            ghsl_values
            .fillna(country_median)
            .fillna(global_median)
        )

    return profile_df


def print_cluster_profile(
    cluster_code,
    cluster_df,
    total_valid_segments,
):
    """
    Print one cluster profile vertically before the naming prompt.
    """
    segment_count = len(cluster_df)

    thailand_share = (
        100
        * cluster_df[COUNTRY_COLUMN]
        .eq(THAILAND_LABEL)
        .mean()
    )

    maharashtra_share = (
        100
        * cluster_df[COUNTRY_COLUMN]
        .eq(MAHARASHTRA_LABEL)
        .mean()
    )

    print("\n" + "=" * 90)
    print(cluster_code.upper())
    print("=" * 90)

    print(
        f"- Segment count: {segment_count:,} "
        f"({100 * segment_count / total_valid_segments:.2f}% "
        f"of valid road segments)"
    )

    print(
        f"  - Thailand: {thailand_share:.2f}%"
    )

    print(
        f"  - Maharashtra: {maharashtra_share:.2f}%"
    )

    print(
        "- Median shape length: "
        f"{format_number(cluster_df[ROAD_LENGTH_COLUMN].median())} m"
    )

    print("- Median POI density per km:")

    for poi_config in POI_PROFILE_CONFIG:
        poi_group = poi_config["poi_group"]

        density_column = (
            f"_{poi_group}_density_per_km"
        )

        print(
            f"  - {poi_group.title()}: "
            f"{format_number(cluster_df[density_column].median(), 4)}"
        )

    print("- Median POI distance to road segment:")

    for poi_config in POI_PROFILE_CONFIG:
        poi_group = poi_config["poi_group"]
        radius_cap_m = poi_config["radius_cap_m"]

        capped_distance_column = (
            f"_{poi_group}_median_distance_capped_m"
        )

        print(
            f"  - {poi_group.title()}: "
            f"{format_number(cluster_df[capped_distance_column].median())} m "
            f"(cap: {radius_cap_m:,} m)"
        )

    print(
        "- Median built-up share: "
        f"{format_percentage(cluster_df['built_up_share_pct'].median())}"
    )

    print(
        "- Median built-up continuity: "
        f"{format_percentage(cluster_df['built_up_continuity_pct'].median())}"
    )


def request_cluster_name(
    cluster_code,
    existing_name,
):
    """
    Ask the user for one cluster name.
    """
    if existing_name:
        prompt = (
            f"\nEnter final name for {cluster_code}\n"
            f"Current name: {existing_name}\n"
            "New name (press Enter to keep current): "
        )
    else:
        prompt = (
            f"\nEnter final name for {cluster_code}\n"
            "Example: LAND USE A — Low-activity / open context\n"
            "New name: "
        )

    try:
        submitted_name = input(prompt).strip()

    except EOFError:
        raise RuntimeError(
            "Interactive input is unavailable. "
            "Run this cell directly in Google Colab."
        )

    if (
        not submitted_name
        and KEEP_EXISTING_VALUE_ON_BLANK
    ):
        return existing_name

    return submitted_name


# ------------------------------------------------------------------------------
# 3. LOAD AND VALIDATE CLUSTERED GEOPACKAGE
# ------------------------------------------------------------------------------
if not os.path.exists(CLUSTERED_GPKG):
    raise FileNotFoundError(
        f"Clustered GeoPackage was not found:\n"
        f"{CLUSTERED_GPKG}\n\n"
        "Please run Step 5 first."
    )

print("=" * 90)
print("STEP 5A — INTERACTIVE CLUSTER NAMING")
print("=" * 90)

input_layer_name = get_first_layer_name(
    CLUSTERED_GPKG
)

gdf_original = gpd.read_file(
    CLUSTERED_GPKG,
    layer=input_layer_name,
    engine="pyogrio",
).copy()

required_columns = [
    PRIMARY_KEY,
    COUNTRY_COLUMN,
    ROAD_LENGTH_COLUMN,
    CLUSTER_CODE_COLUMN,
]

for poi_config in POI_PROFILE_CONFIG:
    required_columns.extend(
        [
            poi_config["count_column"],
            poi_config["distance_column"],
        ]
    )

required_columns.extend(GHSL_COLUMNS)

missing_columns = [
    column
    for column in required_columns
    if column not in gdf_original.columns
]

if missing_columns:
    raise KeyError(
        "The clustered GeoPackage is missing required "
        "profile field(s):\n"
        + "\n".join(
            f"- {column}"
            for column in missing_columns
        )
    )

if gdf_original[PRIMARY_KEY].isna().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' contains missing values."
    )

if gdf_original[PRIMARY_KEY].duplicated().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' must be unique."
    )

print(f"GeoPackage: {CLUSTERED_GPKG}")
print(f"Layer     : {input_layer_name}")
print(f"Records   : {len(gdf_original):,}")


# ------------------------------------------------------------------------------
# 4. FILTER VALID SEGMENTS AND BUILD PROFILE METRICS
# ------------------------------------------------------------------------------
if "analysis_status" in gdf_original.columns:
    valid_mask = (
        gdf_original["analysis_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("valid")
    )
else:
    valid_mask = pd.Series(
        True,
        index=gdf_original.index,
    )

gdf_valid = gdf_original.loc[
    valid_mask
].copy()

if gdf_valid.empty:
    raise ValueError(
        "No valid road segments are available."
    )

gdf_valid = gdf_valid.loc[
    gdf_valid[CLUSTER_CODE_COLUMN].notna()
].copy()

if gdf_valid.empty:
    raise ValueError(
        f"No valid records contain "
        f"'{CLUSTER_CODE_COLUMN}'."
    )

profile_df = prepare_profile_features(
    valid_df=gdf_valid
)

available_clusters = sort_cluster_codes(
    profile_df[CLUSTER_CODE_COLUMN]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if not INCLUDE_CLUSTER_Z:
    available_clusters = [
        cluster_code
        for cluster_code in available_clusters
        if cluster_code != "Cluster Z"
    ]

if not available_clusters:
    raise ValueError(
        "No clusters are available for naming."
    )

print(
    f"\nNumber of clusters available: "
    f"{len(available_clusters):,}"
)

print("\nAvailable clusters:")

for cluster_code in available_clusters:
    cluster_count = int(
        (
            profile_df[CLUSTER_CODE_COLUMN]
            == cluster_code
        ).sum()
    )

    print(
        f"- {cluster_code}: "
        f"{cluster_count:,} valid segment(s)"
    )


# ------------------------------------------------------------------------------
# 5. SHOW CLUSTER PROFILES AND COLLECT NAMES
# ------------------------------------------------------------------------------
cluster_naming_records = []

for cluster_code in available_clusters:
    cluster_df = profile_df.loc[
        profile_df[CLUSTER_CODE_COLUMN]
        == cluster_code
    ].copy()

    print_cluster_profile(
        cluster_code=cluster_code,
        cluster_df=cluster_df,
        total_valid_segments=len(profile_df),
    )

    existing_name = get_existing_cluster_name(
        frame=gdf_original,
        cluster_code=cluster_code,
    )

    cluster_name = request_cluster_name(
        cluster_code=cluster_code,
        existing_name=existing_name,
    )

    cluster_naming_records.append(
        {
            CLUSTER_CODE_COLUMN: cluster_code,
            CLUSTER_NAME_COLUMN: cluster_name,
        }
    )

cluster_naming_df = pd.DataFrame(
    cluster_naming_records
)

print("\n" + "=" * 90)
print("FINAL CLUSTER NAMING TABLE")
print("=" * 90)

display(cluster_naming_df)


# ------------------------------------------------------------------------------
# 6. APPLY NAMES TO EVERY MATCHING ROAD SEGMENT
# ------------------------------------------------------------------------------
cluster_name_lookup = (
    cluster_naming_df.set_index(
        CLUSTER_CODE_COLUMN
    )[CLUSTER_NAME_COLUMN]
    .to_dict()
)

gdf_output = gdf_original.copy()

gdf_output[CLUSTER_NAME_COLUMN] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(cluster_name_lookup)
)

gdf_output[CLUSTER_NAME_COLUMN] = (
    gdf_output[CLUSTER_NAME_COLUMN]
    .astype(object)
)

# Do not assign names to invalid/non-clustered records.
gdf_output.loc[
    ~valid_mask,
    CLUSTER_NAME_COLUMN,
] = None

# Remove the old insight field to keep the output simple.
if (
    REMOVE_LEGACY_INSIGHT_COLUMN
    and LEGACY_INSIGHT_COLUMN in gdf_output.columns
):
    gdf_output = gdf_output.drop(
        columns=[LEGACY_INSIGHT_COLUMN]
    )

missing_name_count = int(
    gdf_output.loc[
        valid_mask
        & gdf_output[CLUSTER_CODE_COLUMN].notna(),
        CLUSTER_NAME_COLUMN,
    ]
    .isna()
    .sum()
)

blank_name_count = int(
    gdf_output.loc[
        valid_mask
        & gdf_output[CLUSTER_CODE_COLUMN].notna(),
        CLUSTER_NAME_COLUMN,
    ]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

if missing_name_count > 0 or blank_name_count > 0:
    raise RuntimeError(
        "Every valid cluster must have a non-empty name before "
        "replacing the GeoPackage.\n"
        f"Missing names: {missing_name_count:,}\n"
        f"Blank names: {blank_name_count:,}"
    )


# ------------------------------------------------------------------------------
# 7. SAFELY REPLACE THE SAME GEOPACKAGE
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("REPLACING THE CLUSTERED GEOPACKAGE")
print("=" * 90)

gpkg_folder = os.path.dirname(
    CLUSTERED_GPKG
)

gpkg_stem = Path(
    CLUSTERED_GPKG
).stem

temporary_gpkg = os.path.join(
    gpkg_folder,
    f"__temporary_{gpkg_stem}.gpkg",
)

for temporary_file in [
    temporary_gpkg,
    temporary_gpkg + "-wal",
    temporary_gpkg + "-shm",
]:
    if os.path.exists(temporary_file):
        os.remove(temporary_file)

# Write temporary file first.
gdf_output.to_file(
    temporary_gpkg,
    layer=input_layer_name,
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

# Validate temporary output before replacing the original.
gdf_check = gpd.read_file(
    temporary_gpkg,
    layer=input_layer_name,
    engine="pyogrio",
)

if CLUSTER_NAME_COLUMN not in gdf_check.columns:
    raise RuntimeError(
        f"Temporary output is missing "
        f"'{CLUSTER_NAME_COLUMN}'. "
        "Original GeoPackage was not replaced."
    )

if (
    REMOVE_LEGACY_INSIGHT_COLUMN
    and LEGACY_INSIGHT_COLUMN in gdf_check.columns
):
    raise RuntimeError(
        f"Temporary output still contains "
        f"'{LEGACY_INSIGHT_COLUMN}'. "
        "Original GeoPackage was not replaced."
    )

if len(gdf_check) != len(gdf_original):
    raise RuntimeError(
        "Temporary output row count does not match "
        "the original GeoPackage. "
        "Original file was not replaced."
    )

if gdf_check[PRIMARY_KEY].duplicated().any():
    raise RuntimeError(
        "Temporary output contains duplicate "
        "analysis_segment_id values. "
        "Original file was not replaced."
    )

del gdf_check
gc.collect()

# Remove GeoPackage sidecar files before direct replacement.
for sidecar_file in [
    CLUSTERED_GPKG + "-wal",
    CLUSTERED_GPKG + "-shm",
]:
    if os.path.exists(sidecar_file):
        os.remove(sidecar_file)

os.replace(
    temporary_gpkg,
    CLUSTERED_GPKG,
)

print("GeoPackage replaced successfully:")
print(CLUSTERED_GPKG)


# ------------------------------------------------------------------------------
# 8. FINAL PREVIEW
# ------------------------------------------------------------------------------
print("\nUpdated field preview:")

preview_columns = [
    column
    for column in [
        PRIMARY_KEY,
        COUNTRY_COLUMN,
        CLUSTER_CODE_COLUMN,
        CLUSTER_NAME_COLUMN,
    ]
    if column in gdf_output.columns
]

display(
    gdf_output.loc[
        valid_mask,
        preview_columns,
    ]
    .sort_values(
        [
            CLUSTER_CODE_COLUMN,
            PRIMARY_KEY,
        ]
    )
    .head(20)
)

del gdf_valid
del profile_df
del gdf_output

gc.collect()

# SPEED LIMIT DETERMINATION AND SCORING

## Speed Limit Determination

In [ ]:
# ==============================================================================
# STEP 6 — TARGET SPEED LIMIT BY INTERSECTION DENSITY
#
# PROCESS
# ------------------------------------------------------------------------------
# 1. Load Step 5 clustered roads.
# 2. Display distinct Cluster Code and Cluster Name combinations.
# 3. Ask the user to define, for each cluster:
#    - Initial speed limit (km/h)
#    - Surrounding area: rural-like or built-up
# 4. Categorise intersection_count_per_km:
#
#    Rural-like:
#    - Low      : exactly 0 intersections/km
#    - Moderate : more than 0 and less than 1 intersections/km
#    - High     : 1 or more intersections/km
#
#    Built-up:
#    - Low      : less than 2 intersections/km
#    - Moderate : 2 to less than 6 intersections/km
#    - High     : 6 or more intersections/km
#
# 5. Target speed logic:
#    - Cluster Z: retain user-input initial speed.
#    - All other clusters:
#        High intersection density -> min(50, initial speed)
#        Moderate / Low            -> retain initial speed
#
# 6. Save final GeoPackage with:
#    - target_speed_limit
#    - intersection_density
# ==============================================================================


# ------------------------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------------------------
PRIMARY_KEY = "analysis_segment_id"

COUNTRY_COLUMN = "dataset"
CLUSTER_CODE_COLUMN = (
    "poi_ghsl_context_cluster_code"
)
CLUSTER_NAME_COLUMN = (
    "poi_ghsl_context_cluster_name"
)

INTERSECTION_COLUMN = (
    "intersection_count_per_km"
)

TARGET_SPEED_COLUMN = "target_speed_limit"
INTERSECTION_DENSITY_COLUMN = (
    "intersection_density"
)

MOTORWAY_CLUSTER_CODE = "Cluster Z"

VALID_SURROUNDING_AREAS = {
    "rural-like": "rural-like",
    "rural like": "rural-like",
    "rural": "rural-like",
    "built-up": "built-up",
    "built up": "built-up",
    "builtup": "built-up",
}


# ------------------------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    """
    Return the first layer name inside a GeoPackage.
    """
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def sort_cluster_codes(cluster_codes):
    """
    Sort Cluster A, Cluster B, etc.
    Keep Cluster Z last.
    """
    return sorted(
        cluster_codes,
        key=lambda value: (
            str(value).strip().upper()
            == MOTORWAY_CLUSTER_CODE.upper(),
            str(value),
        ),
    )


def clean_text(value):
    """
    Convert null-like values into an empty string.
    """
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return value


def request_initial_speed(
    cluster_code,
    cluster_name,
):
    """
    Request a whole-number speed limit in km/h.
    """
    print("\n" + "=" * 70)
    print("PLEASE FILL THE INITIAL SPEED AND SURROUNDING AREA")
    print("=" * 70)
    print(f"- Cluster code & name: {cluster_code}, {cluster_name}")
    print(
        "- Surrounding area determines speed adjustment "
        "based on intersection density."
    )

    while True:
        raw_value = input(
            "Initial speed (km/h): "
        ).strip()

        try:
            initial_speed = int(raw_value)

            if not 1 <= initial_speed <= 250:
                raise ValueError

            return initial_speed

        except ValueError:
            print(
                "Please enter a whole number between "
                "1 and 250."
            )


def request_surrounding_area():
    """
    Request rural-like or built-up context.
    """
    print("\nChoose surrounding area:")
    print("- rural-like")
    print("- built-up")

    while True:
        raw_value = input(
            "Surrounding area: "
        ).strip().lower()

        if raw_value in VALID_SURROUNDING_AREAS:
            return VALID_SURROUNDING_AREAS[
                raw_value
            ]

        print(
            "Invalid value. Enter: rural-like or built-up."
        )


def classify_intersection_density(
    intersection_count_per_km,
    surrounding_area,
):
    """
    Classify intersection density using the required thresholds.
    """
    if pd.isna(intersection_count_per_km):
        raise ValueError(
            "Intersection density cannot be classified because "
            "intersection_count_per_km is missing."
        )

    value = float(intersection_count_per_km)

    if value < 0:
        raise ValueError(
            "intersection_count_per_km cannot be negative."
        )

    if surrounding_area == "rural-like":
        # User-defined intended interpretation:
        # Low = 0
        # Moderate = >0 and <1
        # High = >=1
        if value == 0:
            return "low"

        if value < 1:
            return "moderate"

        return "high"

    if surrounding_area == "built-up":
        if value < 2:
            return "low"

        if value < 6:
            return "moderate"

        return "high"

    raise ValueError(
        f"Unsupported surrounding area: "
        f"'{surrounding_area}'"
    )


def calculate_target_speed(
    cluster_code,
    initial_speed,
    intersection_density,
):
    """
    Apply the target-speed rule.
    """
    if cluster_code == MOTORWAY_CLUSTER_CODE:
        return initial_speed

    if intersection_density == "high":
        return min(50, initial_speed)

    return initial_speed


# ------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED STEP 6 PATH VARIABLES
# ------------------------------------------------------------------------------
required_variables = [
    "TARGET_SPEED_INPUT",
    "FINAL_RESULT_OUTPUT_FOLDER",
    "TARGET_SPEED_RESULT",
    "TARGET_SPEED_RESULT_LAYER",
    "TARGET_SPEED_CLUSTER_CONFIG_CSV",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

if not os.path.exists(TARGET_SPEED_INPUT):
    raise FileNotFoundError(
        f"Step 5 input GeoPackage was not found:\n"
        f"{TARGET_SPEED_INPUT}"
    )

os.makedirs(
    FINAL_RESULT_OUTPUT_FOLDER,
    exist_ok=True,
)


# ------------------------------------------------------------------------------
# 3. LOAD STEP 5 CLUSTERED DATA
# ------------------------------------------------------------------------------
print("=" * 90)
print("STEP 6 — TARGET SPEED LIMIT BY INTERSECTION DENSITY")
print("=" * 90)

input_layer_name = get_first_layer_name(
    TARGET_SPEED_INPUT
)

gdf_original = gpd.read_file(
    TARGET_SPEED_INPUT,
    layer=input_layer_name,
    engine="pyogrio",
).copy()

required_columns = [
    PRIMARY_KEY,
    CLUSTER_CODE_COLUMN,
    CLUSTER_NAME_COLUMN,
    INTERSECTION_COLUMN,
]

missing_columns = [
    column
    for column in required_columns
    if column not in gdf_original.columns
]

if missing_columns:
    raise KeyError(
        "The Step 5 GeoPackage is missing required field(s):\n"
        + "\n".join(
            f"- {column}"
            for column in missing_columns
        )
    )

if gdf_original[PRIMARY_KEY].isna().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' contains missing values."
    )

if gdf_original[PRIMARY_KEY].duplicated().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' must be unique."
    )

print(f"Input GeoPackage: {TARGET_SPEED_INPUT}")
print(f"Input layer     : {input_layer_name}")
print(f"Road records    : {len(gdf_original):,}")


# ------------------------------------------------------------------------------
# 4. KEEP VALID CLUSTERED ROAD SEGMENTS
# ------------------------------------------------------------------------------
if "analysis_status" in gdf_original.columns:
    valid_mask = (
        gdf_original["analysis_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("valid")
    )
else:
    valid_mask = pd.Series(
        True,
        index=gdf_original.index,
    )

gdf_valid = gdf_original.loc[
    valid_mask
].copy()

gdf_valid = gdf_valid.loc[
    gdf_valid[CLUSTER_CODE_COLUMN].notna()
].copy()

if gdf_valid.empty:
    raise ValueError(
        "No valid clustered road segments are available."
    )

gdf_valid[CLUSTER_CODE_COLUMN] = (
    gdf_valid[CLUSTER_CODE_COLUMN]
    .map(clean_text)
)

gdf_valid[CLUSTER_NAME_COLUMN] = (
    gdf_valid[CLUSTER_NAME_COLUMN]
    .map(clean_text)
)

blank_cluster_name_mask = (
    gdf_valid[CLUSTER_NAME_COLUMN].eq("")
)

if blank_cluster_name_mask.any():
    missing_cluster_codes = sorted(
        gdf_valid.loc[
            blank_cluster_name_mask,
            CLUSTER_CODE_COLUMN,
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "Some clusters do not have a saved cluster name.\n"
        f"Missing name for: {missing_cluster_codes}\n\n"
        "Please run the interactive cluster naming cell first."
    )

gdf_valid[INTERSECTION_COLUMN] = pd.to_numeric(
    gdf_valid[INTERSECTION_COLUMN],
    errors="coerce",
)

invalid_intersection_mask = (
    gdf_valid[INTERSECTION_COLUMN].isna()
    | (gdf_valid[INTERSECTION_COLUMN] < 0)
)

if invalid_intersection_mask.any():
    invalid_examples = gdf_valid.loc[
        invalid_intersection_mask,
        [
            PRIMARY_KEY,
            CLUSTER_CODE_COLUMN,
            INTERSECTION_COLUMN,
        ],
    ].head(20)

    raise ValueError(
        f"{invalid_intersection_mask.sum():,} valid road segment(s) "
        "have missing or negative intersection_count_per_km.\n\n"
        f"Examples:\n{invalid_examples}"
    )


# ------------------------------------------------------------------------------
# 5. DISPLAY DISTINCT CLUSTER CODE / NAME COMBINATIONS
# ------------------------------------------------------------------------------
cluster_list_df = (
    gdf_valid[
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
        ]
    ]
    .drop_duplicates()
    .copy()
)

cluster_counts = (
    gdf_valid.groupby(
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="segment_count"
    )
)

cluster_list_df = cluster_list_df.merge(
    cluster_counts,
    on=[
        CLUSTER_CODE_COLUMN,
        CLUSTER_NAME_COLUMN,
    ],
    how="left",
    validate="one_to_one",
)

cluster_list_df["_sort_order"] = (
    cluster_list_df[CLUSTER_CODE_COLUMN]
    .map(
        {
            cluster_code: position
            for position, cluster_code in enumerate(
                sort_cluster_codes(
                    cluster_list_df[
                        CLUSTER_CODE_COLUMN
                    ].tolist()
                )
            )
        }
    )
)

cluster_list_df = (
    cluster_list_df
    .sort_values("_sort_order")
    .drop(columns="_sort_order")
    .reset_index(drop=True)
)

print("\nDistinct Cluster Code and Cluster Name:")
display(cluster_list_df)

print(
    f"\nNumber of available clusters: "
    f"{len(cluster_list_df):,}"
)


# ------------------------------------------------------------------------------
# 6. REQUEST INITIAL SPEED AND SURROUNDING AREA FOR EACH CLUSTER
# ------------------------------------------------------------------------------
cluster_config_records = []

for _, cluster_row in cluster_list_df.iterrows():
    cluster_code = cluster_row[
        CLUSTER_CODE_COLUMN
    ]

    cluster_name = cluster_row[
        CLUSTER_NAME_COLUMN
    ]

    initial_speed = request_initial_speed(
        cluster_code=cluster_code,
        cluster_name=cluster_name,
    )

    surrounding_area = request_surrounding_area()

    cluster_config_records.append(
        {
            CLUSTER_CODE_COLUMN: cluster_code,
            CLUSTER_NAME_COLUMN: cluster_name,
            "initial_speed_kph": initial_speed,
            "surrounding_area": surrounding_area,
            "intersection_density_rule": (
                "motorway_no_adjustment"
                if cluster_code == MOTORWAY_CLUSTER_CODE
                else surrounding_area
            ),
        }
    )

cluster_config_df = pd.DataFrame(
    cluster_config_records
)

print("\n" + "=" * 90)
print("USER-DEFINED TARGET SPEED CONFIGURATION")
print("=" * 90)

display(cluster_config_df)


# ------------------------------------------------------------------------------
# 7. APPLY INTERSECTION DENSITY AND TARGET SPEED LOGIC
# ------------------------------------------------------------------------------
cluster_config_lookup = (
    cluster_config_df.set_index(
        CLUSTER_CODE_COLUMN
    )
)

gdf_output = gdf_original.copy()

gdf_output[CLUSTER_CODE_COLUMN] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(clean_text)
)

gdf_output["_initial_speed_kph"] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(
        cluster_config_lookup[
            "initial_speed_kph"
        ]
    )
)

gdf_output["_surrounding_area"] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(
        cluster_config_lookup[
            "surrounding_area"
        ]
    )
)

gdf_output[INTERSECTION_COLUMN] = pd.to_numeric(
    gdf_output[INTERSECTION_COLUMN],
    errors="coerce",
)

gdf_output[INTERSECTION_DENSITY_COLUMN] = None
gdf_output[TARGET_SPEED_COLUMN] = np.nan

rows_to_process_mask = (
    valid_mask
    & gdf_output[CLUSTER_CODE_COLUMN].notna()
)

for surrounding_area in [
    "rural-like",
    "built-up",
]:
    context_mask = (
        rows_to_process_mask
        & (
            gdf_output["_surrounding_area"]
            == surrounding_area
        )
    )

    if not context_mask.any():
        continue

    gdf_output.loc[
        context_mask,
        INTERSECTION_DENSITY_COLUMN,
    ] = (
        gdf_output.loc[
            context_mask,
            INTERSECTION_COLUMN,
        ]
        .apply(
            lambda value: classify_intersection_density(
                intersection_count_per_km=value,
                surrounding_area=surrounding_area,
            )
        )
    )

# Set target speed:
# - Cluster Z: initial speed unchanged
# - High density, other clusters: min(50, initial speed)
# - Low / moderate density: initial speed unchanged
gdf_output.loc[
    rows_to_process_mask,
    TARGET_SPEED_COLUMN,
] = gdf_output.loc[
    rows_to_process_mask,
    "_initial_speed_kph",
]

high_intersection_mask = (
    rows_to_process_mask
    & (
        gdf_output[INTERSECTION_DENSITY_COLUMN]
        == "high"
    )
    & (
        gdf_output[CLUSTER_CODE_COLUMN]
        != MOTORWAY_CLUSTER_CODE
    )
)

gdf_output.loc[
    high_intersection_mask,
    TARGET_SPEED_COLUMN,
] = np.minimum(
    50,
    gdf_output.loc[
        high_intersection_mask,
        "_initial_speed_kph",
    ],
)

gdf_output[TARGET_SPEED_COLUMN] = (
    pd.to_numeric(
        gdf_output[TARGET_SPEED_COLUMN],
        errors="coerce",
    )
    .round()
    .astype("Int64")
)

gdf_output[INTERSECTION_DENSITY_COLUMN] = (
    gdf_output[INTERSECTION_DENSITY_COLUMN]
    .astype(object)
)


# ------------------------------------------------------------------------------
# 8. VALIDATE FINAL RESULTS
# ------------------------------------------------------------------------------
missing_target_speed_count = int(
    gdf_output.loc[
        rows_to_process_mask,
        TARGET_SPEED_COLUMN,
    ]
    .isna()
    .sum()
)

missing_density_count = int(
    gdf_output.loc[
        rows_to_process_mask,
        INTERSECTION_DENSITY_COLUMN,
    ]
    .isna()
    .sum()
)

if missing_target_speed_count > 0:
    raise RuntimeError(
        f"{missing_target_speed_count:,} valid road segment(s) "
        "did not receive target_speed_limit."
    )

if missing_density_count > 0:
    raise RuntimeError(
        f"{missing_density_count:,} valid road segment(s) "
        "did not receive intersection_density."
    )

print("\n" + "=" * 90)
print("TARGET SPEED SUMMARY")
print("=" * 90)

summary_df = (
    gdf_output.loc[
        rows_to_process_mask,
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
            INTERSECTION_DENSITY_COLUMN,
            TARGET_SPEED_COLUMN,
        ],
    ]
    .groupby(
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
            INTERSECTION_DENSITY_COLUMN,
            TARGET_SPEED_COLUMN,
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="segment_count"
    )
    .sort_values(
        [
            CLUSTER_CODE_COLUMN,
            INTERSECTION_DENSITY_COLUMN,
            TARGET_SPEED_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

display(summary_df)


# ------------------------------------------------------------------------------
# 9. SAVE CONFIGURATION AUDIT FILE
# ------------------------------------------------------------------------------
cluster_config_df.to_csv(
    TARGET_SPEED_CLUSTER_CONFIG_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved user-input configuration:")
print(TARGET_SPEED_CLUSTER_CONFIG_CSV)


# ------------------------------------------------------------------------------
# 10. SAVE FINAL GEOPACKAGE
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("SAVING FINAL TARGET SPEED GEOPACKAGE")
print("=" * 90)

# Remove temporary helper fields before saving.
gdf_output = gdf_output.drop(
    columns=[
        "_initial_speed_kph",
        "_surrounding_area",
    ],
    errors="ignore",
)

for extension in [
    "",
    "-wal",
    "-shm",
]:
    output_file = TARGET_SPEED_RESULT + extension

    if os.path.exists(output_file):
        os.remove(output_file)

gdf_output.to_file(
    TARGET_SPEED_RESULT,
    layer=TARGET_SPEED_RESULT_LAYER,
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("Final GeoPackage saved:")
print(TARGET_SPEED_RESULT)


# ------------------------------------------------------------------------------
# 11. FINAL PREVIEW
# ------------------------------------------------------------------------------
print("\nFinal output preview:")

preview_columns = [
    column
    for column in [
        PRIMARY_KEY,
        COUNTRY_COLUMN,
        CLUSTER_CODE_COLUMN,
        CLUSTER_NAME_COLUMN,
        INTERSECTION_COLUMN,
        INTERSECTION_DENSITY_COLUMN,
        TARGET_SPEED_COLUMN,
    ]
    if column in gdf_output.columns
]

display(
    gdf_output.loc[
        rows_to_process_mask,
        preview_columns,
    ]
    .sort_values(
        [
            CLUSTER_CODE_COLUMN,
            PRIMARY_KEY,
        ]
    )
    .head(30)
)

print("\nStep 6 completed.")

del gdf_valid
del gdf_output

gc.collect()

## Corridor Priority Scoring

In [ ]:
# ==============================================================================
# STEP 7 — FINAL RISK ASSESSMENT AND SCORING
#
# INPUT:
# /content/6. Final Result/ADB_Innovation_target_speed_limit.gpkg
#
# OUTPUT:
# /content/6. Final Result/
# ADB_Innovation_target_speed_limit_with_scoring.gpkg
#
# TOTAL RISK SCORE:
# speed_limit_target_misalignment_point
# + vru_point
# + safe_system_misalignment_point
# + intersection_density_point
# + traffic_exposure_point
#
# Maximum total score = 100
# ==============================================================================


# ------------------------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------------------------
PRIMARY_KEY = "analysis_segment_id"

COUNTRY_COLUMN = "dataset"

CLUSTER_CODE_COLUMN = (
    "poi_ghsl_context_cluster_code"
)

CLUSTER_NAME_COLUMN = (
    "poi_ghsl_context_cluster_name"
)

P85_SPEED_COLUMN = "p85_speed"
POSTED_SPEED_LIMIT_COLUMN = "speed_limit"
TARGET_SPEED_LIMIT_COLUMN = "target_speed_limit"

INTERSECTION_DENSITY_COLUMN = (
    "intersection_density"
)

RANKED_PERCENTILE_COLUMN = (
    "ranked_percentile"
)

MOTORWAY_CLUSTER_CODE = "Cluster Z"

# Replace missing, zero, or negative existing speed limits with 80 km/h.
MISSING_POSTED_SPEED_LIMIT_KPH = 80

# VRU input and scoring settings.
VRU_BASELINE_MINIMUM = 0
VRU_BASELINE_MAXIMUM = 20
VRU_POINT_MAXIMUM = 25
VRU_NEAR_DISTANCE_THRESHOLD_M = 100

# Non-industrial POI groups used for VRU proximity adjustment.
VRU_POI_DISTANCE_COLUMNS = {
    "public_transport": (
        "public_transport_min_distance_m_250m"
    ),
    "office": (
        "office_min_distance_m_250m"
    ),
    "school": (
        "school_min_distance_m_250m"
    ),
    "commercial": (
        "commercial_min_distance_m_250m"
    ),
}

# Output columns.
SPEED_LIMIT_IMPUTED_COLUMN = (
    "speed_limit_imputed"
)

# Existing / posted speed limit minus Safe System target speed.
EXCESS_ABOVE_SPEED_TARGET_COLUMN = (
    "excess_above_speed_target"
)

SPEED_TARGET_MISALIGNMENT_COLUMN = (
    "speed_limit_target_misalignment"
)

SPEED_TARGET_MISALIGNMENT_POINT_COLUMN = (
    "speed_limit_target_misalignment_point"
)

VRU_BASELINE_POINT_COLUMN = (
    "vru_baseline_point"
)

VRU_NEAR_POI_COUNT_COLUMN = (
    "vru_near_poi_count_lt_100m"
)

VRU_POI_PROXIMITY_POINT_COLUMN = (
    "vru_poi_proximity_point"
)

VRU_POINT_COLUMN = "vru_point"

# P85 speed minus Safe System target speed.
EXCESS_ABOVE_TARGET_COLUMN = (
    "excess_above_safe_system_target"
)

SAFE_SYSTEM_MISALIGNMENT_COLUMN = (
    "safe_system_misalignment"
)

SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN = (
    "safe_system_misalignment_point"
)

SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN = (
    "safe_system_misalignment_insight"
)

INTERSECTION_DENSITY_POINT_COLUMN = (
    "intersection_density_point"
)

TRAFFIC_EXPOSURE_COLUMN = (
    "traffic_exposure"
)

TRAFFIC_EXPOSURE_POINT_COLUMN = (
    "traffic_exposure_point"
)

TOTAL_RISK_POINT_COLUMN = (
    "total_risk_point"
)


# ------------------------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    """
    Return the first layer name inside a GeoPackage.
    """
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def clean_text(value):
    """
    Convert null-like values into empty strings.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return text


def sort_cluster_codes(cluster_codes):
    """
    Sort Cluster A, Cluster B, etc.
    Keep Cluster Z last.
    """
    return sorted(
        cluster_codes,
        key=lambda value: (
            str(value).strip().upper()
            == MOTORWAY_CLUSTER_CODE.upper(),
            str(value),
        ),
    )


def request_vru_baseline(
    cluster_code,
    cluster_name,
    segment_count,
):
    """
    Request a baseline VRU score from 0 to 20.
    """
    print("\n" + "=" * 75)
    print("SET BASELINE VRU POINT")
    print("=" * 75)
    print(f"- Cluster code & name: {cluster_code}, {cluster_name}")
    print(f"- Valid road segments: {segment_count:,}")
    print(
        f"- Allowed baseline VRU point: "
        f"{VRU_BASELINE_MINIMUM} to {VRU_BASELINE_MAXIMUM}"
    )

    while True:
        raw_value = input(
            "Baseline VRU point: "
        ).strip()

        try:
            baseline_point = int(raw_value)

            if (
                VRU_BASELINE_MINIMUM
                <= baseline_point
                <= VRU_BASELINE_MAXIMUM
            ):
                return baseline_point

            raise ValueError

        except ValueError:
            print(
                "Please enter a whole number from "
                f"{VRU_BASELINE_MINIMUM} to "
                f"{VRU_BASELINE_MAXIMUM}."
            )


def assess_speed_excess(
    excess_series,
    point_scheme,
):
    """
    Convert a speed excess value into:
    - misalignment label
    - misalignment point
    """
    excess_series = pd.to_numeric(
        excess_series,
        errors="coerce",
    )

    labels = pd.Series(
        None,
        index=excess_series.index,
        dtype="object",
    )

    points = pd.Series(
        np.nan,
        index=excess_series.index,
        dtype="float64",
    )

    align_mask = excess_series <= 0

    slight_mask = (
        (excess_series > 0)
        & (excess_series <= 10)
    )

    moderate_mask = (
        (excess_series > 10)
        & (excess_series <= 20)
    )

    high_mask = (
        (excess_series > 20)
        & (excess_series <= 30)
    )

    very_high_mask = excess_series > 30

    labels.loc[align_mask] = "Align"
    labels.loc[slight_mask] = "Slight Excess"
    labels.loc[moderate_mask] = "Moderate Excess"
    labels.loc[high_mask] = "High Excess"
    labels.loc[very_high_mask] = "Very High Excess"

    points.loc[align_mask] = point_scheme["align"]
    points.loc[slight_mask] = point_scheme["slight"]
    points.loc[moderate_mask] = point_scheme["moderate"]
    points.loc[high_mask] = point_scheme["high"]
    points.loc[very_high_mask] = point_scheme["very_high"]

    return labels, points


def normalise_intersection_density(value):
    """
    Step 6 may use 'moderate', while scoring may use 'medium'.
    Both represent the 5-point middle category.
    """
    text = clean_text(value).lower()

    mapping = {
        "low": "low",
        "moderate": "medium",
        "medium": "medium",
        "high": "high",
    }

    return mapping.get(text, "")


def get_safe_system_insight(
    speed_limit,
    target_speed_limit,
    p85_speed,
):
    """
    Return Safe System corridor insight.
    """
    if (
        speed_limit > target_speed_limit
        and p85_speed > target_speed_limit
    ):
        return (
            "Limit and road environment both support "
            "unsafe speed"
        )

    if (
        speed_limit <= target_speed_limit
        and p85_speed > target_speed_limit
    ):
        return (
            "Limit may be appropriate, but road is not "
            "self-explaining"
        )

    if (
        speed_limit > target_speed_limit
        and p85_speed <= target_speed_limit
    ):
        return (
            "Posted limit may be too high, but drivers "
            "already travel slowly"
        )

    return (
        "Broadly aligned at corridor scale"
    )


# ------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED PATH VARIABLES FROM CELL 2
# ------------------------------------------------------------------------------
required_variables = [
    "SCORING_INPUT",
    "SCORING_RESULT",
    "SCORING_RESULT_LAYER",
    "SCORING_CLUSTER_VRU_CONFIG_CSV",
    "SCORING_SUMMARY_CSV",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Please run Cell 2 first. Missing variables: "
        + ", ".join(missing_variables)
    )

if not os.path.exists(SCORING_INPUT):
    raise FileNotFoundError(
        f"Step 6 GeoPackage was not found:\n"
        f"{SCORING_INPUT}"
    )


# ------------------------------------------------------------------------------
# 3. LOAD STEP 6 TARGET SPEED DATA
# ------------------------------------------------------------------------------
print("=" * 90)
print("STEP 7 — FINAL RISK ASSESSMENT AND SCORING")
print("=" * 90)

input_layer_name = get_first_layer_name(
    SCORING_INPUT
)

gdf_original = gpd.read_file(
    SCORING_INPUT,
    layer=input_layer_name,
    engine="pyogrio",
).copy()

required_columns = [
    PRIMARY_KEY,
    CLUSTER_CODE_COLUMN,
    CLUSTER_NAME_COLUMN,
    P85_SPEED_COLUMN,
    POSTED_SPEED_LIMIT_COLUMN,
    TARGET_SPEED_LIMIT_COLUMN,
    INTERSECTION_DENSITY_COLUMN,
    RANKED_PERCENTILE_COLUMN,
] + list(
    VRU_POI_DISTANCE_COLUMNS.values()
)

missing_columns = [
    column
    for column in required_columns
    if column not in gdf_original.columns
]

if missing_columns:
    raise KeyError(
        "The Step 6 GeoPackage is missing required field(s):\n"
        + "\n".join(
            f"- {column}"
            for column in missing_columns
        )
    )

if gdf_original[PRIMARY_KEY].isna().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' contains missing values."
    )

if gdf_original[PRIMARY_KEY].duplicated().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' must be unique."
    )

print(f"Input GeoPackage: {SCORING_INPUT}")
print(f"Input layer     : {input_layer_name}")
print(f"Road records    : {len(gdf_original):,}")


# ------------------------------------------------------------------------------
# 4. PREPARE VALID CLUSTERED SEGMENTS
# ------------------------------------------------------------------------------
if "analysis_status" in gdf_original.columns:
    valid_mask = (
        gdf_original["analysis_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("valid")
    )
else:
    valid_mask = pd.Series(
        True,
        index=gdf_original.index,
    )

gdf_output = gdf_original.copy()

gdf_output[CLUSTER_CODE_COLUMN] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(clean_text)
)

gdf_output[CLUSTER_NAME_COLUMN] = (
    gdf_output[CLUSTER_NAME_COLUMN]
    .map(clean_text)
)

rows_to_score_mask = (
    valid_mask
    & gdf_output[CLUSTER_CODE_COLUMN].ne("")
)

if not rows_to_score_mask.any():
    raise ValueError(
        "No valid clustered road segments are available "
        "for scoring."
    )

missing_cluster_name_mask = (
    rows_to_score_mask
    & gdf_output[CLUSTER_NAME_COLUMN].eq("")
)

if missing_cluster_name_mask.any():
    missing_cluster_codes = sorted(
        gdf_output.loc[
            missing_cluster_name_mask,
            CLUSTER_CODE_COLUMN,
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "Some clusters do not have a saved cluster name:\n"
        + "\n".join(
            f"- {cluster_code}"
            for cluster_code in missing_cluster_codes
        )
        + "\n\nPlease run Step 5A first."
    )


# ------------------------------------------------------------------------------
# 5. CLEAN NUMERIC FIELDS AND IMPUTE EXISTING SPEED LIMIT
# ------------------------------------------------------------------------------
numeric_input_columns = [
    P85_SPEED_COLUMN,
    POSTED_SPEED_LIMIT_COLUMN,
    TARGET_SPEED_LIMIT_COLUMN,
    RANKED_PERCENTILE_COLUMN,
]

for column in numeric_input_columns:
    gdf_output[column] = pd.to_numeric(
        gdf_output[column],
        errors="coerce",
    )

for poi_distance_column in (
    VRU_POI_DISTANCE_COLUMNS.values()
):
    gdf_output[poi_distance_column] = pd.to_numeric(
        gdf_output[poi_distance_column],
        errors="coerce",
    )

gdf_output[SPEED_LIMIT_IMPUTED_COLUMN] = False

invalid_posted_speed_mask = (
    rows_to_score_mask
    & (
        gdf_output[POSTED_SPEED_LIMIT_COLUMN].isna()
        | (
            gdf_output[POSTED_SPEED_LIMIT_COLUMN]
            <= 0
        )
    )
)

gdf_output.loc[
    invalid_posted_speed_mask,
    POSTED_SPEED_LIMIT_COLUMN,
] = MISSING_POSTED_SPEED_LIMIT_KPH

gdf_output.loc[
    invalid_posted_speed_mask,
    SPEED_LIMIT_IMPUTED_COLUMN,
] = True

print(
    "Missing / zero / negative existing speed limits "
    f"replaced with {MISSING_POSTED_SPEED_LIMIT_KPH} km/h: "
    f"{int(invalid_posted_speed_mask.sum()):,} segment(s)"
)

invalid_p85_mask = (
    rows_to_score_mask
    & (
        gdf_output[P85_SPEED_COLUMN].isna()
        | (
            gdf_output[P85_SPEED_COLUMN]
            < 0
        )
    )
)

if invalid_p85_mask.any():
    invalid_examples = gdf_output.loc[
        invalid_p85_mask,
        [
            PRIMARY_KEY,
            CLUSTER_CODE_COLUMN,
            P85_SPEED_COLUMN,
        ],
    ].head(20)

    raise ValueError(
        f"{invalid_p85_mask.sum():,} valid road segment(s) "
        "have missing or negative p85_speed.\n\n"
        f"Examples:\n{invalid_examples}"
    )

invalid_target_speed_mask = (
    rows_to_score_mask
    & (
        gdf_output[TARGET_SPEED_LIMIT_COLUMN].isna()
        | (
            gdf_output[TARGET_SPEED_LIMIT_COLUMN]
            <= 0
        )
    )
)

if invalid_target_speed_mask.any():
    invalid_examples = gdf_output.loc[
        invalid_target_speed_mask,
        [
            PRIMARY_KEY,
            CLUSTER_CODE_COLUMN,
            TARGET_SPEED_LIMIT_COLUMN,
        ],
    ].head(20)

    raise ValueError(
        f"{invalid_target_speed_mask.sum():,} valid road segment(s) "
        "have missing or non-positive target_speed_limit.\n\n"
        f"Examples:\n{invalid_examples}"
    )

invalid_percentile_mask = (
    rows_to_score_mask
    & (
        gdf_output[RANKED_PERCENTILE_COLUMN].isna()
        | (
            gdf_output[RANKED_PERCENTILE_COLUMN]
            < 0
        )
        | (
            gdf_output[RANKED_PERCENTILE_COLUMN]
            > 100
        )
    )
)

if invalid_percentile_mask.any():
    invalid_examples = gdf_output.loc[
        invalid_percentile_mask,
        [
            PRIMARY_KEY,
            RANKED_PERCENTILE_COLUMN,
        ],
    ].head(20)

    raise ValueError(
        f"{invalid_percentile_mask.sum():,} valid road segment(s) "
        "have invalid ranked_percentile values.\n\n"
        f"Examples:\n{invalid_examples}"
    )

print("\nInput validation completed.")

print(
    "ranked_percentile range: "
    f"{gdf_output.loc[rows_to_score_mask, RANKED_PERCENTILE_COLUMN].min():.2f} "
    "to "
    f"{gdf_output.loc[rows_to_score_mask, RANKED_PERCENTILE_COLUMN].max():.2f}"
)


# ------------------------------------------------------------------------------
# 6. DISPLAY CLUSTERS AND REQUEST BASELINE VRU POINTS
# ------------------------------------------------------------------------------
cluster_counts_df = (
    gdf_output.loc[
        rows_to_score_mask,
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
        ],
    ]
    .groupby(
        [
            CLUSTER_CODE_COLUMN,
            CLUSTER_NAME_COLUMN,
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="segment_count"
    )
)

cluster_order_map = {
    cluster_code: position
    for position, cluster_code in enumerate(
        sort_cluster_codes(
            cluster_counts_df[
                CLUSTER_CODE_COLUMN
            ].tolist()
        )
    )
}

cluster_counts_df["_sort_order"] = (
    cluster_counts_df[CLUSTER_CODE_COLUMN]
    .map(cluster_order_map)
)

cluster_counts_df = (
    cluster_counts_df
    .sort_values("_sort_order")
    .drop(columns="_sort_order")
    .reset_index(drop=True)
)

print("\nAvailable clusters:")
display(cluster_counts_df)

cluster_vru_config_records = []

for _, cluster_row in cluster_counts_df.iterrows():
    cluster_code = cluster_row[
        CLUSTER_CODE_COLUMN
    ]

    cluster_name = cluster_row[
        CLUSTER_NAME_COLUMN
    ]

    segment_count = int(
        cluster_row["segment_count"]
    )

    baseline_vru_point = request_vru_baseline(
        cluster_code=cluster_code,
        cluster_name=cluster_name,
        segment_count=segment_count,
    )

    cluster_vru_config_records.append(
        {
            CLUSTER_CODE_COLUMN: cluster_code,
            CLUSTER_NAME_COLUMN: cluster_name,
            "segment_count": segment_count,
            VRU_BASELINE_POINT_COLUMN: baseline_vru_point,
        }
    )

cluster_vru_config_df = pd.DataFrame(
    cluster_vru_config_records
)

print("\n" + "=" * 90)
print("USER-DEFINED BASELINE VRU CONFIGURATION")
print("=" * 90)

display(cluster_vru_config_df)


# ------------------------------------------------------------------------------
# 7. ASSESS EXISTING SPEED LIMIT ABOVE TARGET SPEED
#
# excess_above_speed_target = speed_limit - target_speed_limit
#
# <= 0  : Align            = 0
# <= 10 : Slight Excess    = 10
# <= 20 : Moderate Excess  = 20
# <= 30 : High Excess      = 30
# > 30  : Very High Excess = 40
# ------------------------------------------------------------------------------
speed_target_excess = (
    gdf_output[POSTED_SPEED_LIMIT_COLUMN]
    - gdf_output[TARGET_SPEED_LIMIT_COLUMN]
)

(
    speed_target_misalignment_labels,
    speed_target_misalignment_points,
) = assess_speed_excess(
    excess_series=speed_target_excess,
    point_scheme={
        "align": 0,
        "slight": 10,
        "moderate": 20,
        "high": 30,
        "very_high": 40,
    },
)

gdf_output[EXCESS_ABOVE_SPEED_TARGET_COLUMN] = np.nan
gdf_output[SPEED_TARGET_MISALIGNMENT_COLUMN] = None
gdf_output[SPEED_TARGET_MISALIGNMENT_POINT_COLUMN] = np.nan

gdf_output.loc[
    rows_to_score_mask,
    EXCESS_ABOVE_SPEED_TARGET_COLUMN,
] = speed_target_excess.loc[
    rows_to_score_mask
].round(2)

gdf_output.loc[
    rows_to_score_mask,
    SPEED_TARGET_MISALIGNMENT_COLUMN,
] = speed_target_misalignment_labels.loc[
    rows_to_score_mask
]

gdf_output.loc[
    rows_to_score_mask,
    SPEED_TARGET_MISALIGNMENT_POINT_COLUMN,
] = speed_target_misalignment_points.loc[
    rows_to_score_mask
]


# ------------------------------------------------------------------------------
# 8. ASSESS VRU EXPOSURE
#
# Baseline:
# - User input by cluster, from 0 to 20.
#
# Proximity adjustment:
# - At least one non-industrial POI group within 100 m -> +5
# - All four non-industrial POI groups within 100 m -> +10 total
#
# Final:
# - vru_point = baseline + proximity adjustment
# - vru_point capped at 25
# ------------------------------------------------------------------------------
baseline_vru_lookup = (
    cluster_vru_config_df.set_index(
        CLUSTER_CODE_COLUMN
    )[VRU_BASELINE_POINT_COLUMN]
    .to_dict()
)

gdf_output[VRU_BASELINE_POINT_COLUMN] = (
    gdf_output[CLUSTER_CODE_COLUMN]
    .map(baseline_vru_lookup)
)

missing_baseline_mask = (
    rows_to_score_mask
    & gdf_output[VRU_BASELINE_POINT_COLUMN].isna()
)

if missing_baseline_mask.any():
    raise RuntimeError(
        "Some scored road segments did not receive "
        "a baseline VRU point."
    )

poi_near_boolean_columns = []

for poi_group, poi_distance_column in (
    VRU_POI_DISTANCE_COLUMNS.items()
):
    poi_near_column = (
        f"_is_{poi_group}_poi_lt_100m"
    )

    gdf_output[poi_near_column] = (
        gdf_output[poi_distance_column].notna()
        & (
            gdf_output[poi_distance_column]
            >= 0
        )
        & (
            gdf_output[poi_distance_column]
            < VRU_NEAR_DISTANCE_THRESHOLD_M
        )
    )

    poi_near_boolean_columns.append(
        poi_near_column
    )

gdf_output[VRU_NEAR_POI_COUNT_COLUMN] = (
    gdf_output[poi_near_boolean_columns]
    .sum(axis=1)
)

gdf_output[VRU_POI_PROXIMITY_POINT_COLUMN] = 0

any_poi_near_mask = (
    rows_to_score_mask
    & (
        gdf_output[VRU_NEAR_POI_COUNT_COLUMN]
        >= 1
    )
)

all_poi_groups_near_mask = (
    rows_to_score_mask
    & (
        gdf_output[VRU_NEAR_POI_COUNT_COLUMN]
        == len(VRU_POI_DISTANCE_COLUMNS)
    )
)

gdf_output.loc[
    any_poi_near_mask,
    VRU_POI_PROXIMITY_POINT_COLUMN,
] = 5

gdf_output.loc[
    all_poi_groups_near_mask,
    VRU_POI_PROXIMITY_POINT_COLUMN,
] = 10

gdf_output[VRU_POINT_COLUMN] = np.nan

gdf_output.loc[
    rows_to_score_mask,
    VRU_POINT_COLUMN,
] = np.minimum(
    VRU_POINT_MAXIMUM,
    gdf_output.loc[
        rows_to_score_mask,
        VRU_BASELINE_POINT_COLUMN,
    ]
    + gdf_output.loc[
        rows_to_score_mask,
        VRU_POI_PROXIMITY_POINT_COLUMN,
    ],
)


# ------------------------------------------------------------------------------
# 9. ASSESS P85 SPEED ABOVE SAFE SYSTEM TARGET SPEED
#
# excess_above_safe_system_target = p85_speed - target_speed_limit
#
# <= 0  : Align            = 0
# <= 10 : Slight Excess    = 4
# <= 20 : Moderate Excess  = 8
# <= 30 : High Excess      = 12
# > 30  : Very High Excess = 15
# ------------------------------------------------------------------------------
safe_system_speed_excess = (
    gdf_output[P85_SPEED_COLUMN]
    - gdf_output[TARGET_SPEED_LIMIT_COLUMN]
)

(
    safe_system_labels,
    safe_system_points,
) = assess_speed_excess(
    excess_series=safe_system_speed_excess,
    point_scheme={
        "align": 0,
        "slight": 4,
        "moderate": 8,
        "high": 12,
        "very_high": 15,
    },
)

gdf_output[EXCESS_ABOVE_TARGET_COLUMN] = np.nan
gdf_output[SAFE_SYSTEM_MISALIGNMENT_COLUMN] = None
gdf_output[SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN] = np.nan
gdf_output[SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN] = None

gdf_output.loc[
    rows_to_score_mask,
    EXCESS_ABOVE_TARGET_COLUMN,
] = safe_system_speed_excess.loc[
    rows_to_score_mask
].round(2)

gdf_output.loc[
    rows_to_score_mask,
    SAFE_SYSTEM_MISALIGNMENT_COLUMN,
] = safe_system_labels.loc[
    rows_to_score_mask
]

gdf_output.loc[
    rows_to_score_mask,
    SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN,
] = safe_system_points.loc[
    rows_to_score_mask
]

gdf_output.loc[
    rows_to_score_mask,
    SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN,
] = gdf_output.loc[
    rows_to_score_mask,
    [
        POSTED_SPEED_LIMIT_COLUMN,
        TARGET_SPEED_LIMIT_COLUMN,
        P85_SPEED_COLUMN,
    ],
].apply(
    lambda row: get_safe_system_insight(
        speed_limit=row[
            POSTED_SPEED_LIMIT_COLUMN
        ],
        target_speed_limit=row[
            TARGET_SPEED_LIMIT_COLUMN
        ],
        p85_speed=row[P85_SPEED_COLUMN],
    ),
    axis=1,
)


# ------------------------------------------------------------------------------
# 10. ASSESS INTERSECTION DENSITY POINT
#
# low              -> 0
# moderate / medium -> 5
# high             -> 10
# ------------------------------------------------------------------------------
gdf_output["_intersection_density_normalised"] = (
    gdf_output[INTERSECTION_DENSITY_COLUMN]
    .map(normalise_intersection_density)
)

invalid_density_mask = (
    rows_to_score_mask
    & ~gdf_output[
        "_intersection_density_normalised"
    ].isin(
        [
            "low",
            "medium",
            "high",
        ]
    )
)

if invalid_density_mask.any():
    invalid_values = sorted(
        gdf_output.loc[
            invalid_density_mask,
            INTERSECTION_DENSITY_COLUMN,
        ]
        .map(clean_text)
        .unique()
        .tolist()
    )

    raise ValueError(
        "Unexpected intersection_density value(s):\n"
        + "\n".join(
            f"- {value}"
            for value in invalid_values
        )
    )

intersection_density_point_map = {
    "low": 0,
    "medium": 5,
    "high": 10,
}

gdf_output[INTERSECTION_DENSITY_POINT_COLUMN] = np.nan

gdf_output.loc[
    rows_to_score_mask,
    INTERSECTION_DENSITY_POINT_COLUMN,
] = gdf_output.loc[
    rows_to_score_mask,
    "_intersection_density_normalised",
].map(
    intersection_density_point_map
)


# ------------------------------------------------------------------------------
# 11. ASSESS TRAFFIC EXPOSURE
#
# ranked_percentile <= 25  -> low            = 0
# ranked_percentile <= 55  -> low to middle  = 3
# ranked_percentile <= 75  -> middle to high = 7
# ranked_percentile <= 100 -> high           = 10
# ------------------------------------------------------------------------------
ranked_percentile = gdf_output[
    RANKED_PERCENTILE_COLUMN
]

traffic_exposure_labels = pd.Series(
    None,
    index=gdf_output.index,
    dtype="object",
)

traffic_exposure_points = pd.Series(
    np.nan,
    index=gdf_output.index,
    dtype="float64",
)

low_traffic_mask = ranked_percentile <= 25

low_middle_traffic_mask = (
    (ranked_percentile > 25)
    & (ranked_percentile <= 55)
)

middle_high_traffic_mask = (
    (ranked_percentile > 55)
    & (ranked_percentile <= 75)
)

high_traffic_mask = ranked_percentile > 75

traffic_exposure_labels.loc[
    low_traffic_mask
] = "low"

traffic_exposure_labels.loc[
    low_middle_traffic_mask
] = "low to middle"

traffic_exposure_labels.loc[
    middle_high_traffic_mask
] = "middle to high"

traffic_exposure_labels.loc[
    high_traffic_mask
] = "high"

traffic_exposure_points.loc[
    low_traffic_mask
] = 0

traffic_exposure_points.loc[
    low_middle_traffic_mask
] = 3

traffic_exposure_points.loc[
    middle_high_traffic_mask
] = 7

traffic_exposure_points.loc[
    high_traffic_mask
] = 10

gdf_output[TRAFFIC_EXPOSURE_COLUMN] = None
gdf_output[TRAFFIC_EXPOSURE_POINT_COLUMN] = np.nan

gdf_output.loc[
    rows_to_score_mask,
    TRAFFIC_EXPOSURE_COLUMN,
] = traffic_exposure_labels.loc[
    rows_to_score_mask
]

gdf_output.loc[
    rows_to_score_mask,
    TRAFFIC_EXPOSURE_POINT_COLUMN,
] = traffic_exposure_points.loc[
    rows_to_score_mask
]


# ------------------------------------------------------------------------------
# 12. BUILD FINAL TOTAL RISK SCORE
#
# total_risk_point =
# speed_limit_target_misalignment_point
# + vru_point
# + safe_system_misalignment_point
# + intersection_density_point
# + traffic_exposure_point
#
# Maximum score = 40 + 25 + 15 + 10 + 10 = 100
# ------------------------------------------------------------------------------
integer_score_columns = [
    SPEED_TARGET_MISALIGNMENT_POINT_COLUMN,
    VRU_BASELINE_POINT_COLUMN,
    VRU_NEAR_POI_COUNT_COLUMN,
    VRU_POI_PROXIMITY_POINT_COLUMN,
    VRU_POINT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN,
    INTERSECTION_DENSITY_POINT_COLUMN,
    TRAFFIC_EXPOSURE_POINT_COLUMN,
]

for score_column in integer_score_columns:
    gdf_output[score_column] = (
        pd.to_numeric(
            gdf_output[score_column],
            errors="coerce",
        )
        .round()
        .astype("Int64")
    )

total_risk_component_columns = [
    SPEED_TARGET_MISALIGNMENT_POINT_COLUMN,
    VRU_POINT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN,
    INTERSECTION_DENSITY_POINT_COLUMN,
    TRAFFIC_EXPOSURE_POINT_COLUMN,
]

gdf_output[TOTAL_RISK_POINT_COLUMN] = (
    gdf_output[
        total_risk_component_columns
    ]
    .sum(
        axis=1,
        min_count=len(total_risk_component_columns),
    )
    .round()
    .astype("Int64")
)

required_scoring_columns = [
    SPEED_LIMIT_IMPUTED_COLUMN,
    EXCESS_ABOVE_SPEED_TARGET_COLUMN,
    SPEED_TARGET_MISALIGNMENT_COLUMN,
    SPEED_TARGET_MISALIGNMENT_POINT_COLUMN,
    VRU_BASELINE_POINT_COLUMN,
    VRU_NEAR_POI_COUNT_COLUMN,
    VRU_POI_PROXIMITY_POINT_COLUMN,
    VRU_POINT_COLUMN,
    EXCESS_ABOVE_TARGET_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN,
    INTERSECTION_DENSITY_POINT_COLUMN,
    TRAFFIC_EXPOSURE_COLUMN,
    TRAFFIC_EXPOSURE_POINT_COLUMN,
    TOTAL_RISK_POINT_COLUMN,
]

missing_scoring_mask = (
    gdf_output.loc[
        rows_to_score_mask,
        required_scoring_columns,
    ]
    .isna()
    .any(axis=1)
)

if missing_scoring_mask.any():
    failed_examples = gdf_output.loc[
        rows_to_score_mask,
        [
            PRIMARY_KEY,
            CLUSTER_CODE_COLUMN,
        ]
        + required_scoring_columns,
    ].loc[
        missing_scoring_mask
    ].head(20)

    raise RuntimeError(
        f"{missing_scoring_mask.sum():,} valid road segment(s) "
        "did not receive complete scoring results.\n\n"
        f"Examples:\n{failed_examples}"
    )

if (
    gdf_output.loc[
        rows_to_score_mask,
        VRU_POINT_COLUMN,
    ]
    > VRU_POINT_MAXIMUM
).any():
    raise RuntimeError(
        f"vru_point exceeds the maximum allowed value "
        f"of {VRU_POINT_MAXIMUM}."
    )

if (
    gdf_output.loc[
        rows_to_score_mask,
        TOTAL_RISK_POINT_COLUMN,
    ]
    > 100
).any():
    raise RuntimeError(
        "total_risk_point exceeds the maximum "
        "allowed score of 100."
    )


# ------------------------------------------------------------------------------
# 13. SAVE CLUSTER VRU CONFIGURATION
# ------------------------------------------------------------------------------
cluster_vru_config_df.to_csv(
    SCORING_CLUSTER_VRU_CONFIG_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved cluster VRU configuration:")
print(SCORING_CLUSTER_VRU_CONFIG_CSV)


# ------------------------------------------------------------------------------
# 14. BUILD AND SAVE SUMMARY
# ------------------------------------------------------------------------------
summary_group_columns = [
    CLUSTER_CODE_COLUMN,
    CLUSTER_NAME_COLUMN,
    SPEED_TARGET_MISALIGNMENT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_COLUMN,
    INTERSECTION_DENSITY_COLUMN,
    TRAFFIC_EXPOSURE_COLUMN,
]

scoring_summary_df = (
    gdf_output.loc[
        rows_to_score_mask,
        summary_group_columns + [
            TOTAL_RISK_POINT_COLUMN,
        ],
    ]
    .groupby(
        summary_group_columns,
        dropna=False,
    )
    .agg(
        segment_count=(
            TOTAL_RISK_POINT_COLUMN,
            "size",
        ),
        median_total_risk_point=(
            TOTAL_RISK_POINT_COLUMN,
            "median",
        ),
        mean_total_risk_point=(
            TOTAL_RISK_POINT_COLUMN,
            "mean",
        ),
    )
    .reset_index()
)

scoring_summary_df[
    "median_total_risk_point"
] = (
    scoring_summary_df[
        "median_total_risk_point"
    ]
    .round(2)
)

scoring_summary_df[
    "mean_total_risk_point"
] = (
    scoring_summary_df[
        "mean_total_risk_point"
    ]
    .round(2)
)

scoring_summary_df.to_csv(
    SCORING_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved scoring summary:")
print(SCORING_SUMMARY_CSV)


# ------------------------------------------------------------------------------
# 15. SAVE FINAL DASHBOARD-READY GEOPACKAGE
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("SAVING FINAL DASHBOARD-READY GEOPACKAGE")
print("=" * 90)

temporary_columns = [
    "_intersection_density_normalised",
] + [
    f"_is_{poi_group}_poi_lt_100m"
    for poi_group in VRU_POI_DISTANCE_COLUMNS
]

gdf_output = gdf_output.drop(
    columns=temporary_columns,
    errors="ignore",
)

for extension in [
    "",
    "-wal",
    "-shm",
]:
    output_file = SCORING_RESULT + extension

    if os.path.exists(output_file):
        os.remove(output_file)

gdf_output.to_file(
    SCORING_RESULT,
    layer=SCORING_RESULT_LAYER,
    driver="GPKG",
    engine="pyogrio",
    index=False,
)

print("Final scored GeoPackage saved:")
print(SCORING_RESULT)


# ------------------------------------------------------------------------------
# 16. FINAL REVIEW
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("STEP 7 COMPLETED — FINAL ASSESSMENT AND SCORING")
print("=" * 90)

print("\nCluster-level VRU baseline configuration:")
display(cluster_vru_config_df)

print("\nScoring summary:")
display(scoring_summary_df.head(30))

print("\nFinal output preview:")

preview_columns = [
    column
    for column in [
        PRIMARY_KEY,
        COUNTRY_COLUMN,
        CLUSTER_CODE_COLUMN,
        CLUSTER_NAME_COLUMN,
        P85_SPEED_COLUMN,
        POSTED_SPEED_LIMIT_COLUMN,
        SPEED_LIMIT_IMPUTED_COLUMN,
        TARGET_SPEED_LIMIT_COLUMN,
        EXCESS_ABOVE_SPEED_TARGET_COLUMN,
        SPEED_TARGET_MISALIGNMENT_COLUMN,
        SPEED_TARGET_MISALIGNMENT_POINT_COLUMN,
        VRU_BASELINE_POINT_COLUMN,
        VRU_NEAR_POI_COUNT_COLUMN,
        VRU_POI_PROXIMITY_POINT_COLUMN,
        VRU_POINT_COLUMN,
        EXCESS_ABOVE_TARGET_COLUMN,
        SAFE_SYSTEM_MISALIGNMENT_COLUMN,
        SAFE_SYSTEM_MISALIGNMENT_POINT_COLUMN,
        SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN,
        INTERSECTION_DENSITY_COLUMN,
        INTERSECTION_DENSITY_POINT_COLUMN,
        RANKED_PERCENTILE_COLUMN,
        TRAFFIC_EXPOSURE_COLUMN,
        TRAFFIC_EXPOSURE_POINT_COLUMN,
        TOTAL_RISK_POINT_COLUMN,
    ]
    if column in gdf_output.columns
]

display(
    gdf_output.loc[
        rows_to_score_mask,
        preview_columns,
    ]
    .sort_values(
        [
            TOTAL_RISK_POINT_COLUMN,
            CLUSTER_CODE_COLUMN,
            PRIMARY_KEY,
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .head(30)
)

del gdf_output
gc.collect()

## Interactive Maps Creation

In [ ]:
# ==============================================================================
# STEP 8 — LIGHTWEIGHT INTERACTIVE TOTAL RISK MAPS
#
# FILE-SIZE REDUCTION:
# - Simplifies only map-display geometry in a local UTM CRS.
# - Keeps the original GeoPackage unchanged.
# - Keeps all popup fields, percentile-band filters, colours, and interactivity.
# - Uses compact GeoJSON property names internally to reduce HTML size.
#
# OUTPUT:
# /content/6. Final Result/
# - thailand_total_risk_map.html
# - maharashtra_total_risk_map.html
#
# CLICK POPUP FIELDS:
# - analysis_segment_id
# - road_name
# - cluster_name
# - shape_length
# - ranked_percentile
# - percentile_band
# - speed_limit
# - target_speed_limit
# - speed_limit_vs_target_speed_limit
# - p85_vs_target_speed_limit
# - intersection_density
# - traffic_exposure
# - safe_system_misalignment
# - total_risk_point
# ==============================================================================


# ------------------------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------------------------
import os
import gc
import html
import json
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import folium

from IPython.display import display


# ------------------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------------------
PRIMARY_KEY = "analysis_segment_id"

COUNTRY_COLUMN = "dataset"
ROAD_NAME_COLUMN = "road_name"
SHAPE_LENGTH_COLUMN = "shape_length"

CLUSTER_NAME_COLUMN = (
    "poi_ghsl_context_cluster_name"
)

RANKED_PERCENTILE_COLUMN = "ranked_percentile"
PERCENTILE_BAND_COLUMN = "percentile_band"

SPEED_LIMIT_COLUMN = "speed_limit"
TARGET_SPEED_LIMIT_COLUMN = "target_speed_limit"

# Existing speed limit compared with target speed limit.
SPEED_TARGET_MISALIGNMENT_COLUMN = (
    "speed_limit_target_misalignment"
)

# P85 speed compared with target speed limit.
SAFE_SYSTEM_MISALIGNMENT_COLUMN = (
    "safe_system_misalignment"
)

SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN = (
    "safe_system_misalignment_insight"
)

INTERSECTION_DENSITY_COLUMN = (
    "intersection_density"
)

TRAFFIC_EXPOSURE_COLUMN = (
    "traffic_exposure"
)

TOTAL_RISK_POINT_COLUMN = (
    "total_risk_point"
)

THAILAND_LABEL = "Thailand"
MAHARASHTRA_LABEL = "Maharashtra"

MAP_INITIAL_ZOOM = 8
DISPLAY_MAPS = True

TOTAL_RISK_MINIMUM = 0
TOTAL_RISK_MIDPOINT = 50
TOTAL_RISK_MAXIMUM = 100

TOTAL_RISK_GREEN = "#1a9850"
TOTAL_RISK_YELLOW = "#fee08b"
TOTAL_RISK_RED = "#d73027"

# Increase to 30.0 or 40.0 only if HTML remains too large.
GEOMETRY_SIMPLIFICATION_TOLERANCE_M = 20.0

GITHUB_BROWSER_UPLOAD_LIMIT_MIB = 25.0

SCORING_MAP_INPUT = (
    "/content/6. Final Result/"
    "ADB_Innovation_target_speed_limit_with_scoring.gpkg"
)

SCORING_MAP_OUTPUT_FOLDER = (
    "/content/6. Final Result"
)

THAILAND_TOTAL_RISK_MAP_HTML = os.path.join(
    SCORING_MAP_OUTPUT_FOLDER,
    "thailand_total_risk_map.html",
)

MAHARASHTRA_TOTAL_RISK_MAP_HTML = os.path.join(
    SCORING_MAP_OUTPUT_FOLDER,
    "maharashtra_total_risk_map.html",
)

os.makedirs(
    SCORING_MAP_OUTPUT_FOLDER,
    exist_ok=True,
)


# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------------------
def get_first_layer_name(gpkg_path):
    """
    Return the first available layer inside a GeoPackage.
    """
    layers = pyogrio.list_layers(gpkg_path)

    if len(layers) == 0:
        raise ValueError(
            f"No layer found inside:\n{gpkg_path}"
        )

    return str(layers[0][0])


def clean_text(value):
    """
    Convert null-like values into an empty string.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return ""

    return text


def format_popup_text(series):
    """
    Convert values into clean, HTML-safe popup text.
    """
    return (
        series
        .map(clean_text)
        .map(html.escape)
        .astype("string")
    )


def format_popup_number(
    series,
    decimal_places=1,
    suffix="",
):
    """
    Format numeric values for popup display.
    """
    numeric_series = pd.to_numeric(
        series,
        errors="coerce",
    )

    return numeric_series.apply(
        lambda value: (
            f"{value:,.{decimal_places}f}{suffix}"
            if pd.notna(value)
            else ""
        )
    )


def hex_to_rgb(hex_color):
    """
    Convert #RRGGBB into an RGB tuple.
    """
    hex_color = hex_color.lstrip("#")

    return tuple(
        int(
            hex_color[position:position + 2],
            16,
        )
        for position in (
            0,
            2,
            4,
        )
    )


def rgb_to_hex(rgb_value):
    """
    Convert RGB tuple into #RRGGBB.
    """
    red, green, blue = rgb_value

    return (
        f"#{int(red):02x}"
        f"{int(green):02x}"
        f"{int(blue):02x}"
    )


def interpolate_color(
    start_color,
    end_color,
    fraction,
):
    """
    Create a linear colour interpolation.
    """
    fraction = min(
        max(float(fraction), 0.0),
        1.0,
    )

    start_rgb = hex_to_rgb(start_color)
    end_rgb = hex_to_rgb(end_color)

    interpolated_rgb = tuple(
        round(
            start_value
            + (
                end_value
                - start_value
            )
            * fraction
        )
        for start_value, end_value in zip(
            start_rgb,
            end_rgb,
        )
    )

    return rgb_to_hex(interpolated_rgb)


def get_total_risk_color(total_risk_point):
    """
    0   -> green
    50  -> yellow
    100 -> red
    """
    try:
        value = float(total_risk_point)

    except (
        TypeError,
        ValueError,
    ):
        return "#bdbdbd"

    if np.isnan(value):
        return "#bdbdbd"

    value = min(
        max(value, TOTAL_RISK_MINIMUM),
        TOTAL_RISK_MAXIMUM,
    )

    if value <= TOTAL_RISK_MIDPOINT:
        fraction = (
            value
            / TOTAL_RISK_MIDPOINT
        )

        return interpolate_color(
            start_color=TOTAL_RISK_GREEN,
            end_color=TOTAL_RISK_YELLOW,
            fraction=fraction,
        )

    fraction = (
        value
        - TOTAL_RISK_MIDPOINT
    ) / (
        TOTAL_RISK_MAXIMUM
        - TOTAL_RISK_MIDPOINT
    )

    return interpolate_color(
        start_color=TOTAL_RISK_YELLOW,
        end_color=TOTAL_RISK_RED,
        fraction=fraction,
    )


def percentile_band_sort_key(value):
    """
    Sort values such as:
    0-5%
    5-10%
    10-15%
    """
    text = clean_text(value)

    if text == "Unspecified":
        return (
            float("inf"),
            float("inf"),
            text.casefold(),
        )

    numbers = re.findall(
        r"[-+]?\d*\.?\d+",
        text,
    )

    if len(numbers) == 0:
        return (
            float("inf"),
            float("inf"),
            text.casefold(),
        )

    first_value = float(numbers[0])

    second_value = (
        float(numbers[1])
        if len(numbers) >= 2
        else first_value
    )

    return (
        first_value,
        second_value,
        text.casefold(),
    )


def count_geometry_vertices(geometry):
    """
    Count coordinate vertices for line geometries.
    """
    if (
        geometry is None
        or geometry.is_empty
    ):
        return 0

    geometry_type = geometry.geom_type

    if geometry_type == "LineString":
        return len(geometry.coords)

    if geometry_type == "MultiLineString":
        return sum(
            len(part.coords)
            for part in geometry.geoms
        )

    if geometry_type == "GeometryCollection":
        return sum(
            count_geometry_vertices(part)
            for part in geometry.geoms
        )

    return 0


def count_total_vertices(gdf):
    """
    Count all road-line vertices in a GeoDataFrame.
    """
    return int(
        gdf.geometry.map(
            count_geometry_vertices
        ).sum()
    )


def simplify_map_geometries(
    source_gdf,
    tolerance_m,
):
    """
    Simplify only the map geometry.

    The source GeoPackage is never changed.
    Geometry is projected to an estimated UTM CRS first,
    so the tolerance is measured in metres.
    """
    map_gdf = source_gdf.copy()

    if map_gdf.crs is None:
        raise ValueError(
            "Map GeoDataFrame has no CRS."
        )

    if map_gdf.crs.to_epsg() != 4326:
        map_gdf = map_gdf.to_crs(
            epsg=4326
        )

    original_vertex_count = count_total_vertices(
        map_gdf
    )

    simplification_crs = "Not applied"

    if tolerance_m <= 0:
        simplified_gdf = map_gdf.copy()

    else:
        try:
            local_utm_crs = (
                map_gdf.estimate_utm_crs()
            )

        except Exception:
            local_utm_crs = None

        if local_utm_crs is not None:
            projected_gdf = map_gdf.to_crs(
                local_utm_crs
            )

            projected_gdf.geometry = (
                projected_gdf.geometry.simplify(
                    tolerance=float(tolerance_m),
                    preserve_topology=False,
                )
            )

            simplified_gdf = projected_gdf.to_crs(
                epsg=4326
            )

            simplification_crs = str(
                local_utm_crs
            )

        else:
            degree_tolerance = (
                float(tolerance_m)
                / 111_320
            )

            simplified_gdf = map_gdf.copy()

            simplified_gdf.geometry = (
                simplified_gdf.geometry.simplify(
                    tolerance=degree_tolerance,
                    preserve_topology=False,
                )
            )

            simplification_crs = (
                "WGS84 fallback approximation"
            )

    simplified_gdf = simplified_gdf.loc[
        simplified_gdf.geometry.notna()
        & ~simplified_gdf.geometry.is_empty
    ].copy()

    simplified_vertex_count = count_total_vertices(
        simplified_gdf
    )

    return (
        simplified_gdf,
        original_vertex_count,
        simplified_vertex_count,
        simplification_crs,
    )


def to_compact_geojson(gdf):
    """
    Convert GeoDataFrame to compact GeoJSON.
    """
    try:
        geojson_text = gdf.to_json(
            drop_id=True
        )

    except TypeError:
        geojson_text = gdf.to_json()

    return json.dumps(
        json.loads(geojson_text),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def get_road_style(feature):
    """
    Style road colour from compact risk field: r
    """
    total_risk_point = feature["properties"].get(
        "r",
        None,
    )

    return {
        "color": get_total_risk_color(
            total_risk_point
        ),
        "weight": 2.2,
        "opacity": 0.90,
    }


def create_total_risk_legend_html():
    """
    Create total-risk gradient legend.
    """
    return f"""
    <div style="
        position: fixed;
        bottom: 25px;
        right: 25px;
        z-index: 9999;
        width: 245px;
        background-color: rgba(255, 255, 255, 0.96);
        border: 1px solid #777;
        border-radius: 4px;
        padding: 10px;
        font-family: Arial, sans-serif;
        font-size: 12px;
        line-height: 15px;
        box-shadow: 0 1px 5px rgba(0, 0, 0, 0.4);
    ">
        <div style="
            font-weight: bold;
            margin-bottom: 8px;
            font-size: 13px;
        ">
            Total Risk Point
        </div>

        <div style="
            height: 15px;
            border: 1px solid #666;
            background: linear-gradient(
                to right,
                {TOTAL_RISK_GREEN} 0%,
                {TOTAL_RISK_YELLOW} 50%,
                {TOTAL_RISK_RED} 100%
            );
        "></div>

        <div style="
            display: flex;
            justify-content: space-between;
            margin-top: 4px;
        ">
            <span>0</span>
            <span>50</span>
            <span>100</span>
        </div>

        <div style="
            margin-top: 7px;
            color: #444;
        ">
            Green = lower risk<br>
            Yellow = medium risk<br>
            Red = higher risk
        </div>
    </div>
    """


def create_popup_css_html():
    """
    Improve popup readability.
    """
    return """
    <style>
        .leaflet-popup-content {
            width: 670px !important;
            max-width: 670px !important;
            max-height: 540px !important;
            overflow-y: auto !important;
        }

        .leaflet-popup-content table {
            width: 100% !important;
            table-layout: fixed !important;
            border-collapse: collapse !important;
        }

        .leaflet-popup-content th {
            width: 250px !important;
            vertical-align: top !important;
            text-align: left !important;
            white-space: normal !important;
            overflow-wrap: anywhere !important;
        }

        .leaflet-popup-content td {
            vertical-align: top !important;
            white-space: normal !important;
            overflow-wrap: anywhere !important;
        }
    </style>
    """


def get_file_size_mib(file_path):
    """
    Return file size in MiB.
    """
    return (
        os.path.getsize(file_path)
        / (1024 ** 2)
    )


def create_total_risk_map(
    source_gdf,
    country_label,
    output_html_path,
):
    """
    Create one country-level interactive total-risk map.
    """
    if source_gdf.empty:
        raise ValueError(
            f"No road segments found for {country_label}."
        )

    map_gdf = source_gdf.copy()

    if map_gdf.crs is None:
        raise ValueError(
            f"{country_label} GeoDataFrame has no CRS."
        )

    map_gdf = map_gdf.loc[
        map_gdf.geometry.notna()
        & ~map_gdf.geometry.is_empty
    ].copy()

    if map_gdf.empty:
        raise ValueError(
            f"No valid road geometries found for "
            f"{country_label}."
        )

    # --------------------------------------------------------------------------
    # Validate total-risk score.
    # --------------------------------------------------------------------------
    map_gdf["_risk"] = pd.to_numeric(
        map_gdf[TOTAL_RISK_POINT_COLUMN],
        errors="coerce",
    )

    invalid_risk_mask = (
        map_gdf["_risk"].isna()
        | (
            map_gdf["_risk"]
            < TOTAL_RISK_MINIMUM
        )
        | (
            map_gdf["_risk"]
            > TOTAL_RISK_MAXIMUM
        )
    )

    excluded_risk_count = int(
        invalid_risk_mask.sum()
    )

    map_gdf = map_gdf.loc[
        ~invalid_risk_mask
    ].copy()

    if map_gdf.empty:
        raise ValueError(
            f"No valid total_risk_point values found for "
            f"{country_label}."
        )

    # --------------------------------------------------------------------------
    # Simplify road geometry only for the interactive map.
    # --------------------------------------------------------------------------
    (
        map_gdf,
        original_vertex_count,
        simplified_vertex_count,
        simplification_crs,
    ) = simplify_map_geometries(
        source_gdf=map_gdf,
        tolerance_m=GEOMETRY_SIMPLIFICATION_TOLERANCE_M,
    )

    # Render higher-risk roads later.
    map_gdf = (
        map_gdf
        .sort_values("_risk")
        .copy()
    )

    # --------------------------------------------------------------------------
    # Prepare percentile-band filtering.
    # --------------------------------------------------------------------------
    map_gdf["_band"] = (
        map_gdf[PERCENTILE_BAND_COLUMN]
        .map(clean_text)
        .replace(
            "",
            "Unspecified",
        )
    )

    available_percentile_bands = sorted(
        map_gdf["_band"]
        .drop_duplicates()
        .tolist(),
        key=percentile_band_sort_key,
    )

    # --------------------------------------------------------------------------
    # Compact properties for smaller HTML output.
    # --------------------------------------------------------------------------
    map_gdf["i"] = format_popup_text(
        map_gdf[PRIMARY_KEY]
    )

    map_gdf["n"] = format_popup_text(
        map_gdf[ROAD_NAME_COLUMN]
    )

    map_gdf["c"] = format_popup_text(
        map_gdf[CLUSTER_NAME_COLUMN]
    )

    map_gdf["l"] = format_popup_number(
        map_gdf[SHAPE_LENGTH_COLUMN],
        decimal_places=1,
        suffix=" m",
    )

    map_gdf["p"] = format_popup_number(
        map_gdf[RANKED_PERCENTILE_COLUMN],
        decimal_places=2,
    )

    map_gdf["b"] = format_popup_text(
        map_gdf["_band"]
    )

    map_gdf["s"] = format_popup_number(
        map_gdf[SPEED_LIMIT_COLUMN],
        decimal_places=0,
        suffix=" km/h",
    )

    map_gdf["t"] = format_popup_number(
        map_gdf[TARGET_SPEED_LIMIT_COLUMN],
        decimal_places=0,
        suffix=" km/h",
    )

    # Existing speed_limit - target_speed_limit.
    map_gdf["v"] = format_popup_text(
        map_gdf[
            SPEED_TARGET_MISALIGNMENT_COLUMN
        ]
    )

    # P85-based Safe System interpretation.
    map_gdf["u"] = format_popup_text(
        map_gdf[
            SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN
        ]
    )

    map_gdf["d"] = format_popup_text(
        map_gdf[
            INTERSECTION_DENSITY_COLUMN
        ]
    )

    map_gdf["e"] = format_popup_text(
        map_gdf[
            TRAFFIC_EXPOSURE_COLUMN
        ]
    )

    # P85 speed - target_speed_limit category.
    map_gdf["m"] = format_popup_text(
        map_gdf[
            SAFE_SYSTEM_MISALIGNMENT_COLUMN
        ]
    )

    # Used for colour styling and popup total risk score.
    map_gdf["r"] = (
        pd.to_numeric(
            map_gdf["_risk"],
            errors="coerce",
        )
        .round()
        .astype(int)
    )

    geometry_column = map_gdf.geometry.name

    popup_fields = [
        "i",
        "n",
        "c",
        "l",
        "p",
        "b",
        "s",
        "t",
        "v",
        "u",
        "d",
        "e",
        "m",
        "r",
    ]

    popup_aliases = [
        "analysis_segment_id",
        "road_name",
        "cluster_name",
        "shape_length",
        "ranked_percentile",
        "percentile_band",
        "speed_limit",
        "target_speed_limit",
        "speed_limit_vs_target_speed_limit",
        "p85_vs_target_speed_limit",
        "intersection_density",
        "traffic_exposure",
        "safe_system_misalignment",
        "total_risk_point",
    ]

    tooltip_fields = [
        "n",
        "c",
        "b",
        "r",
    ]

    tooltip_aliases = [
        "road_name",
        "cluster_name",
        "percentile_band",
        "total_risk_point",
    ]

    map_display_gdf = map_gdf[
        popup_fields + [
            "_band",
            geometry_column,
        ]
    ].copy()

    min_x, min_y, max_x, max_y = (
        map_display_gdf.total_bounds
    )

    center_latitude = (min_y + max_y) / 2
    center_longitude = (min_x + max_x) / 2

    country_map = folium.Map(
        location=[
            center_latitude,
            center_longitude,
        ],
        zoom_start=MAP_INITIAL_ZOOM,
        tiles=None,
        control_scale=True,
        prefer_canvas=True,
    )

    # --------------------------------------------------------------------------
    # Light-grey basemap without labels.
    # --------------------------------------------------------------------------
    folium.TileLayer(
        tiles=(
            "https://{s}.basemaps.cartocdn.com/"
            "light_nolabels/{z}/{x}/{y}.png"
        ),
        attr=(
            "&copy; OpenStreetMap contributors "
            "&copy; CARTO"
        ),
        name="Light Grey (No Labels)",
        overlay=False,
        control=False,
        show=True,
    ).add_to(country_map)

    # --------------------------------------------------------------------------
    # Create percentile-band checkbox layers.
    # --------------------------------------------------------------------------
    for percentile_band in available_percentile_bands:
        band_gdf = map_display_gdf.loc[
            map_display_gdf["_band"].eq(
                percentile_band
            )
        ].copy()

        band_gdf = band_gdf.drop(
            columns="_band"
        )

        layer_name = (
            f"percentile_band = "
            f"{html.escape(percentile_band)} "
            f"({len(band_gdf):,})"
        )

        percentile_band_layer = folium.FeatureGroup(
            name=layer_name,
            overlay=True,
            control=True,
            show=True,
        )

        folium.GeoJson(
            data=to_compact_geojson(
                band_gdf
            ),
            name=layer_name,
            style_function=get_road_style,
            highlight_function=lambda feature: {
                "color": "#000000",
                "weight": 4,
                "opacity": 1.0,
            },
            popup=folium.GeoJsonPopup(
                fields=popup_fields,
                aliases=popup_aliases,
                labels=True,
                localize=False,
                style=(
                    "background-color: white; "
                    "color: black; "
                    "font-family: Arial; "
                    "font-size: 12px; "
                    "width: 670px; "
                    "max-width: 670px;"
                ),
            ),
            tooltip=folium.GeoJsonTooltip(
                fields=tooltip_fields,
                aliases=tooltip_aliases,
                labels=True,
                localize=False,
                sticky=True,
                style=(
                    "background-color: white; "
                    "color: black; "
                    "font-family: Arial; "
                    "font-size: 12px; "
                    "max-width: 370px;"
                ),
            ),
        ).add_to(
            percentile_band_layer
        )

        percentile_band_layer.add_to(
            country_map
        )

    folium.LayerControl(
        collapsed=False,
        position="topright",
    ).add_to(country_map)

    country_map.fit_bounds(
        [
            [min_y, min_x],
            [max_y, max_x],
        ]
    )

    country_map.get_root().header.add_child(
        folium.Element(
            create_popup_css_html()
        )
    )

    country_map.get_root().html.add_child(
        folium.Element(
            create_total_risk_legend_html()
        )
    )

    country_map.save(
        output_html_path
    )

    output_size_mib = get_file_size_mib(
        output_html_path
    )

    return (
        country_map,
        len(map_display_gdf),
        excluded_risk_count,
        available_percentile_bands,
        original_vertex_count,
        simplified_vertex_count,
        simplification_crs,
        output_size_mib,
    )


# ------------------------------------------------------------------------------
# 3. LOAD SCORED GEOPACKAGE
# ------------------------------------------------------------------------------
print("=" * 88)
print("STEP 8 — LIGHTWEIGHT INTERACTIVE TOTAL RISK MAPS")
print("=" * 88)

if not os.path.exists(SCORING_MAP_INPUT):
    raise FileNotFoundError(
        "Scored GeoPackage was not found:\n"
        f"{SCORING_MAP_INPUT}"
    )

input_layer_name = get_first_layer_name(
    SCORING_MAP_INPUT
)

gdf_scored = gpd.read_file(
    SCORING_MAP_INPUT,
    layer=input_layer_name,
    engine="pyogrio",
).copy()

print(f"Input GeoPackage: {SCORING_MAP_INPUT}")
print(f"Input layer     : {input_layer_name}")
print(f"Road records    : {len(gdf_scored):,}")
print(f"CRS             : {gdf_scored.crs}")


# ------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------------------------
required_columns = [
    PRIMARY_KEY,
    COUNTRY_COLUMN,
    ROAD_NAME_COLUMN,
    CLUSTER_NAME_COLUMN,
    SHAPE_LENGTH_COLUMN,
    RANKED_PERCENTILE_COLUMN,
    PERCENTILE_BAND_COLUMN,
    SPEED_LIMIT_COLUMN,
    TARGET_SPEED_LIMIT_COLUMN,
    SPEED_TARGET_MISALIGNMENT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_COLUMN,
    SAFE_SYSTEM_MISALIGNMENT_INSIGHT_COLUMN,
    INTERSECTION_DENSITY_COLUMN,
    TRAFFIC_EXPOSURE_COLUMN,
    TOTAL_RISK_POINT_COLUMN,
]

missing_columns = [
    column
    for column in required_columns
    if column not in gdf_scored.columns
]

if missing_columns:
    raise KeyError(
        "The scored GeoPackage is missing required field(s):\n"
        + "\n".join(
            f"- {column}"
            for column in missing_columns
        )
    )

if gdf_scored[PRIMARY_KEY].isna().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' contains missing values."
    )

if gdf_scored[PRIMARY_KEY].duplicated().any():
    raise ValueError(
        f"'{PRIMARY_KEY}' must be unique."
    )

if gdf_scored.crs is None:
    raise ValueError(
        "Input GeoPackage has no CRS."
    )

print("Required-column validation passed.")


# ------------------------------------------------------------------------------
# 5. SPLIT THAILAND AND MAHARASHTRA DATA
# ------------------------------------------------------------------------------
country_clean = (
    gdf_scored[COUNTRY_COLUMN]
    .astype("string")
    .fillna("")
    .str.strip()
)

thailand_gdf = gdf_scored.loc[
    country_clean.str.casefold().eq(
        THAILAND_LABEL.casefold()
    )
].copy()

maharashtra_gdf = gdf_scored.loc[
    country_clean.str.casefold().eq(
        MAHARASHTRA_LABEL.casefold()
    )
].copy()

available_countries = sorted(
    country_clean.unique().tolist()
)

if thailand_gdf.empty:
    raise ValueError(
        "No Thailand road segments found.\n"
        f"Available dataset values: {available_countries}"
    )

if maharashtra_gdf.empty:
    raise ValueError(
        "No Maharashtra road segments found.\n"
        f"Available dataset values: {available_countries}"
    )

print(f"Thailand segments: {len(thailand_gdf):,}")
print(f"Maharashtra segments: {len(maharashtra_gdf):,}")


# ------------------------------------------------------------------------------
# 6. CREATE THAILAND TOTAL RISK MAP
# ------------------------------------------------------------------------------
print("\n" + "=" * 88)
print("CREATING THAILAND TOTAL RISK MAP")
print("=" * 88)

(
    thailand_total_risk_map,
    thailand_mapped_count,
    thailand_excluded_count,
    thailand_percentile_bands,
    thailand_original_vertices,
    thailand_simplified_vertices,
    thailand_simplification_crs,
    thailand_size_mib,
) = create_total_risk_map(
    source_gdf=thailand_gdf,
    country_label=THAILAND_LABEL,
    output_html_path=THAILAND_TOTAL_RISK_MAP_HTML,
)

if DISPLAY_MAPS:
    display(thailand_total_risk_map)

thailand_vertex_reduction_pct = (
    100
    * (
        1
        - (
            thailand_simplified_vertices
            / max(
                thailand_original_vertices,
                1,
            )
        )
    )
)

print(
    f"Thailand mapped segments: "
    f"{thailand_mapped_count:,}"
)

print(
    f"Thailand excluded invalid-score segments: "
    f"{thailand_excluded_count:,}"
)

print(
    "Thailand geometry simplification:"
    f"\n- Tolerance: {GEOMETRY_SIMPLIFICATION_TOLERANCE_M:.1f} m"
    f"\n- Projection: {thailand_simplification_crs}"
    f"\n- Vertices before: {thailand_original_vertices:,}"
    f"\n- Vertices after: {thailand_simplified_vertices:,}"
    f"\n- Vertex reduction: {thailand_vertex_reduction_pct:.1f}%"
)

print(
    f"Thailand HTML size: "
    f"{thailand_size_mib:.2f} MiB"
)

print(
    "Thailand map saved:\n"
    f"{THAILAND_TOTAL_RISK_MAP_HTML}"
)


# ------------------------------------------------------------------------------
# 7. CREATE MAHARASHTRA TOTAL RISK MAP
# ------------------------------------------------------------------------------
print("\n" + "=" * 88)
print("CREATING MAHARASHTRA TOTAL RISK MAP")
print("=" * 88)

(
    maharashtra_total_risk_map,
    maharashtra_mapped_count,
    maharashtra_excluded_count,
    maharashtra_percentile_bands,
    maharashtra_original_vertices,
    maharashtra_simplified_vertices,
    maharashtra_simplification_crs,
    maharashtra_size_mib,
) = create_total_risk_map(
    source_gdf=maharashtra_gdf,
    country_label=MAHARASHTRA_LABEL,
    output_html_path=MAHARASHTRA_TOTAL_RISK_MAP_HTML,
)

if DISPLAY_MAPS:
    display(maharashtra_total_risk_map)

maharashtra_vertex_reduction_pct = (
    100
    * (
        1
        - (
            maharashtra_simplified_vertices
            / max(
                maharashtra_original_vertices,
                1,
            )
        )
    )
)

print(
    f"Maharashtra mapped segments: "
    f"{maharashtra_mapped_count:,}"
)

print(
    f"Maharashtra excluded invalid-score segments: "
    f"{maharashtra_excluded_count:,}"
)

print(
    "Maharashtra geometry simplification:"
    f"\n- Tolerance: {GEOMETRY_SIMPLIFICATION_TOLERANCE_M:.1f} m"
    f"\n- Projection: {maharashtra_simplification_crs}"
    f"\n- Vertices before: {maharashtra_original_vertices:,}"
    f"\n- Vertices after: {maharashtra_simplified_vertices:,}"
    f"\n- Vertex reduction: {maharashtra_vertex_reduction_pct:.1f}%"
)

print(
    f"Maharashtra HTML size: "
    f"{maharashtra_size_mib:.2f} MiB"
)

print(
    "Maharashtra map saved:\n"
    f"{MAHARASHTRA_TOTAL_RISK_MAP_HTML}"
)


# ------------------------------------------------------------------------------
# 8. FINAL SIZE REVIEW
# ------------------------------------------------------------------------------
print("\n" + "=" * 88)
print("STEP 8 COMPLETED")
print("=" * 88)

print(
    "\nHTML file-size target for GitHub browser upload:"
    f"\n- Limit: {GITHUB_BROWSER_UPLOAD_LIMIT_MIB:.0f} MiB"
)

for country_label, file_size_mib in [
    (
        "Thailand",
        thailand_size_mib,
    ),
    (
        "Maharashtra",
        maharashtra_size_mib,
    ),
]:
    if file_size_mib <= GITHUB_BROWSER_UPLOAD_LIMIT_MIB:
        status = (
            "Within GitHub browser upload limit"
        )
    else:
        status = (
            "Above GitHub browser upload limit"
        )

    print(
        f"\n{country_label}: "
        f"{file_size_mib:.2f} MiB — {status}"
    )

print(
    "\nRoad colours:"
    "\n0 = green | 50 = yellow | 100 = red"
)

print(
    "\nUse percentile_band checkboxes in the upper-right "
    "map control to show or hide each band."
)

del gdf_scored
gc.collect()